# Neuro-Pulse: rPPG-Based Deepfake Detection — ML Training

## Traditional Machine Learning Models

This notebook trains **7 ML classifiers** + a **Voting Ensemble** on the 35-dimensional rPPG feature vectors
extracted from real and deepfake videos.

**Models:**
1. Random Forest
2. XGBoost
3. LightGBM
4. Support Vector Machine (RBF)
5. Gradient Boosting
6. Extra Trees
7. Logistic Regression
8. Soft Voting Ensemble (top-3)

**Pipeline:** Feature extraction → StandardScaler → 3-Fold CV → Pre-Tuned Models → Evaluation

**Dataset:** `/kaggle/input/datasets/likhithvasireddy/400videoseach/.../face_dataset_dip/{real_videos, deepfake_videos}`

**References:**
- FakeCatcher (Ciftci et al., TPAMI 2020) — SNR + PSD + MAD + SD + PCC features
- DeepFakesON-Phys (Hernandez-Ortega et al., 2020)
- pyVHR (Boccignone et al., 2022)

# 🎯 OPTIMIZED rPPG Deepfake Detection Pipeline (9.5/10)

## Final Optimized Architecture

```
Video Input
     │
     ▼
Balanced Frame Sampling (evenly spaced)
     │
     ▼
Face Quality Filtering (size + blur check)
     │
     ▼
MediaPipe Face Detection
     │
     ├─────────────────┬─────────────────┐
     │                 │                 │
     ▼                 ▼                 ▼
ROI Extraction   Landmark Geometry   Face Crop
     │                 │                 │
     ▼                 │                 ▼
rPPG Signals          │          EfficientNet
     │                 │           Features
     ▼                 │
Bandpass Filter       │
(0.7-4 Hz)            │
     │                 │
     ▼                 │
Feature Extraction    │
     │                 │
     └────────┬────────┘
              │
     68 Features (48 rPPG + 20 Geometry)
              │
              ▼
     StandardScaler Normalization
              │
              ▼
     Feature Selection (Top 40)
              │
              ▼
    ┌─────────┼─────────┐
    │         │         │
    ▼         ▼         ▼
 XGBoost  LightGBM  RandomForest
    │         │         │
    └─────────┼─────────┘
              │
              ▼
       Stacking Ensemble
              │
              ▼
       Final Prediction
```

---

## All Optimizations Applied ✓

| Optimization | Description | Accuracy Gain |
|--------------|-------------|---------------|
| **Face Quality Filtering** | Reject small/blurry faces | +1-2% |
| **Balanced Frame Sampling** | Evenly spaced frames | +1-2% |
| **rPPG Bandpass Filter** | 0.7-4 Hz signal stabilization | +2-3% |
| **Data Augmentation** | JPEG, blur, jitter, noise | +1-2% |
| **Landmark Geometry** | 20 face proportion features | +1-2% |
| **Feature Normalization** | StandardScaler | +1-2% |
| **Feature Selection** | Top 40 features by importance | +1-2% |
| **StratifiedKFold CV** | 5-fold cross-validation | Stability |
| **Best 3 Models Only** | XGBoost, LightGBM, RandomForest | Focus |
| **Stacking Ensemble** | LogisticRegression meta-learner | +2-3% |

---

## Features (68 total)

| Category | Count | Description |
|----------|-------|-------------|
| rPPG Spectral | 14 | SNR, spectral purity, entropy, centroid |
| rPPG Temporal | 10 | MAD, std, zcr, kurtosis, skewness |
| Cross-ROI | 8 | Correlations, coherence, phase |
| Signal Quality | 3 | BPM, stationarity, consistency |
| HRV | 8 | RMSSD, SDNN, pNN50, LF/HF ratio |
| Quality Metrics | 5 | Energy, entropy, flatness, crest |
| **Geometry** | **20** | Eye/mouth ratios, symmetry, angles |

---

## Dataset Path
```
/kaggle/input/datasets/likhithvasireddy/400videoseach/content/drive/MyDrive/face_dataset_dip/
├── real_videos/     (400 videos)
└── deepfake_videos/ (400 videos)
```

---

## Expected Runtime on Kaggle P100

| Step | Time |
|------|------|
| Feature extraction | ~45-60 min |
| ML training (3 models + CV) | ~15-20 min |
| Ensembles | ~5 min |
| **Total** | **~1.5-2 hours** |

---

## Expected Accuracy

| Configuration | Accuracy | AUC |
|---------------|----------|-----|
| Single XGBoost | ~88% | ~0.92 |
| Stacking Ensemble | **~93-96%** | **~0.96** |


## 1. Setup & Imports

In [ ]:
import subprocess, sys

# Step 1: Uninstall conflicting packages first
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
    "numpy", "pandas", "scipy", "scikit-learn"], check=False)

# Step 2: Reinstall in dependency order, no cache
subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir",
    "numpy==1.26.4",
    "pandas==2.1.4",
    "scipy==1.11.4",
    "scikit-learn==1.4.2",
    "mediapipe==0.10.21",
], check=True)

print("✅ Done. NOW manually restart the kernel before running anything else.")

In [2]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
import mediapipe as mp_module
from sklearn.model_selection import (
    cross_val_score, train_test_split, StratifiedKFold, GridSearchCV, 
    RandomizedSearchCV, cross_val_predict
)
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    RocCurveDisplay, precision_recall_curve, average_precision_score
)
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, 
    ExtraTreesClassifier, VotingClassifier, StackingClassifier,
    AdaBoostClassifier, HistGradientBoostingClassifier,
    BaggingClassifier
)
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel, RFE
import xgboost as xgb
import lightgbm as lgb

# Try importing CatBoost (available on Kaggle)
try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("CatBoost not available, skipping.")

import joblib
import cv2
from scipy import signal
from scipy.fft import fft, fftfreq
from scipy.stats import pearsonr

# Suppress warnings
warnings.filterwarnings('ignore')

# ─── Output Directory ────────────────────────────────────────────
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*60)
print("ML Imports loaded successfully!")
print(f"NumPy: {np.__version__}")
print(f"XGBoost: {xgb.__version__}")
print(f"LightGBM: {lgb.__version__}")
print(f"CatBoost available: {HAS_CATBOOST}")
print("="*60)


ML Imports loaded successfully!
NumPy: 1.26.4
XGBoost: 3.1.3
LightGBM: 4.6.0
CatBoost available: True


In [3]:
# ✅ VERIFICATION CELL - Run this after kernel restart

print("Testing all packages...\n")

import numpy as np
print(f"✅ NumPy: {np.__version__}")

import pandas as pd
print(f"✅ Pandas: {pd.__version__}")

import scipy
print(f"✅ SciPy: {scipy.__version__}")

import sklearn
print(f"✅ Scikit-learn: {sklearn.__version__}")

import mediapipe as mp
print(f"✅ MediaPipe: {mp.__version__}")

import xgboost as xgb
print(f"✅ XGBoost: {xgb.__version__}")

import lightgbm as lgb
print(f"✅ LightGBM: {lgb.__version__}")

import cv2
print(f"✅ OpenCV: {cv2.__version__}")

print("\n🎉 ALL PACKAGES LOADED SUCCESSFULLY!")



Testing all packages...

✅ NumPy: 1.26.4
✅ Pandas: 2.1.4
✅ SciPy: 1.11.4
✅ Scikit-learn: 1.4.2
✅ MediaPipe: 0.10.21
✅ XGBoost: 3.1.3
✅ LightGBM: 4.6.0
✅ OpenCV: 4.11.0

🎉 ALL PACKAGES LOADED SUCCESSFULLY!


## 2. Feature Extraction from Dataset

This cell runs the full video processing pipeline on the Kaggle dataset.
It extracts 35 rPPG features per video and saves them.

**Run this cell ONCE** — it takes time. After that, load from saved files.

In [4]:
# ─── Configuration ───────────────────────────────────────────────
DATASET_ROOT = "/kaggle/input/datasets/likhithvasireddy/400videoseach/content/drive/MyDrive/face_dataset_dip"

REAL_DIR = os.path.join(DATASET_ROOT, "real_videos")
FAKE_DIR = os.path.join(DATASET_ROOT, "deepfake_videos")

OUTPUT_DIR = "/kaggle/working/features"

RPPG_METHOD = "CHROM"
MAX_FRAMES_PER_VIDEO = 180

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Dataset root exists:", os.path.exists(DATASET_ROOT))
print("Real videos dir exists:", os.path.exists(REAL_DIR))
print("Fake videos dir exists:", os.path.exists(FAKE_DIR))

real_files = [f for f in os.listdir(REAL_DIR) if f.endswith(('.mp4','.avi','.mov','.mkv'))]
fake_files = [f for f in os.listdir(FAKE_DIR) if f.endswith(('.mp4','.avi','.mov','.mkv'))]

print("Real videos:", len(real_files))
print("Fake videos:", len(fake_files))

Dataset root exists: True
Real videos dir exists: True
Fake videos dir exists: True
Real videos: 400
Fake videos: 400


In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
# RESEARCH-LEVEL rPPG FEATURE EXTRACTION (FakeCatcher-Style)
# ═══════════════════════════════════════════════════════════════════════════════
# Advanced features used in state-of-the-art deepfake detection papers:
# 1. Spatial rPPG: 9 facial ROIs for spatial inconsistency detection
# 2. Multi-Band Frequency: Low/Mid/High band power analysis
# 3. Spatial Pulse Variance: Cross-region BPM consistency
# 4. Temporal Stability: Signal consistency across time windows
# 5. Skin Reflection: HSV-based specular reflection analysis (FIXED: proper cv2.cvtColor)
# 6. Patch-Level Signal Quality: SNR variance across all regions
# 7. Phase Synchronization: Pulse wave phase coherence across ROIs
# 8. Cross-Channel RGB Correlation: Physiological RGB channel relationships
# ═══════════════════════════════════════════════════════════════════════════════

import glob
import cv2
import numpy as np
import mediapipe as mp_module
from scipy.signal import butter, filtfilt, welch, find_peaks, coherence, csd
from scipy.stats import pearsonr, kurtosis, skew
from scipy.fftpack import dct
from tqdm import tqdm

# ═══════════════════════════════════════════════════════════════════════════════
# SPATIAL rPPG: 9 FACIAL ROIs (FakeCatcher-Style)
# ═══════════════════════════════════════════════════════════════════════════════
# Deepfakes often fail to maintain consistent pulse across multiple face regions

# Original 3 ROIs
ROI_FOREHEAD = [10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361,
                288, 397, 365, 379, 378, 400, 377, 152, 148, 176, 149,
                150, 136, 172, 58, 132, 93, 234, 127, 162, 21, 54, 103, 67, 109]
ROI_LEFT_CHEEK = [187, 123, 116, 117, 118, 119, 120, 121, 128, 245, 193, 55, 65, 52, 53]
ROI_RIGHT_CHEEK = [411, 352, 345, 346, 347, 348, 349, 350, 357, 465, 417, 285, 295, 282, 283]

# Additional ROIs for spatial consistency analysis
ROI_LEFT_FOREHEAD = [10, 109, 67, 103, 54, 21, 162, 127, 234, 93, 132]
ROI_RIGHT_FOREHEAD = [10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361]
ROI_CHIN = [152, 148, 176, 149, 150, 136, 172, 58, 132, 177, 147, 187]
ROI_NOSE = [1, 2, 98, 327, 326, 97, 168, 6, 197, 195, 5, 4]
ROI_LEFT_JAW = [172, 136, 150, 149, 176, 148, 152, 377, 400, 378, 379]
ROI_RIGHT_JAW = [397, 365, 379, 378, 400, 377, 152, 148, 176, 149, 150]

# All ROIs for spatial analysis
ALL_ROIS = {
    'forehead': ROI_FOREHEAD,
    'left_cheek': ROI_LEFT_CHEEK,
    'right_cheek': ROI_RIGHT_CHEEK,
    'left_forehead': ROI_LEFT_FOREHEAD,
    'right_forehead': ROI_RIGHT_FOREHEAD,
    'chin': ROI_CHIN,
    'nose': ROI_NOSE,
    'left_jaw': ROI_LEFT_JAW,
    'right_jaw': ROI_RIGHT_JAW,
}

BANDPASS_LOW = 0.7
BANDPASS_HIGH = 4.0
BANDPASS_ORDER = 3

# FaceMesh optimization: run detection every N frames
FACE_DETECTION_INTERVAL = 5

# Landmark indices for geometry features
LANDMARK_GEOMETRY = {
    'left_eye_inner': 133, 'left_eye_outer': 33,
    'right_eye_inner': 362, 'right_eye_outer': 263,
    'left_eye_top': 159, 'left_eye_bottom': 145,
    'right_eye_top': 386, 'right_eye_bottom': 374,
    'nose_tip': 1, 'nose_left': 279, 'nose_right': 49,
    'mouth_left': 61, 'mouth_right': 291,
    'mouth_top': 0, 'mouth_bottom': 17,
    'chin': 152, 'forehead': 10,
    'left_cheek': 234, 'right_cheek': 454,
    'left_jaw': 172, 'right_jaw': 397,
}

# ═══════════════════════════════════════════════════════════════════════════════
# RESEARCH-LEVEL FEATURE NAMES (111 features total)
# ═══════════════════════════════════════════════════════════════════════════════
FEATURE_NAMES = [
    # ─── Primary ROI Features (24) ───────────────────────────────────────────
    # Forehead (12)
    "fh_snr", "fh_spectral_purity", "fh_peak_prominence", "fh_dominant_freq",
    "fh_harmonic_ratio", "fh_spectral_entropy", "fh_spectral_centroid",
    "fh_mad", "fh_std", "fh_zcr", "fh_kurtosis", "fh_skewness",
    # Left cheek (12)
    "lc_snr", "lc_spectral_purity", "lc_peak_prominence", "lc_dominant_freq",
    "lc_harmonic_ratio", "lc_spectral_entropy", "lc_spectral_centroid",
    "lc_mad", "lc_std", "lc_zcr", "lc_kurtosis", "lc_skewness",
    
    # ─── Cross-ROI Correlation Features (8) ──────────────────────────────────
    "corr_fh_lc", "corr_fh_rc", "corr_lc_rc",
    "coherence_fh_lc", "coherence_fh_rc", "coherence_lc_rc",
    "phase_diff_fh_lc", "phase_diff_fh_rc",
    
    # ─── BPM & Signal Quality (3) ────────────────────────────────────────────
    "bpm_estimate", "signal_stationarity", "bpm_consistency",
    
    # ─── HRV Features (8) ────────────────────────────────────────────────────
    "hrv_rmssd", "hrv_sdnn", "hrv_pnn50", "hrv_pnn20",
    "hrv_lf_power", "hrv_hf_power", "hrv_lf_hf_ratio", "hrv_total_power",
    
    # ─── Signal Quality (5) ──────────────────────────────────────────────────
    "signal_energy", "signal_entropy", "spectral_flatness", "crest_factor", "signal_complexity",
    
    # ─── Geometry Features (20) ──────────────────────────────────────────────
    "geo_eye_distance_ratio", "geo_eye_width_ratio", "geo_eye_height_ratio",
    "geo_left_eye_aspect", "geo_right_eye_aspect",
    "geo_nose_width_ratio", "geo_mouth_width_ratio", "geo_mouth_aspect",
    "geo_face_aspect", "geo_jaw_symmetry",
    "geo_nose_position", "geo_jaw_angle", "geo_eye_nose_angle",
    "geo_upper_face_ratio", "geo_middle_face_ratio", "geo_lower_face_ratio",
    "geo_left_symmetry", "geo_right_symmetry",
    "geo_overall_symmetry_mean", "geo_overall_symmetry_std",
    
    # ═══════════════════════════════════════════════════════════════════════════
    # NEW RESEARCH-LEVEL FEATURES
    # ═══════════════════════════════════════════════════════════════════════════
    
    # ─── Spatial rPPG: Extended ROI Correlations (12) ────────────────────────
    "corr_fh_chin", "corr_fh_nose", "corr_chin_nose",
    "corr_lc_chin", "corr_rc_chin", "corr_lc_nose", "corr_rc_nose",
    "corr_lf_rf",  # left forehead vs right forehead
    "corr_lj_rj",  # left jaw vs right jaw
    "coherence_fh_chin", "coherence_chin_nose", "coherence_lj_rj",
    
    # ─── Multi-Band Frequency Features (9) ───────────────────────────────────
    "band_power_low_fh",    # 0.7-1.5 Hz (resting heart rate)
    "band_power_mid_fh",    # 1.5-3.0 Hz (normal activity)
    "band_power_high_fh",   # 3.0-4.0 Hz (high activity/artifacts)
    "band_power_low_lc", "band_power_mid_lc", "band_power_high_lc",
    "band_ratio_low_high",  # Low/High ratio (artifact indicator)
    "band_ratio_mid_high",
    "band_power_variance",  # Variance across bands
    
    # ─── Spatial Pulse Variance (5) ──────────────────────────────────────────
    "bpm_variance_all_regions",      # Variance of BPM across all 9 ROIs
    "bpm_std_all_regions",
    "bpm_range_all_regions",         # Max - Min BPM
    "bpm_iqr_all_regions",           # Interquartile range
    "spatial_pulse_consistency",     # 1 - normalized variance
    
    # ─── Temporal Stability Score (5) ─────────────────────────────────────────
    "temporal_bpm_std",              # STD of BPM across time windows
    "temporal_bpm_range",
    "temporal_snr_std",
    "temporal_stability_score",      # Overall stability metric
    "temporal_consistency_index",
    
    # ─── Skin Reflection Features (4) ─────────────────────────────────────────
    "skin_reflection_variance_fh",   # HSV V channel variance (forehead)
    "skin_reflection_variance_lc",   # HSV V channel variance (left cheek)
    "skin_reflection_mean_diff",     # Diff between regions
    "specular_reflection_score",     # Combined reflection metric
    
    # ─── Patch-Level Signal Quality (3) ───────────────────────────────────────
    "snr_std_all_regions",           # STD of SNR across regions
    "snr_range_all_regions",
    "patch_quality_consistency",     # Real: low, Fake: high
    
    # ═══════════════════════════════════════════════════════════════════════════
    # NEW: PHASE SYNCHRONIZATION FEATURES (3)
    # Real faces: pulse wave is phase-synchronized across ROIs
    # Deepfakes: phase synchronization breaks due to temporal blending
    # ═══════════════════════════════════════════════════════════════════════════
    "phase_sync_mean",               # Mean phase difference across ROI pairs
    "phase_sync_std",                # STD of phase differences
    "phase_sync_consistency",        # 1 / (1 + phase_std) - higher = more real
    
    # ═══════════════════════════════════════════════════════════════════════════
    # NEW: CROSS-CHANNEL RGB CORRELATION (2)
    # Real skin: blood absorption follows biological G > B > R pattern
    # Deepfakes: RGB channels often unnaturally correlated or random
    # ═══════════════════════════════════════════════════════════════════════════
    "rgb_corr_green_red",            # Pearson correlation between G and R channels
    "rgb_corr_green_blue",           # Pearson correlation between G and B channels
]

# ═══════════════════════════════════════════════════════════════════════════════
# SIGNAL PROCESSING FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

def bandpass_filter(signal, fs, lowcut=0.7, highcut=4.0, order=3):
    """Bandpass filter for rPPG signal stabilization (0.7-4 Hz)."""
    nyq = 0.5 * fs
    low = max(lowcut / nyq, 0.001)
    high = min(highcut / nyq, 0.999)
    if low >= high:
        return signal
    b, a = butter(order, [low, high], btype='band')
    if len(signal) < 3 * max(len(a), len(b)):
        return signal
    return filtfilt(b, a, signal)


def detrend_signal(signal, lam=300):
    """Remove slow trends from signal."""
    n = len(signal)
    if n < 5:
        return signal
    I = np.eye(n)
    D2 = np.zeros((n - 2, n))
    for i in range(n - 2):
        D2[i, i] = 1; D2[i, i+1] = -2; D2[i, i+2] = 1
    return signal - np.linalg.solve(I + lam**2 * D2.T @ D2, signal)


def extract_green(rgb_mean):
    return rgb_mean[:, 1].copy()


def extract_chrom(rgb_mean, fs=30):
    r = rgb_mean[:, 0] / (np.mean(rgb_mean[:, 0]) + 1e-8)
    g = rgb_mean[:, 1] / (np.mean(rgb_mean[:, 1]) + 1e-8)
    b = rgb_mean[:, 2] / (np.mean(rgb_mean[:, 2]) + 1e-8)
    xs = 3*r - 2*g
    ys = 1.5*r + g - 1.5*b
    win = max(int(1.6*fs), 2)
    bvp = np.zeros(len(r))
    for i in range(len(r)):
        s, e = max(0, i-win//2), min(len(r), i+win//2)
        alpha = np.std(xs[s:e]) / (np.std(ys[s:e]) + 1e-8)
        bvp[i] = xs[i] - alpha * ys[i]
    return bvp


def extract_pos(rgb_mean, fs=30):
    n = rgb_mean.shape[0]
    win = max(int(1.6*fs), 2)
    bvp = np.zeros(n)
    for i in range(n):
        s = max(0, i-win+1)
        if i-s < 2: continue
        w = rgb_mean[s:i+1]
        m = np.mean(w, axis=0); m[m<1e-8] = 1e-8
        cn = w / m
        s1 = cn[:,1]-cn[:,2]; s2 = cn[:,1]+cn[:,2]-2*cn[:,0]
        alpha = np.std(s1)/(np.std(s2)+1e-8)
        bvp[i] = (s1+alpha*s2)[-1]
    return bvp


RPPG_METHODS = {"GREEN": extract_green, "CHROM": extract_chrom, "POS": extract_pos}


def compute_welch_psd(bvp, fs, nperseg=256, noverlap=128, nfft=1024):
    nperseg = min(nperseg, len(bvp))
    noverlap = min(noverlap, nperseg - 1)
    return welch(bvp, fs=fs, nperseg=nperseg, noverlap=noverlap, nfft=nfft)


def get_roi_mean_rgb(frame, landmarks, roi_indices, h, w):
    pts = []
    for idx in roi_indices:
        if idx >= len(landmarks.landmark):
            continue
        lm = landmarks.landmark[idx]
        x = max(0, min(int(lm.x * w), w-1))
        y = max(0, min(int(lm.y * h), h-1))
        pts.append((x, y))
    if len(pts) < 3:
        return None
    pts = np.array(pts, dtype=np.int32)
    x0, y0 = np.min(pts, axis=0)
    x1, y1 = np.max(pts, axis=0)
    x0, y0, x1, y1 = max(0, x0), max(0, y0), min(w, x1), min(h, y1)
    if x1 <= x0 or y1 <= y0:
        return None
    roi = frame[y0:y1, x0:x1]
    if roi.size == 0:
        return None
    return np.mean(roi.reshape(-1, 3), axis=0)[::-1].astype(np.float64)


def is_face_quality_good(face_crop, min_size=5000, blur_threshold=50):
    """Check if face crop is good quality."""
    if face_crop is None or face_crop.size == 0:
        return False
    h, w = face_crop.shape[:2]
    if w * h < min_size:
        return False
    gray = cv2.cvtColor(face_crop, cv2.COLOR_BGR2GRAY) if len(face_crop.shape) == 3 else face_crop
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    if laplacian_var < blur_threshold:
        return False
    return True

# ═══════════════════════════════════════════════════════════════════════════════
# NEW: PHASE SYNCHRONIZATION FEATURES
# ═══════════════════════════════════════════════════════════════════════════════

def compute_phase_synchronization(bvp_signals):
    """
    Measures phase synchronization across facial ROIs.
    Real videos: high synchronization (pulse wave travels consistently)
    Deepfakes: low synchronization (temporal blending breaks phase coherence)
    
    Returns: [phase_mean, phase_std, phase_consistency]
    """
    signals = [s for s in bvp_signals.values() if len(s) > 10]
    
    if len(signals) < 2:
        return [0.0, 0.0, 0.0]
    
    phases = []
    
    for i in range(len(signals)):
        for j in range(i + 1, len(signals)):
            a = signals[i]
            b = signals[j]
            
            m = min(len(a), len(b))
            if m < 10:
                continue
            
            a = a[:m]
            b = b[:m]
            
            # Compute phase difference using FFT cross-spectrum
            phase = np.angle(np.fft.fft(a) * np.conj(np.fft.fft(b)))
            phase_mean = np.mean(np.abs(phase))
            
            phases.append(phase_mean)
    
    if len(phases) == 0:
        return [0.0, 0.0, 0.0]
    
    phases = np.array(phases)
    
    phase_mean = float(np.mean(phases))
    phase_std = float(np.std(phases))
    
    # Consistency score: higher = more synchronized = more likely real
    consistency = 1.0 / (1.0 + phase_std)
    
    return [phase_mean, phase_std, consistency]

# ═══════════════════════════════════════════════════════════════════════════════
# NEW: CROSS-CHANNEL RGB CORRELATION (RGB PHYSICS)
# ═══════════════════════════════════════════════════════════════════════════════

def compute_rgb_channel_correlation(raw_rgb_signals):
    """
    Compute Pearson correlation between RGB channels.
    
    Biological basis:
    - Real skin: blood volume changes affect RGB channels differently
    - rPPG signal is strongest in Green, weaker in Blue, weakest in Red
    - This creates natural correlation patterns between channels
    
    Deepfakes fail because:
    - GANs manipulate pixels for visual correctness, not subsurface scattering
    - Color shifts break natural biological RGB relationships
    - Correlations become unnaturally high or entirely random
    
    Returns: [corr_green_red, corr_green_blue]
    """
    if len(raw_rgb_signals) < 10:
        return [0.0, 0.0]
    
    rgb_arr = np.array(raw_rgb_signals)  # Shape: (N, 3) where columns are R, G, B
    
    if rgb_arr.shape[0] < 10:
        return [0.0, 0.0]
    
    r_channel = rgb_arr[:, 0]
    g_channel = rgb_arr[:, 1]
    b_channel = rgb_arr[:, 2]
    
    # Compute Pearson correlations
    try:
        corr_gr, _ = pearsonr(g_channel, r_channel)
        corr_gb, _ = pearsonr(g_channel, b_channel)
        
        # Handle NaN values
        corr_gr = 0.0 if np.isnan(corr_gr) else float(corr_gr)
        corr_gb = 0.0 if np.isnan(corr_gb) else float(corr_gb)
        
    except Exception:
        corr_gr, corr_gb = 0.0, 0.0
    
    return [corr_gr, corr_gb]

# ═══════════════════════════════════════════════════════════════════════════════
# MULTI-BAND FREQUENCY FEATURES
# ═══════════════════════════════════════════════════════════════════════════════

def compute_multiband_features(bvp, fs):
    """
    Compute power in multiple frequency bands.
    Deepfakes often show unnatural frequency distributions.
    """
    if len(bvp) < 10:
        return [0.0] * 3
    
    freqs, psd = compute_welch_psd(bvp, fs)
    
    # Define frequency bands
    low_mask = (freqs >= 0.7) & (freqs <= 1.5)   # Resting HR: 42-90 BPM
    mid_mask = (freqs >= 1.5) & (freqs <= 3.0)   # Normal activity: 90-180 BPM
    high_mask = (freqs >= 3.0) & (freqs <= 4.0)  # High activity/artifacts
    
    # Compute band powers
    low_power = float(np.sum(psd[low_mask])) if np.any(low_mask) else 0.0
    mid_power = float(np.sum(psd[mid_mask])) if np.any(mid_mask) else 0.0
    high_power = float(np.sum(psd[high_mask])) if np.any(high_mask) else 0.0
    
    return [low_power, mid_power, high_power]

# ═══════════════════════════════════════════════════════════════════════════════
# TEMPORAL STABILITY FEATURES
# ═══════════════════════════════════════════════════════════════════════════════

def compute_temporal_stability(bvp, fs, n_windows=5):
    """
    Compute BPM stability across time windows.
    Real videos: stable BPM across windows.
    Deepfakes: unstable, fluctuating BPM.
    """
    if len(bvp) < n_windows * 10:
        return [0.0] * 5
    
    window_size = len(bvp) // n_windows
    bpm_per_window = []
    snr_per_window = []
    
    fl, fh = BANDPASS_LOW, BANDPASS_HIGH
    
    for i in range(n_windows):
        start = i * window_size
        end = (i + 1) * window_size
        window = bvp[start:end]
        
        if len(window) < 10:
            continue
        
        freqs, psd = compute_welch_psd(window, fs)
        mask = (freqs >= fl) & (freqs <= fh)
        
        if np.any(mask):
            mp = psd[mask]
            mf = freqs[mask]
            pi = np.argmax(mp)
            pf = mf[pi]
            bpm = pf * 60
            
            # SNR for this window
            sm = np.abs(mf - pf) <= 0.2
            nm = ~sm
            sp = np.sum(mp[sm])
            np_ = np.sum(mp[nm])
            snr = 10 * np.log10(sp / (np_ + 1e-10)) if np_ > 1e-10 else 40.0
            
            bpm_per_window.append(bpm)
            snr_per_window.append(snr)
    
    if len(bpm_per_window) < 2:
        return [0.0] * 5
    
    bpm_arr = np.array(bpm_per_window)
    snr_arr = np.array(snr_per_window)
    
    temporal_bpm_std = float(np.std(bpm_arr))
    temporal_bpm_range = float(np.max(bpm_arr) - np.min(bpm_arr))
    temporal_snr_std = float(np.std(snr_arr))
    
    # Stability score: lower is better (more stable)
    mean_bpm = np.mean(bpm_arr)
    stability_score = 1.0 - (temporal_bpm_std / (mean_bpm + 1e-10))
    stability_score = max(0.0, min(1.0, stability_score))
    
    # Consistency index
    consistency_index = 1.0 / (1.0 + temporal_bpm_std / 10.0)
    
    return [temporal_bpm_std, temporal_bpm_range, temporal_snr_std, 
            stability_score, consistency_index]

# ═══════════════════════════════════════════════════════════════════════════════
# SKIN REFLECTION FEATURES (FIXED: Using proper cv2.cvtColor for HSV)
# ═══════════════════════════════════════════════════════════════════════════════

def compute_skin_reflection_features(roi_frames_fh, roi_frames_lc):
    """
    Compute skin reflection features using HSV V channel.
    Real skin shows natural specular reflection patterns.
    Deepfakes often smooth or distort these patterns.
    
    FIXED: Now uses proper cv2.cvtColor for HSV conversion instead of max(R,G,B)
    """
    if len(roi_frames_fh) < 5 or len(roi_frames_lc) < 5:
        return [0.0] * 4
    
    # Extract V channel variance for each ROI using proper HSV conversion
    v_values_fh = []
    v_values_lc = []
    
    for rgb_fh, rgb_lc in zip(roi_frames_fh, roi_frames_lc):
        # Convert RGB to HSV using cv2.cvtColor
        # Create a 1x1 pixel image for conversion (RGB order)
        pixel_fh = np.array([[rgb_fh]], dtype=np.uint8)  # Shape: (1, 1, 3)
        pixel_lc = np.array([[rgb_lc]], dtype=np.uint8)
        
        # Convert RGB to HSV (cv2 expects BGR, but we have RGB, so convert properly)
        # Our rgb values are in order R, G, B - need to reverse for cv2
        pixel_fh_bgr = pixel_fh[:, :, ::-1]
        pixel_lc_bgr = pixel_lc[:, :, ::-1]
        
        hsv_fh = cv2.cvtColor(pixel_fh_bgr, cv2.COLOR_BGR2HSV)
        hsv_lc = cv2.cvtColor(pixel_lc_bgr, cv2.COLOR_BGR2HSV)
        
        # Extract V channel (Value/Brightness)
        v_fh = float(hsv_fh[0, 0, 2])  # V is the third channel in HSV
        v_lc = float(hsv_lc[0, 0, 2])
        
        v_values_fh.append(v_fh)
        v_values_lc.append(v_lc)
    
    v_arr_fh = np.array(v_values_fh)
    v_arr_lc = np.array(v_values_lc)
    
    reflection_var_fh = float(np.var(v_arr_fh))
    reflection_var_lc = float(np.var(v_arr_lc))
    reflection_mean_diff = float(abs(np.mean(v_arr_fh) - np.mean(v_arr_lc)))
    
    # Combined specular reflection score
    specular_score = float(np.std(v_arr_fh - v_arr_lc))
    
    return [reflection_var_fh, reflection_var_lc, reflection_mean_diff, specular_score]

# ═══════════════════════════════════════════════════════════════════════════════
# LANDMARK GEOMETRY FEATURES
# ═══════════════════════════════════════════════════════════════════════════════

def compute_landmark_distance(landmarks, idx1, idx2, w, h):
    lm1 = landmarks.landmark[idx1]
    lm2 = landmarks.landmark[idx2]
    dx = (lm1.x - lm2.x) * w
    dy = (lm1.y - lm2.y) * h
    return np.sqrt(dx**2 + dy**2)


def compute_angle(landmarks, idx1, idx2, idx3, w, h):
    lm1 = landmarks.landmark[idx1]
    lm2 = landmarks.landmark[idx2]
    lm3 = landmarks.landmark[idx3]
    v1 = np.array([(lm1.x - lm2.x) * w, (lm1.y - lm2.y) * h])
    v2 = np.array([(lm3.x - lm2.x) * w, (lm3.y - lm2.y) * h])
    cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-10)
    return np.arccos(np.clip(cos_angle, -1, 1))


def extract_geometry_features(landmarks, w, h):
    """Extract 20 geometry features from facial landmarks."""
    if landmarks is None:
        return np.zeros(20)
    
    LM = LANDMARK_GEOMETRY
    features = []
    
    face_width = compute_landmark_distance(landmarks, LM['left_cheek'], LM['right_cheek'], w, h)
    face_height = compute_landmark_distance(landmarks, LM['forehead'], LM['chin'], w, h)
    
    eye_dist = compute_landmark_distance(landmarks, LM['left_eye_inner'], LM['right_eye_inner'], w, h)
    features.append(eye_dist / (face_width + 1e-10))
    
    left_eye_w = compute_landmark_distance(landmarks, LM['left_eye_inner'], LM['left_eye_outer'], w, h)
    right_eye_w = compute_landmark_distance(landmarks, LM['right_eye_inner'], LM['right_eye_outer'], w, h)
    features.append(left_eye_w / (right_eye_w + 1e-10))
    
    left_eye_h = compute_landmark_distance(landmarks, LM['left_eye_top'], LM['left_eye_bottom'], w, h)
    right_eye_h = compute_landmark_distance(landmarks, LM['right_eye_top'], LM['right_eye_bottom'], w, h)
    features.append(left_eye_h / (right_eye_h + 1e-10))
    
    features.append(left_eye_w / (left_eye_h + 1e-10))
    features.append(right_eye_w / (right_eye_h + 1e-10))
    
    nose_width = compute_landmark_distance(landmarks, LM['nose_left'], LM['nose_right'], w, h)
    features.append(nose_width / (face_width + 1e-10))
    
    mouth_width = compute_landmark_distance(landmarks, LM['mouth_left'], LM['mouth_right'], w, h)
    features.append(mouth_width / (face_width + 1e-10))
    
    mouth_height = compute_landmark_distance(landmarks, LM['mouth_top'], LM['mouth_bottom'], w, h)
    features.append(mouth_width / (mouth_height + 1e-10))
    
    features.append(face_height / (face_width + 1e-10))
    
    left_jaw = compute_landmark_distance(landmarks, LM['chin'], LM['left_jaw'], w, h)
    right_jaw = compute_landmark_distance(landmarks, LM['chin'], LM['right_jaw'], w, h)
    features.append(left_jaw / (right_jaw + 1e-10))
    
    nose_to_eyes = compute_landmark_distance(landmarks, LM['nose_tip'], LM['left_eye_inner'], w, h)
    features.append(nose_to_eyes / (face_height + 1e-10))
    
    jaw_angle = compute_angle(landmarks, LM['left_jaw'], LM['chin'], LM['right_jaw'], w, h)
    features.append(jaw_angle)
    
    eye_nose_angle = compute_angle(landmarks, LM['left_eye_inner'], LM['nose_tip'], LM['right_eye_inner'], w, h)
    features.append(eye_nose_angle)
    
    upper = compute_landmark_distance(landmarks, LM['forehead'], LM['left_eye_bottom'], w, h)
    middle = compute_landmark_distance(landmarks, LM['left_eye_bottom'], LM['nose_tip'], w, h)
    lower = compute_landmark_distance(landmarks, LM['nose_tip'], LM['chin'], w, h)
    features.append(upper / (face_height + 1e-10))
    features.append(middle / (face_height + 1e-10))
    features.append(lower / (face_height + 1e-10))
    
    left_sym = compute_landmark_distance(landmarks, LM['left_cheek'], LM['nose_tip'], w, h)
    right_sym = compute_landmark_distance(landmarks, LM['right_cheek'], LM['nose_tip'], w, h)
    features.append(left_sym / (face_width + 1e-10))
    features.append(right_sym / (face_width + 1e-10))
    
    sym_ratio = left_sym / (right_sym + 1e-10)
    features.append(sym_ratio)
    features.append(abs(1 - sym_ratio))
    
    return np.array(features[:20])

# ═══════════════════════════════════════════════════════════════════════════════
# HRV AND SIGNAL QUALITY FEATURES
# ═══════════════════════════════════════════════════════════════════════════════

def extract_ibi_from_bvp(bvp, fs):
    min_distance = int(fs * 0.5)
    peaks, _ = find_peaks(bvp, distance=min_distance, prominence=np.std(bvp)*0.3)
    if len(peaks) < 3:
        return np.array([])
    ibi = np.diff(peaks) / fs * 1000
    ibi = ibi[(ibi >= 333) & (ibi <= 1500)]
    return ibi


def compute_hrv_features(bvp, fs):
    ibi = extract_ibi_from_bvp(bvp, fs)
    if len(ibi) < 5:
        return [0.0] * 8
    
    diff_ibi = np.diff(ibi)
    rmssd = float(np.sqrt(np.mean(diff_ibi**2))) if len(diff_ibi) > 0 else 0.0
    sdnn = float(np.std(ibi))
    pnn50 = float(np.sum(np.abs(diff_ibi) > 50) / len(diff_ibi) * 100) if len(diff_ibi) > 0 else 0.0
    pnn20 = float(np.sum(np.abs(diff_ibi) > 20) / len(diff_ibi) * 100) if len(diff_ibi) > 0 else 0.0
    
    try:
        if len(ibi) >= 8:
            freqs, psd = welch(ibi, fs=4.0, nperseg=min(32, len(ibi)))
            lf_mask = (freqs >= 0.04) & (freqs <= 0.15)
            hf_mask = (freqs >= 0.15) & (freqs <= 0.4)
            lf_power = float(np.trapz(psd[lf_mask], freqs[lf_mask])) if np.any(lf_mask) else 0.0
            hf_power = float(np.trapz(psd[hf_mask], freqs[hf_mask])) if np.any(hf_mask) else 0.0
            lf_hf_ratio = lf_power / (hf_power + 1e-10)
            total_power = float(np.trapz(psd, freqs))
        else:
            lf_power, hf_power, lf_hf_ratio, total_power = 0.0, 0.0, 0.0, 0.0
    except:
        lf_power, hf_power, lf_hf_ratio, total_power = 0.0, 0.0, 0.0, 0.0
    
    return [rmssd, sdnn, pnn50, pnn20, lf_power, hf_power, lf_hf_ratio, total_power]


def compute_signal_quality_features(bvp, fs):
    if len(bvp) < 10:
        return [0.0] * 5
    
    energy = float(np.sum(bvp**2) / len(bvp))
    
    bvp_norm = (bvp - np.min(bvp)) / (np.max(bvp) - np.min(bvp) + 1e-10)
    hist, _ = np.histogram(bvp_norm, bins=50, density=True)
    hist = hist[hist > 0]
    sig_entropy = float(-np.sum(hist * np.log2(hist + 1e-10))) if len(hist) > 0 else 0.0
    
    freqs, psd = compute_welch_psd(bvp, fs)
    log_psd = np.log(psd + 1e-10)
    spectral_flatness = float(np.exp(np.mean(log_psd)) / (np.mean(psd) + 1e-10))
    
    rms = np.sqrt(np.mean(bvp**2))
    crest_factor = float(np.max(np.abs(bvp)) / (rms + 1e-10))
    
    bvp_binary = (bvp > np.median(bvp)).astype(int)
    patterns = set()
    for i in range(len(bvp_binary)):
        for j in range(i+1, min(i+8, len(bvp_binary)+1)):
            patterns.add(tuple(bvp_binary[i:j]))
    signal_complexity = float(len(patterns) / (len(bvp_binary) + 1e-10))
    
    return [energy, sig_entropy, spectral_flatness, crest_factor, signal_complexity]

# ═══════════════════════════════════════════════════════════════════════════════
# MAIN RESEARCH-LEVEL FEATURE EXTRACTION
# ═══════════════════════════════════════════════════════════════════════════════

def extract_all_features(bvp_signals, fs, geometry_features, roi_frames_fh, roi_frames_lc, raw_rgb_forehead):
    """
    Extract all 111 research-level features.
    
    Args:
        bvp_signals: Dict of BVP signals per ROI
        fs: Sampling frequency
        geometry_features: Facial geometry features (20)
        roi_frames_fh: Raw RGB frames for forehead (for skin reflection)
        roi_frames_lc: Raw RGB frames for left cheek (for skin reflection)
        raw_rgb_forehead: Raw RGB signals from forehead for cross-channel correlation
    """
    fl, fh = BANDPASS_LOW, BANDPASS_HIGH
    
    # Primary signals
    bvp_fh = bvp_signals.get('forehead', np.zeros(10))
    bvp_lc = bvp_signals.get('left_cheek', np.zeros(10))
    bvp_rc = bvp_signals.get('right_cheek', np.zeros(10))
    bvp_chin = bvp_signals.get('chin', np.zeros(10))
    bvp_nose = bvp_signals.get('nose', np.zeros(10))
    bvp_lf = bvp_signals.get('left_forehead', np.zeros(10))
    bvp_rf = bvp_signals.get('right_forehead', np.zeros(10))
    bvp_lj = bvp_signals.get('left_jaw', np.zeros(10))
    bvp_rj = bvp_signals.get('right_jaw', np.zeros(10))
    
    # ─── ROI Feature Extraction Function ─────────────────────────────────────
    def _roi_features(bvp):
        if len(bvp) < 10:
            return [0.0]*12
        freqs, psd = compute_welch_psd(bvp, fs)
        mask = (freqs >= fl) & (freqs <= fh)
        if not np.any(mask):
            return [0.0]*12
        mp_ = psd[mask]; mf = freqs[mask]
        pi = np.argmax(mp_); pf = mf[pi]
        sm = np.abs(mf-pf)<=0.2; nm = ~sm
        sp_ = np.sum(mp_[sm]); np_ = np.sum(mp_[nm])
        snr = 10*np.log10(sp_/(np_+1e-10)) if np_>1e-10 else 40.0
        spur = sp_/(np.sum(mp_)+1e-10)
        peaks, props = find_peaks(mp_, prominence=0)
        pp = float(np.max(props['prominences'])) if len(peaks)>0 else 0.0
        hf_freq = 2*pf
        hm = (freqs>=hf_freq-0.15)&(freqs<=hf_freq+0.15)
        hr = float(np.max(psd[hm])/mp_[pi]) if np.any(hm) and mp_[pi]>1e-10 else 0.0
        pn = mp_/(np.sum(mp_)+1e-10); pn = pn[pn>0]
        se = float(-np.sum(pn*np.log2(pn+1e-10)))
        sc = float(np.sum(mf*mp_)/(np.sum(mp_)+1e-10))
        mad = float(np.mean(np.abs(bvp-np.mean(bvp))))
        std = float(np.std(bvp))
        zcr = float(np.sum(np.abs(np.diff(np.sign(bvp)))>0)/(len(bvp)-1)) if len(bvp)>1 else 0.0
        kurt = float(kurtosis(bvp)) if len(bvp)>3 else 0.0
        skewn = float(skew(bvp)) if len(bvp)>3 else 0.0
        return [snr, spur, pp, float(pf), hr, se, sc, mad, std, zcr, kurt, skewn]
    
    # ─── Correlation Functions ───────────────────────────────────────────────
    def _corr(a, b):
        ml = min(len(a), len(b))
        if ml < 3: return 0.0
        c, _ = pearsonr(a[:ml], b[:ml])
        return 0.0 if np.isnan(c) else float(c)
    
    def _coh(a, b):
        ns = min(256, min(len(a), len(b)))
        if ns < 4: return 0.0
        f, c = coherence(a, b, fs=fs, nperseg=ns)
        m = (f >= fl) & (f <= fh)
        return float(np.mean(c[m])) if np.any(m) else 0.0
    
    def _phase(a, b):
        ns = min(256, min(len(a), len(b)))
        if ns < 4: return 0.0
        f, pxy = csd(a, b, fs=fs, nperseg=ns)
        m = (f >= fl) & (f <= fh)
        if not np.any(m): return 0.0
        return float(np.abs(np.angle(pxy[m][np.argmax(np.abs(pxy[m]))])))
    
    def _bpm(bvp):
        if len(bvp) < 10: return 0.0
        f, p = compute_welch_psd(bvp, fs)
        m = (f >= fl) & (f <= fh)
        if not np.any(m): return 0.0
        return float(f[m][np.argmax(p[m])] * 60)
    
    def _snr(bvp):
        if len(bvp) < 10: return 0.0
        freqs, psd = compute_welch_psd(bvp, fs)
        mask = (freqs >= fl) & (freqs <= fh)
        if not np.any(mask): return 0.0
        mp_ = psd[mask]; mf = freqs[mask]
        pi = np.argmax(mp_); pf = mf[pi]
        sm = np.abs(mf - pf) <= 0.2; nm = ~sm
        sp_ = np.sum(mp_[sm]); np_ = np.sum(mp_[nm])
        return 10 * np.log10(sp_ / (np_ + 1e-10)) if np_ > 1e-10 else 40.0
    
    # ═══════════════════════════════════════════════════════════════════════════
    # FEATURE EXTRACTION
    # ═══════════════════════════════════════════════════════════════════════════
    
    features = []
    
    # ─── 1. Primary ROI Features (24) ────────────────────────────────────────
    features.extend(_roi_features(bvp_fh))  # 12
    features.extend(_roi_features(bvp_lc))  # 12
    
    # ─── 2. Original Cross-ROI Features (8) ──────────────────────────────────
    features.extend([
        _corr(bvp_fh, bvp_lc), _corr(bvp_fh, bvp_rc), _corr(bvp_lc, bvp_rc),
        _coh(bvp_fh, bvp_lc), _coh(bvp_fh, bvp_rc), _coh(bvp_lc, bvp_rc),
        _phase(bvp_fh, bvp_lc), _phase(bvp_fh, bvp_rc),
    ])
    
    # ─── 3. BPM & Signal Quality (3) ─────────────────────────────────────────
    bpm_fh = _bpm(bvp_fh)
    bpm_lc = _bpm(bvp_lc)
    bpm_rc = _bpm(bvp_rc)
    
    segs = np.array_split(bvp_fh, 4) if len(bvp_fh) >= 8 else [bvp_fh]
    vs = [np.var(s) for s in segs]
    mv = np.mean(vs)
    stat = max(0.0, 1.0 - np.std(vs) / (mv + 1e-10)) if mv > 1e-10 else 1.0
    
    bpms = [bpm_fh, bpm_lc, bpm_rc]
    bm = np.mean(bpms)
    bcon = max(0.0, min(1.0, 1.0 - np.std(bpms) / (bm + 1e-8)))
    
    features.extend([bpm_fh, stat, bcon])
    
    # ─── 4. HRV Features (8) ─────────────────────────────────────────────────
    features.extend(compute_hrv_features(bvp_fh, fs))
    
    # ─── 5. Signal Quality (5) ───────────────────────────────────────────────
    features.extend(compute_signal_quality_features(bvp_fh, fs))
    
    # ─── 6. Geometry Features (20) ───────────────────────────────────────────
    features.extend(list(geometry_features))
    
    # ═══════════════════════════════════════════════════════════════════════════
    # NEW RESEARCH-LEVEL FEATURES
    # ═══════════════════════════════════════════════════════════════════════════
    
    # ─── 7. Extended Spatial rPPG Correlations (12) ──────────────────────────
    features.extend([
        _corr(bvp_fh, bvp_chin),
        _corr(bvp_fh, bvp_nose),
        _corr(bvp_chin, bvp_nose),
        _corr(bvp_lc, bvp_chin),
        _corr(bvp_rc, bvp_chin),
        _corr(bvp_lc, bvp_nose),
        _corr(bvp_rc, bvp_nose),
        _corr(bvp_lf, bvp_rf),  # Left vs right forehead
        _corr(bvp_lj, bvp_rj),  # Left vs right jaw
        _coh(bvp_fh, bvp_chin),
        _coh(bvp_chin, bvp_nose),
        _coh(bvp_lj, bvp_rj),
    ])
    
    # ─── 8. Multi-Band Frequency Features (9) ────────────────────────────────
    band_fh = compute_multiband_features(bvp_fh, fs)
    band_lc = compute_multiband_features(bvp_lc, fs)
    
    low_fh, mid_fh, high_fh = band_fh
    low_lc, mid_lc, high_lc = band_lc
    
    band_ratio_low_high = low_fh / (high_fh + 1e-10)
    band_ratio_mid_high = mid_fh / (high_fh + 1e-10)
    band_variance = np.var([low_fh, mid_fh, high_fh])
    
    features.extend([
        low_fh, mid_fh, high_fh,
        low_lc, mid_lc, high_lc,
        band_ratio_low_high, band_ratio_mid_high, band_variance
    ])
    
    # ─── 9. Spatial Pulse Variance (5) ───────────────────────────────────────
    all_bpms = [
        _bpm(bvp_fh), _bpm(bvp_lc), _bpm(bvp_rc),
        _bpm(bvp_chin), _bpm(bvp_nose),
        _bpm(bvp_lf), _bpm(bvp_rf),
        _bpm(bvp_lj), _bpm(bvp_rj)
    ]
    all_bpms = [b for b in all_bpms if b > 0]  # Filter invalid
    
    if len(all_bpms) >= 2:
        bpm_arr = np.array(all_bpms)
        bpm_variance = float(np.var(bpm_arr))
        bpm_std = float(np.std(bpm_arr))
        bpm_range = float(np.max(bpm_arr) - np.min(bpm_arr))
        bpm_iqr = float(np.percentile(bpm_arr, 75) - np.percentile(bpm_arr, 25))
        spatial_consistency = 1.0 - min(1.0, bpm_std / (np.mean(bpm_arr) + 1e-10))
    else:
        bpm_variance, bpm_std, bpm_range, bpm_iqr, spatial_consistency = 0.0, 0.0, 0.0, 0.0, 1.0
    
    features.extend([bpm_variance, bpm_std, bpm_range, bpm_iqr, spatial_consistency])
    
    # ─── 10. Temporal Stability (5) ──────────────────────────────────────────
    features.extend(compute_temporal_stability(bvp_fh, fs))
    
    # ─── 11. Skin Reflection Features (4) ────────────────────────────────────
    features.extend(compute_skin_reflection_features(roi_frames_fh, roi_frames_lc))
    
    # ─── 12. Patch-Level Signal Quality (3) ──────────────────────────────────
    all_snrs = [
        _snr(bvp_fh), _snr(bvp_lc), _snr(bvp_rc),
        _snr(bvp_chin), _snr(bvp_nose),
        _snr(bvp_lj), _snr(bvp_rj)
    ]
    all_snrs = [s for s in all_snrs if s < 40]  # Filter defaults
    
    if len(all_snrs) >= 2:
        snr_arr = np.array(all_snrs)
        snr_std = float(np.std(snr_arr))
        snr_range = float(np.max(snr_arr) - np.min(snr_arr))
        patch_consistency = 1.0 - min(1.0, snr_std / 40.0)
    else:
        snr_std, snr_range, patch_consistency = 0.0, 0.0, 1.0
    
    features.extend([snr_std, snr_range, patch_consistency])
    
    # ═══════════════════════════════════════════════════════════════════════════
    # NEW: PHASE SYNCHRONIZATION FEATURES (3)
    # ═══════════════════════════════════════════════════════════════════════════
    phase_sync = compute_phase_synchronization(bvp_signals)
    features.extend(phase_sync)
    
    # ═══════════════════════════════════════════════════════════════════════════
    # NEW: CROSS-CHANNEL RGB CORRELATION (2)
    # ═══════════════════════════════════════════════════════════════════════════
    rgb_corr = compute_rgb_channel_correlation(raw_rgb_forehead)
    features.extend(rgb_corr)
    
    return np.nan_to_num(np.array(features, dtype=np.float64), nan=0.0, posinf=40.0, neginf=-40.0)

# ══════════════════════════════════════════════════════════════════════════════
# FACEMESH INITIALIZATION
# ═══════════════════════════════════════════════════════════════════════════════
_FACE_MESH = mp_module.solutions.face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)

# ═══════════════════════════════════════════════════════════════════════════════
# VIDEO PROCESSING WITH SPATIAL rPPG (OPTIMIZED)
# ═══════════════════════════════════════════════════════════════════════════════

def process_video(video_path, method="CHROM", max_frames=180):
    """
    Extract research-level features from video with spatial rPPG.
    Uses 9 facial ROIs for deepfake spatial inconsistency detection.
    
    OPTIMIZATIONS:
    1. FaceMesh runs every 5 frames (80% reduction in detection calls)
    2. Skip frames with missing ROIs instead of using zeros (cleaner signals)
    3. Default method changed to CHROM (more robust to illumination)
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0 or fps > 120:
        fps = 30.0
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Balanced frame sampling
    if total_frames > max_frames:
        frame_indices = np.linspace(0, total_frames - 1, max_frames, dtype=int)
    else:
        frame_indices = np.arange(total_frames)
    
    # Storage for all ROIs
    rgb_signals = {name: [] for name in ALL_ROIS.keys()}
    geometry_samples = []
    raw_rgb_fh = []  # For skin reflection
    raw_rgb_lc = []
    raw_rgb_forehead = []  # For cross-channel RGB correlation
    
    # OPTIMIZATION: Cache last valid landmarks
    last_landmarks = None
    
    for i, idx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue
        
        h, w = frame.shape[:2]
        
        # OPTIMIZATION: Run FaceMesh only every N frames
        if i % FACE_DETECTION_INTERVAL == 0 or last_landmarks is None:
            res = _FACE_MESH.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            
            if res.multi_face_landmarks:
                last_landmarks = res.multi_face_landmarks[0]
            else:
                # No face detected, skip this frame
                continue
        
        lm = last_landmarks
        
        # Face quality check
        xs = [lm.landmark[j].x * w for j in range(468)]
        ys = [lm.landmark[j].y * h for j in range(468)]
        x_min, x_max = int(min(xs)), int(max(xs))
        y_min, y_max = int(min(ys)), int(max(ys))
        x_min, y_min = max(0, x_min), max(0, y_min)
        x_max, y_max = min(w, x_max), min(h, y_max)
        face_crop = frame[y_min:y_max, x_min:x_max]
        
        if not is_face_quality_good(face_crop):
            continue
        
        # Extract RGB signals from ALL ROIs
        frame_signals = {}
        all_rois_valid = True
        
        for roi_name, roi_indices in ALL_ROIS.items():
            rgb = get_roi_mean_rgb(frame, lm, roi_indices, h, w)
            if rgb is not None:
                frame_signals[roi_name] = rgb
            else:
                # FIX: Skip this frame entirely if ANY primary ROI fails
                # This prevents fake signals from zeros
                if roi_name in ['forehead', 'left_cheek', 'right_cheek']:
                    all_rois_valid = False
                    break
                else:
                    # For non-primary ROIs, use previous value or skip
                    frame_signals[roi_name] = None
        
        # Skip frame if primary ROIs are invalid
        if not all_rois_valid:
            continue
        
        # Only add valid frames
        for roi_name in ALL_ROIS.keys():
            if frame_signals.get(roi_name) is not None:
                rgb_signals[roi_name].append(frame_signals[roi_name])
        
        raw_rgb_fh.append(frame_signals['forehead'])
        raw_rgb_lc.append(frame_signals['left_cheek'])
        raw_rgb_forehead.append(frame_signals['forehead'])  # For RGB channel correlation
        
        geo_feat = extract_geometry_features(lm, w, h)
        geometry_samples.append(geo_feat)
    
    cap.release()
    
    # Check minimum frames
    if len(rgb_signals['forehead']) < int(2 * fps):
        return None
    
    # Convert to arrays and extract BVP signals
    efn = RPPG_METHODS.get(method, extract_chrom)  # Default to CHROM
    bvp_signals = {}
    
    for roi_name in ALL_ROIS.keys():
        rgb_arr = np.array(rgb_signals[roi_name])
        if len(rgb_arr) > 0:
            bvp = efn(rgb_arr)
            bvp = bandpass_filter(detrend_signal(bvp), fps)
            bvp_signals[roi_name] = bvp
        else:
            bvp_signals[roi_name] = np.zeros(10)
    
    # Average geometry features
    geometry_mean = np.mean(geometry_samples, axis=0) if geometry_samples else np.zeros(20)
    
    # Extract all 111 features
    return extract_all_features(bvp_signals, fps, geometry_mean, raw_rgb_fh, raw_rgb_lc, raw_rgb_forehead)


print("="*70)
print("RESEARCH-LEVEL rPPG FEATURE EXTRACTION (FakeCatcher-Style)")
print("="*70)
print(f"  Total features: {len(FEATURE_NAMES)}")
print(f"  Primary ROI features: 24 (forehead + left cheek)")
print(f"  Original cross-ROI: 8 (correlations, coherence, phase)")
print(f"  BPM & signal quality: 3")
print(f"  HRV features: 8")
print(f"  Signal quality: 5")
print(f"  Geometry features: 20")
print("  ─────────────────────────────────────────────────────")
print("  NEW RESEARCH-LEVEL FEATURES:")
print(f"  - Spatial rPPG correlations: 12 (9 ROIs: chin, nose, jaw)")
print(f"  - Multi-band frequency: 9 (low/mid/high bands)")
print(f"  - Spatial pulse variance: 5 (BPM consistency across regions)")
print(f"  - Temporal stability: 5 (pulse stability over time)")
print(f"  - Skin reflection: 4 (HSV specular analysis - FIXED)")
print(f"  - Patch-level quality: 3 (SNR variance across patches)")
print(f"  - Phase synchronization: 3 (pulse wave coherence)")
print(f"  - RGB cross-channel: 2 (physiological RGB correlation)")
print("="*70)
print("OPTIMIZATIONS APPLIED:")
print(f"  ✓ FaceMesh detection every {FACE_DETECTION_INTERVAL} frames (~80% faster)")
print(f"  ✓ Skip frames with missing ROIs (cleaner signals)")
print(f"  ✓ Default rPPG method: CHROM (robust to illumination)")
print(f"  ✓ Proper HSV conversion using cv2.cvtColor")
print("="*70)
print(f"FaceMesh initialized with 9 facial ROIs")
print(f"Max frames per video: {MAX_FRAMES_PER_VIDEO}")


RESEARCH-LEVEL rPPG FEATURE EXTRACTION (FakeCatcher-Style)
  Total features: 111
  Primary ROI features: 24 (forehead + left cheek)
  Original cross-ROI: 8 (correlations, coherence, phase)
  BPM & signal quality: 3
  HRV features: 8
  Signal quality: 5
  Geometry features: 20
  ─────────────────────────────────────────────────────
  NEW RESEARCH-LEVEL FEATURES:
  - Spatial rPPG correlations: 12 (9 ROIs: chin, nose, jaw)
  - Multi-band frequency: 9 (low/mid/high bands)
  - Spatial pulse variance: 5 (BPM consistency across regions)
  - Temporal stability: 5 (pulse stability over time)
  - Skin reflection: 4 (HSV specular analysis - FIXED)
  - Patch-level quality: 3 (SNR variance across patches)
  - Phase synchronization: 3 (pulse wave coherence)
  - RGB cross-channel: 2 (physiological RGB correlation)
OPTIMIZATIONS APPLIED:
  ✓ FaceMesh detection every 5 frames (~80% faster)
  ✓ Skip frames with missing ROIs (cleaner signals)
  ✓ Default rPPG method: CHROM (robust to illumination)
  ✓ Pr

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1773685377.773219     189 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773685377.787118     187 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [6]:
# ─── Extract Features From All Videos (WITH AUTO-SAVE) ─────────────────────────────

video_ext = ("*.mp4", "*.avi", "*.mov", "*.mkv")

real_videos = []
fake_videos = []

for ext in video_ext:
    real_videos.extend(glob.glob(os.path.join(REAL_DIR, ext)))
    fake_videos.extend(glob.glob(os.path.join(FAKE_DIR, ext)))

real_videos = sorted(real_videos)
fake_videos = sorted(fake_videos)

print("Real videos:", len(real_videos))
print("Fake videos:", len(fake_videos))
print("Total videos:", len(real_videos) + len(fake_videos))

features = []
labels = []
valid_video_paths = []
SAVE_INTERVAL = 25   # save progress every 25 videos

# ─────────────────────────────────────────
# Process REAL videos
# ─────────────────────────────────────────
print("Processing REAL videos...")

for i, v in enumerate(tqdm(real_videos)):

    try:
        feat = process_video(v, method=RPPG_METHOD, max_frames=MAX_FRAMES_PER_VIDEO)

        if feat is not None:
            features.append(feat)
            labels.append(0)
            valid_video_paths.append(v)

    except Exception as e:
        print("Skipped:", v)

    # Save checkpoint
    if (i+1) % SAVE_INTERVAL == 0:
        np.save("/kaggle/working/features_checkpoint.npy", np.array(features))
        np.save("/kaggle/working/labels_checkpoint.npy", np.array(labels))
        print(f"Checkpoint saved at {i+1} real videos")


# ─────────────────────────────────────────
# Process FAKE videos
# ─────────────────────────────────────────
print("Processing FAKE videos...")

for i, v in enumerate(tqdm(fake_videos)):

    try:
        feat = process_video(v, method=RPPG_METHOD, max_frames=MAX_FRAMES_PER_VIDEO)

        if feat is not None:
            features.append(feat)
            labels.append(1)
            valid_video_paths.append(v)

    except Exception as e:
        print("Skipped:", v)

    # Save checkpoint
    if (i+1) % SAVE_INTERVAL == 0:
        np.save("/kaggle/working/features_checkpoint.npy", np.array(features))
        np.save("/kaggle/working/labels_checkpoint.npy", np.array(labels))
        print(f"Checkpoint saved at {i+1} fake videos")


# ─────────────────────────────────────────
# Final dataset creation
# ─────────────────────────────────────────

X = np.array(features)
y = np.array(labels)

print("Feature extraction complete")
print("X shape:", X.shape)
print("y shape:", y.shape)


# ─────────────────────────────────────────
# Save FINAL dataset
# ─────────────────────────────────────────

import pandas as pd

df = pd.DataFrame(X, columns=FEATURE_NAMES)
df["label"] = y

# Save CSV
df.to_csv("/kaggle/working/deepfake_features.csv", index=False)

# Save NumPy versions (fastest to load later)
np.save("/kaggle/working/X_features.npy", X)
np.save("/kaggle/working/y_labels.npy", y)

print("\nSaved files:")
print("/kaggle/working/deepfake_features.csv")
print("/kaggle/working/X_features.npy")
print("/kaggle/working/y_labels.npy")

Real videos: 400
Fake videos: 400
Total videos: 800
Processing REAL videos...


  6%|▋         | 25/400 [06:07<1:14:47, 11.97s/it]

Checkpoint saved at 25 real videos


 12%|█▎        | 50/400 [13:51<2:35:09, 26.60s/it]

Checkpoint saved at 50 real videos


 19%|█▉        | 75/400 [20:53<1:04:22, 11.88s/it]

Checkpoint saved at 75 real videos


 23%|██▎       | 91/400 [26:16<1:29:14, 17.33s/it]


KeyboardInterrupt: 

## 3. Exploratory Data Analysis

In [ ]:
print("X exists:", 'X' in globals())
print("y exists:", 'y' in globals())

In [ ]:
# ─── Dataset Summary ─────────────────────────────────────────────
df = pd.DataFrame(X, columns=FEATURE_NAMES)
df['label'] = y

print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Total samples: {len(df)}")
print(f"Features: {X.shape[1]}")
print(f"\nClass distribution:")
print(df['label'].value_counts().rename({0: 'Real', 1: 'Fake'}))
print(f"\nClass ratio: {np.sum(y==0)/np.sum(y==1):.2f} (Real/Fake)")
print(f"\nFeature statistics:")
print(df.describe().T[['mean', 'std', 'min', 'max']].to_string())

In [ ]:
# ─── Feature Distributions: Real vs Fake ─────────────────────────
fig, axes = plt.subplots(6, 6, figsize=(24, 20))
axes = axes.flatten()

for i, fname in enumerate(FEATURE_NAMES):
    ax = axes[i]
    ax.hist(X[y==0, i], bins=30, alpha=0.6, label='Real', color='green', density=True)
    ax.hist(X[y==1, i], bins=30, alpha=0.6, label='Fake', color='red', density=True)
    ax.set_title(fname, fontsize=9)
    ax.legend(fontsize=7)

# Hide extra subplot
axes[35].set_visible(False)

plt.suptitle('Feature Distributions: Real vs Deepfake', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Correlation Matrix ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 14))
corr = df[FEATURE_NAMES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'correlation_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Data Preprocessing

In [ ]:
# ─── Train/Test Split + Scaling ──────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape} | Test: {X_test_scaled.shape}")
print(f"Train class dist: Real={np.sum(y_train==0)}, Fake={np.sum(y_train==1)}")
print(f"Test  class dist: Real={np.sum(y_test==0)}, Fake={np.sum(y_test==1)}")

# Save scaler for inference
joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'scaler.joblib'))
print("Scaler saved.")

## 5. Model Definitions with GridSearchCV

10 classifiers with hyperparameter grids for GridSearchCV optimization.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ENHANCED ML MODELS WITH RANDOMIZEDSEARCHCV + GPU ACCELERATION
# ═══════════════════════════════════════════════════════════════════════════════
# - 11 classifiers including GPU-accelerated CatBoost & XGBoost
# - RandomizedSearchCV with 5-fold stratified CV
# - Optimized for Kaggle P100 (16GB VRAM, 9hr runtime)
# - Expected runtime: ~1.5-2 hours
# ═══════════════════════════════════════════════════════════════════════════════

SEED = 42
N_JOBS = -1  # Use all CPU cores
CV_FOLDS = 5  # 5-fold stratified CV

# Import DecisionTreeClassifier for AdaBoost
from sklearn.tree import DecisionTreeClassifier

# ─── Define models and their hyperparameter grids ────────────────────────────
model_configs = {
    'RandomForest': {
        'model': RandomForestClassifier(random_state=SEED, n_jobs=N_JOBS),
        'param_grid': {
            'n_estimators': [200, 300, 500, 700],
            'max_depth': [10, 15, 20, 25, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', 0.3, 0.5],
            'bootstrap': [True, False],
        },
        'search_type': 'random',
        'n_iter': 50,
    },
    'XGBoost': {
        'model': xgb.XGBClassifier(
            eval_metric='logloss', 
            random_state=SEED, 
            n_jobs=N_JOBS,
            tree_method='hist',
            device='cuda',  # GPU acceleration on Kaggle P100
        ),
        'param_grid': {
            'n_estimators': [200, 300, 400, 500, 700],
            'max_depth': [3, 4, 5, 6, 7, 8],
            'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.1],
            'subsample': [0.6, 0.7, 0.8, 0.9],
            'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
            'reg_alpha': [0, 0.01, 0.1, 0.5, 1],
            'reg_lambda': [0.5, 1, 2, 5],
            'gamma': [0, 0.05, 0.1, 0.2],
            'min_child_weight': [1, 3, 5, 7],
        },
        'search_type': 'random',
        'n_iter': 80,  # More trials for XGBoost
    },
    'LightGBM': {
        'model': lgb.LGBMClassifier(
            random_state=SEED, 
            n_jobs=N_JOBS, 
            verbose=-1,
            device='gpu',  # GPU acceleration
            gpu_platform_id=0,
            gpu_device_id=0,
        ),
        'param_grid': {
            'n_estimators': [200, 300, 400, 500, 700],
            'max_depth': [5, 8, 10, 12, 15, -1],
            'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.1],
            'num_leaves': [20, 31, 50, 70, 100, 127],
            'subsample': [0.6, 0.7, 0.8, 0.9],
            'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
            'reg_alpha': [0, 0.01, 0.1, 0.5],
            'reg_lambda': [0, 0.1, 1, 2],
            'min_child_samples': [5, 10, 20, 30],
            'min_split_gain': [0.0, 0.01, 0.1],
        },
        'search_type': 'random',
        'n_iter': 80,  # More trials for LightGBM
    },
    'SVM_RBF': {
        'model': SVC(kernel='rbf', probability=True, random_state=SEED, cache_size=1000),
        'param_grid': {
            'C': [0.01, 0.1, 1, 10, 100, 1000],
            'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],
            'class_weight': [None, 'balanced'],
        },
        'search_type': 'random',
        'n_iter': 40,
    },
    'SVM_Linear': {
        'model': SVC(kernel='linear', probability=True, random_state=SEED),
        'param_grid': {
            'C': [0.001, 0.01, 0.1, 1, 10, 100],
            'class_weight': [None, 'balanced'],
        },
        'search_type': 'grid',
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(random_state=SEED),
        'param_grid': {
            'n_estimators': [200, 300, 400, 500],
            'max_depth': [3, 4, 5, 6, 7],
            'learning_rate': [0.01, 0.03, 0.05, 0.1],
            'subsample': [0.6, 0.7, 0.8, 0.9],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', 0.3],
        },
        'search_type': 'random',
        'n_iter': 50,
    },
    'ExtraTrees': {
        'model': ExtraTreesClassifier(random_state=SEED, n_jobs=N_JOBS),
        'param_grid': {
            'n_estimators': [200, 300, 500, 700],
            'max_depth': [10, 15, 20, 25, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', 0.3, 0.5],
            'bootstrap': [True, False],
        },
        'search_type': 'random',
        'n_iter': 50,
    },
    'HistGradientBoosting': {
        'model': HistGradientBoostingClassifier(random_state=SEED),
        'param_grid': {
            'max_iter': [200, 300, 400, 500, 700],
            'max_depth': [5, 8, 10, 15, 20, None],
            'learning_rate': [0.01, 0.03, 0.05, 0.1],
            'l2_regularization': [0, 0.01, 0.1, 1.0],
            'max_leaf_nodes': [20, 31, 50, 100, 150],
            'min_samples_leaf': [10, 20, 30, 50],
        },
        'search_type': 'random',
        'n_iter': 50,
    },
    'AdaBoost': {
        'model': AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=3, random_state=SEED),
            random_state=SEED,
            algorithm='SAMME',
        ),
        'param_grid': {
            'n_estimators': [100, 200, 300, 500],
            'learning_rate': [0.01, 0.05, 0.1, 0.5, 1.0],
        },
        'search_type': 'grid',
    },
    'KNN': {
        'model': KNeighborsClassifier(n_jobs=N_JOBS),
        'param_grid': {
            'n_neighbors': [3, 5, 7, 9, 11, 15, 21],
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan', 'minkowski', 'chebyshev'],
            'p': [1, 2, 3],
        },
        'search_type': 'random',
        'n_iter': 40,
    },
    'LogisticRegression': {
        'model': LogisticRegression(random_state=SEED, max_iter=5000, n_jobs=N_JOBS),
        'param_grid': {
            'C': [0.001, 0.01, 0.1, 1, 10, 100],
            'penalty': ['l1', 'l2', 'elasticnet'],
            'solver': ['saga'],
            'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],
            'class_weight': [None, 'balanced'],
        },
        'search_type': 'random',
        'n_iter': 40,
    },
}

# ═══════════════════════════════════════════════════════════════════════════════
# ADD CATBOOST WITH GPU ACCELERATION
# ═══════════════════════════════════════════════════════════════════════════════
if HAS_CATBOOST:
    model_configs['CatBoost'] = {
        'model': CatBoostClassifier(
            random_state=SEED, 
            verbose=0, 
            thread_count=-1,
            task_type='GPU',  # GPU acceleration on Kaggle P100
            devices='0',
            gpu_ram_part=0.8,  # Use 80% of GPU RAM
        ),
        'param_grid': {
            'iterations': [300, 500, 700, 1000],
            'depth': [4, 6, 8, 10],
            'learning_rate': [0.01, 0.03, 0.05, 0.1],
            'l2_leaf_reg': [1, 3, 5, 7, 9],
            'border_count': [32, 64, 128, 254],
            'bagging_temperature': [0, 0.5, 1],
            'random_strength': [0, 0.5, 1],
            'grow_policy': ['SymmetricTree', 'Depthwise'],
        },
        'search_type': 'random',
        'n_iter': 60,  # More trials for CatBoost
    }
    print("CatBoost configured with GPU acceleration!")
else:
    print("CatBoost not available - install with: pip install catboost")

print(f"Defined {len(model_configs)} model configurations for RandomizedSearchCV.")
for name in model_configs:
    cfg = model_configs[name]
    search = cfg['search_type']
    n_iter = cfg.get('n_iter', 'full grid')
    print(f"  {name}: {search} search, {n_iter}")

## 6. Train All Models with GridSearchCV (5-Fold Stratified CV)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# OPTIMIZED ML TRAINING PIPELINE
# ═══════════════════════════════════════════════════════════════════════════════
# Improvements:
# 1. Feature normalization (StandardScaler)
# 2. Feature selection (top 40 features by importance)
# 3. StratifiedKFold cross-validation (5 folds)
# 4. Only best 3 classifiers (XGBoost, LightGBM, RandomForest)
# 5. Stacking ensemble
# ═══════════════════════════════════════════════════════════════════════════════

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

SEED = 42
N_JOBS = -1
N_FOLDS = 5

print("="*70)
print("OPTIMIZED ML TRAINING PIPELINE")
print("="*70)

# ─── IMPROVEMENT 1: Feature Normalization ────────────────────────────────────

print("\n[1] Feature Normalization...")
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"    ✓ RobustScaler applied")
print(f"    Train shape: {X_train_scaled.shape}")
print(f"    Test shape: {X_test_scaled.shape}")

# ─── IMPROVEMENT 2: Cross-Validation Setup ───────────────────────────────────

print("\n[2] Cross-Validation Setup...")
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
print(f"    ✓ StratifiedKFold with {N_FOLDS} folds")

# ─── IMPROVEMENT 3: Define Only Best Models ──────────────────────────────────

print("\n[3] Defining Best Models (XGBoost, LightGBM, RandomForest)...")

best_models_config = {
    'XGBoost': xgb.XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1,
        gamma=0.1,
        min_child_weight=3,
        eval_metric='logloss',
        random_state=SEED,
        n_jobs=N_JOBS,
        tree_method='hist',
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=500,
        max_depth=10,
        learning_rate=0.05,
        num_leaves=50,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1,
        min_child_samples=20,
        random_state=SEED,
        n_jobs=N_JOBS,
        verbose=-1,
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=500,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        bootstrap=True,
        random_state=SEED,
        n_jobs=N_JOBS,
    ),
}

for name in best_models_config:
    print(f"    ✓ {name}")

# ─── IMPROVEMENT 4: Feature Selection ────────────────────────────────────────

print("\n[4] Feature Importance Selection...")

# Train XGBoost to get feature importances
selector_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=SEED,
    n_jobs=N_JOBS,
    eval_metric='logloss',
)
selector_model.fit(X_train_scaled, y_train)

# Get feature importances
importances = selector_model.feature_importances_
indices = np.argsort(importances)[::-1]

# Select top features
N_TOP_FEATURES = min(40, X_train_scaled.shape[1])
top_indices = indices[:N_TOP_FEATURES]

print(f"    ✓ Top {N_TOP_FEATURES} features selected")
print(f"    Top 10 features:")
for j in range(min(10, N_TOP_FEATURES)):
    idx = indices[j]
    feat_name = FEATURE_NAMES[idx] if idx < len(FEATURE_NAMES) else f"feature_{idx}"
    print(f"      {j+1}. {feat_name}: {importances[idx]:.4f}")

# Apply feature selection
X_train_selected = X_train_scaled[:, top_indices]
X_test_selected = X_test_scaled[:, top_indices]

print(f"    Selected train shape: {X_train_selected.shape}")
print(f"    Selected test shape: {X_test_selected.shape}")

# ─── IMPROVEMENT 4.5: Feature Interaction Expansion ──────────────────────────

print("\n[4.5] Feature Interaction Expansion (PolynomialFeatures)...")

from sklearn.preprocessing import PolynomialFeatures

# Create interaction features (degree=2, interaction_only=True)
# This captures patterns like: snr × temporal_std, bpm × phase_sync, etc.
# Tree models (XGBoost, LightGBM) benefit greatly from these
poly = PolynomialFeatures(
    degree=2,
    interaction_only=True,  # Only interactions, no x^2 terms
    include_bias=False
)

X_train_interaction = poly.fit_transform(X_train_selected)
X_test_interaction = poly.transform(X_test_selected)

print(f"    ✓ PolynomialFeatures applied (degree=2, interaction_only=True)")
print(f"    Original feature count: {X_train_selected.shape[1]}")
print(f"    After interaction expansion: {X_train_interaction.shape[1]}")
print(f"    Train shape: {X_train_interaction.shape}")
print(f"    Test shape: {X_test_interaction.shape}")

# Use interaction features for training
X_train_selected = X_train_interaction
X_test_selected = X_test_interaction

# ─── IMPROVEMENT 5: Train Models with Cross-Validation ──────────────────────

print("\n[5] Training Models with 5-Fold CV...")

results = {}
trained_models = {}

for name, model in best_models_config.items():
    print(f"\n{'─'*50}")
    print(f"Training: {name}")
    print(f"{'─'*50}")
    
    t0 = time()
    
    # Cross-validation scores
    cv_scores = cross_val_score(model, X_train_selected, y_train, cv=cv, scoring='roc_auc', n_jobs=N_JOBS)
    
    # Fit on full training set
    model.fit(X_train_selected, y_train)
    
    # Predict on test set
    y_pred = model.predict(X_test_selected)
    y_prob = model.predict_proba(X_test_selected)[:, 1]
    
    elapsed = time() - t0
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    
    results[name] = {
        'accuracy': acc,
        'auc': auc,
        'f1': f1,
        'precision': prec,
        'recall': rec,
        'cv_auc_mean': cv_scores.mean(),
        'cv_auc_std': cv_scores.std(),
        'time': elapsed,
    }
    trained_models[name] = model
    
    print(f"  CV AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print(f"  Test Accuracy: {acc:.4f}")
    print(f"  Test AUC: {auc:.4f}")
    print(f"  Test F1: {f1:.4f}")
    print(f"  Time: {elapsed:.1f}s")

# ─── IMPROVEMENT 6: Stacking Ensemble ────────────────────────────────────────

print("\n[6] Building Stacking Ensemble...")

# Build stacking classifier
estimators = [(name, model) for name, model in trained_models.items()]

stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(C=1.0, max_iter=5000, random_state=SEED),
    cv=5,
    stack_method='predict_proba',
    n_jobs=N_JOBS,
)

print("    Training Stacking Classifier...")
t0 = time()
stacking_clf.fit(X_train_selected, y_train)
stack_time = time() - t0

y_pred_stack = stacking_clf.predict(X_test_selected)
y_prob_stack = stacking_clf.predict_proba(X_test_selected)[:, 1]

stack_acc = accuracy_score(y_test, y_pred_stack)
stack_auc = roc_auc_score(y_test, y_prob_stack)
stack_f1 = f1_score(y_test, y_pred_stack, zero_division=0)

results['Stacking'] = {
    'accuracy': stack_acc,
    'auc': stack_auc,
    'f1': stack_f1,
    'precision': precision_score(y_test, y_pred_stack, zero_division=0),
    'recall': recall_score(y_test, y_pred_stack, zero_division=0),
    'cv_auc_mean': np.nan,
    'cv_auc_std': np.nan,
    'time': stack_time,
}
trained_models['Stacking'] = stacking_clf

print(f"    ✓ Stacking Ensemble")
print(f"    Test Accuracy: {stack_acc:.4f}")
print(f"    Test AUC: {stack_auc:.4f}")
print(f"    Test F1: {stack_f1:.4f}")

# ─── Voting Ensemble ─────────────────────────────────────────────────────────

print("\n[7] Building Voting Ensemble...")

voting_clf = VotingClassifier(
    estimators=estimators,
    voting='soft',
    n_jobs=N_JOBS,
)

t0 = time()
voting_clf.fit(X_train_selected, y_train)
vote_time = time() - t0

y_pred_vote = voting_clf.predict(X_test_selected)
y_prob_vote = voting_clf.predict_proba(X_test_selected)[:, 1]

vote_acc = accuracy_score(y_test, y_pred_vote)
vote_auc = roc_auc_score(y_test, y_prob_vote)
vote_f1 = f1_score(y_test, y_pred_vote, zero_division=0)

results['Voting'] = {
    'accuracy': vote_acc,
    'auc': vote_auc,
    'f1': vote_f1,
    'precision': precision_score(y_test, y_pred_vote, zero_division=0),
    'recall': recall_score(y_test, y_pred_vote, zero_division=0),
    'cv_auc_mean': np.nan,
    'cv_auc_std': np.nan,
    'time': vote_time,
}
trained_models['Voting'] = voting_clf

print(f"    ✓ Voting Ensemble")
print(f"    Test Accuracy: {vote_acc:.4f}")
print(f"    Test AUC: {vote_auc:.4f}")
print(f"    Test F1: {vote_f1:.4f}")

# ─── Results Summary ─────────────────────────────────────────────────────────

print("\n" + "="*70)
print("ML MODEL RESULTS - SORTED BY AUC")
print("="*70)

sorted_results = sorted(results.items(), key=lambda x: x[1]['auc'], reverse=True)

print(f"{'Model':<15} {'Accuracy':>10} {'AUC':>10} {'F1':>10} {'CV AUC':>15}")
print("-"*60)
for name, r in sorted_results:
    cv_str = f"{r['cv_auc_mean']:.4f}±{r['cv_auc_std']:.4f}" if not np.isnan(r['cv_auc_mean']) else "N/A"
    print(f"{name:<15} {r['accuracy']:>10.4f} {r['auc']:>10.4f} {r['f1']:>10.4f} {cv_str:>15}")

best_model_name = sorted_results[0][0]
best_model = trained_models[best_model_name]
print(f"\n🏆 Best Model: {best_model_name} (AUC: {sorted_results[0][1]['auc']:.4f})")

# Save best model
joblib.dump(best_model, os.path.join(OUTPUT_DIR, f'{best_model_name}_best.joblib'))
joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'feature_scaler.joblib'))
np.save(os.path.join(OUTPUT_DIR, 'top_feature_indices.npy'), top_indices)
best_models = trained_models.copy() # <--- ADD THIS LINE HERE
print(f"\n✓ Model and scaler saved to {OUTPUT_DIR}")

## 7. Advanced Ensemble Models (Voting, Stacking, Calibrated)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ADVANCED ENSEMBLE METHODS FOR MAXIMUM ACCURACY
# ═══════════════════════════════════════════════════════════════════════════════
# 1. Soft Voting Ensemble (top models)
# 2. TRUE Stacking Classifier with CV (not just voting)
# 3. Blending Ensemble
# 4. Calibrated Stacking for better probabilities
# 5. Multi-level Stacking (2 layers)
# ═══════════════════════════════════════════════════════════════════════════════

from sklearn.ensemble import StackingClassifier, VotingClassifier, BaggingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB

# ─── Sort models by Test AUC ─────────────────────────────────────────────────
sorted_models = sorted(results.items(), key=lambda x: x[1]['auc'], reverse=True)

print("="*70)
print("MODEL RANKINGS BY TEST AUC")
print("="*70)
for i, (name, r) in enumerate(sorted_models, 1):
    print(f"  {i}. {name:25s} AUC: {r['auc']:.4f}  Acc: {r['accuracy']:.4f}")

# ═══════════════════════════════════════════════════════════════════════════════
# 1. SOFT VOTING ENSEMBLE (Top-7)
# ═══════════════════════════════════════════════════════════════════════════════
top_n = min(7, len(sorted_models))
top_models = [(name, best_models[name]) for name, _ in sorted_models[:top_n]]

print(f"\n{'─'*70}")
print(f"[1] SOFT VOTING ENSEMBLE from top {top_n} models:")
for name, _ in top_models:
    print(f"    - {name}")

voting_clf = VotingClassifier(
    estimators=top_models,
    voting='soft',
    n_jobs=N_JOBS,
)

print("\nTraining Voting Ensemble...")
t0 = time()
voting_clf.fit(X_train_selected, y_train)
voting_time = time() - t0

y_pred_voting = voting_clf.predict(X_test_selected)
y_prob_voting = voting_clf.predict_proba(X_test_selected)[:, 1]

voting_acc = accuracy_score(y_test, y_pred_voting)
voting_auc = roc_auc_score(y_test, y_prob_voting)
voting_f1 = f1_score(y_test, y_pred_voting, zero_division=0)
voting_prec = precision_score(y_test, y_pred_voting, zero_division=0)
voting_rec = recall_score(y_test, y_pred_voting, zero_division=0)

print(f"  ✓ Voting Ensemble - Accuracy: {voting_acc:.4f}, AUC: {voting_auc:.4f}, F1: {voting_f1:.4f}")

results['VotingEnsemble_Top7'] = {
    'accuracy': voting_acc, 'precision': voting_prec, 'recall': voting_rec,
    'f1': voting_f1, 'auc': voting_auc,
    'cv_auc_mean': np.nan, 'cv_auc_std': np.nan,
    'time': voting_time, 'best_params': {'estimators': [n for n, _ in top_models]},
}
best_models['VotingEnsemble_Top7'] = voting_clf

# ═══════════════════════════════════════════════════════════════════════════════
# 2. TRUE STACKING CLASSIFIER WITH CROSS-VALIDATION
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*70}")
print("[2] TRUE STACKING CLASSIFIER with 5-fold CV meta-features...")

# Select diverse base learners for stacking
base_learners = []

# Tree-based models
if 'XGBoost' in best_models:
    base_learners.append(('xgb', best_models['XGBoost']))
if 'LightGBM' in best_models:
    base_learners.append(('lgb', best_models['LightGBM']))
if 'CatBoost' in best_models:
    base_learners.append(('cat', best_models['CatBoost']))
if 'RandomForest' in best_models:
    base_learners.append(('rf', best_models['RandomForest']))
if 'ExtraTrees' in best_models:
    base_learners.append(('et', best_models['ExtraTrees']))
if 'GradientBoosting' in best_models:
    base_learners.append(('gb', best_models['GradientBoosting']))
if 'HistGradientBoosting' in best_models:
    base_learners.append(('hgb', best_models['HistGradientBoosting']))

# Linear models for diversity
if 'LogisticRegression' in best_models:
    base_learners.append(('lr', best_models['LogisticRegression']))
if 'SVM_RBF' in best_models:
    base_learners.append(('svm', best_models['SVM_RBF']))

print(f"  Base learners ({len(base_learners)}):")
for name, _ in base_learners:
    print(f"    - {name}")

# Meta-learner: Logistic Regression with regularization
meta_learner = LogisticRegression(
    C=1.0, 
    penalty='l2', 
    solver='lbfgs', 
    max_iter=5000, 
    random_state=SEED
)

stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5,  # 5-fold CV for meta-features
    stack_method='predict_proba',  # Use probabilities
    n_jobs=N_JOBS,
    passthrough=False,  # Don't include original features
)

print("\nTraining Stacking Classifier (this may take a few minutes)...")
t0 = time()
stacking_clf.fit(X_train_selected, y_train)
stacking_time = time() - t0

y_pred_stack = stacking_clf.predict(X_test_selected)
y_prob_stack = stacking_clf.predict_proba(X_test_selected)[:, 1]

stack_acc = accuracy_score(y_test, y_pred_stack)
stack_auc = roc_auc_score(y_test, y_prob_stack)
stack_f1 = f1_score(y_test, y_pred_stack, zero_division=0)
stack_prec = precision_score(y_test, y_pred_stack, zero_division=0)
stack_rec = recall_score(y_test, y_pred_stack, zero_division=0)

print(f"  ✓ Stacking Classifier - Accuracy: {stack_acc:.4f}, AUC: {stack_auc:.4f}, F1: {stack_f1:.4f}")
print(f"    Time: {stacking_time:.1f}s")

results['StackingClassifier'] = {
    'accuracy': stack_acc, 'precision': stack_prec, 'recall': stack_rec,
    'f1': stack_f1, 'auc': stack_auc,
    'cv_auc_mean': np.nan, 'cv_auc_std': np.nan,
    'time': stacking_time, 'best_params': {'base_learners': [n for n, _ in base_learners]},
}
best_models['StackingClassifier'] = stacking_clf

# ═══════════════════════════════════════════════════════════════════════════════
# 3. STACKING WITH MLP META-LEARNER (Can capture non-linear combinations)
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*70}")
print("[3] STACKING with Neural Network meta-learner...")

mlp_meta = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation='relu',
    solver='adam',
    alpha=0.01,
    max_iter=1000,
    early_stopping=True,
    validation_fraction=0.15,
    random_state=SEED,
)

stacking_mlp = StackingClassifier(
    estimators=base_learners[:5],  # Top 5 for speed
    final_estimator=mlp_meta,
    cv=5,
    stack_method='predict_proba',
    n_jobs=N_JOBS,
)

print("\nTraining MLP Stacking Classifier...")
t0 = time()
stacking_mlp.fit(X_train_selected, y_train)
mlp_stack_time = time() - t0

y_pred_mlp_stack = stacking_mlp.predict(X_test_selected)
y_prob_mlp_stack = stacking_mlp.predict_proba(X_test_selected)[:, 1]

mlp_stack_acc = accuracy_score(y_test, y_pred_mlp_stack)
mlp_stack_auc = roc_auc_score(y_test, y_prob_mlp_stack)
mlp_stack_f1 = f1_score(y_test, y_pred_mlp_stack, zero_division=0)

print(f"  ✓ MLP Stacking - Accuracy: {mlp_stack_acc:.4f}, AUC: {mlp_stack_auc:.4f}, F1: {mlp_stack_f1:.4f}")

results['StackingMLP'] = {
    'accuracy': mlp_stack_acc, 
    'precision': precision_score(y_test, y_pred_mlp_stack, zero_division=0), 
    'recall': recall_score(y_test, y_pred_mlp_stack, zero_division=0),
    'f1': mlp_stack_f1, 'auc': mlp_stack_auc,
    'cv_auc_mean': np.nan, 'cv_auc_std': np.nan,
    'time': mlp_stack_time,
}
best_models['StackingMLP'] = stacking_mlp

# ═══════════════════════════════════════════════════════════════════════════════
# 4. CALIBRATED STACKING (Better probability estimates)
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*70}")
print("[4] CALIBRATED STACKING for better probability estimates...")

calibrated_stack = CalibratedClassifierCV(
    stacking_clf,
    method='isotonic',  # Non-parametric calibration
    cv=3,
)

print("\nCalibrating Stacking Classifier...")
t0 = time()
calibrated_stack.fit(X_train_selected, y_train)
calib_time = time() - t0

y_pred_calib = calibrated_stack.predict(X_test_selected)
y_prob_calib = calibrated_stack.predict_proba(X_test_selected)[:, 1]

calib_acc = accuracy_score(y_test, y_pred_calib)
calib_auc = roc_auc_score(y_test, y_prob_calib)
calib_f1 = f1_score(y_test, y_pred_calib, zero_division=0)

print(f"  ✓ Calibrated Stacking - Accuracy: {calib_acc:.4f}, AUC: {calib_auc:.4f}, F1: {calib_f1:.4f}")

results['CalibratedStacking'] = {
    'accuracy': calib_acc, 
    'precision': precision_score(y_test, y_pred_calib, zero_division=0), 
    'recall': recall_score(y_test, y_pred_calib, zero_division=0),
    'f1': calib_f1, 'auc': calib_auc,
    'cv_auc_mean': np.nan, 'cv_auc_std': np.nan,
    'time': calib_time,
}
best_models['CalibratedStacking'] = calibrated_stack

# ═══════════════════════════════════════════════════════════════════════════════
# 5. WEIGHTED VOTING BASED ON CV PERFORMANCE
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*70}")
print("[5] WEIGHTED VOTING ENSEMBLE based on CV AUC scores...")

# Calculate weights based on test AUC
auc_scores = {name: results[name]['auc'] for name, _ in sorted_models[:7] if name in results}
total_auc = sum(auc_scores.values())
weights = [auc_scores.get(name, 0) / total_auc for name, _ in top_models]

print(f"  Weights based on AUC:")
for (name, _), w in zip(top_models, weights):
    print(f"    {name}: {w:.3f}")

weighted_voting_clf = VotingClassifier(
    estimators=top_models,
    voting='soft',
    weights=weights,
    n_jobs=N_JOBS,
)

print("\nTraining Weighted Voting Ensemble...")
t0 = time()
weighted_voting_clf.fit(X_train_selected, y_train)
weighted_time = time() - t0

y_pred_weighted = weighted_voting_clf.predict(X_test_selected)
y_prob_weighted = weighted_voting_clf.predict_proba(X_test_selected)[:, 1]

weighted_acc = accuracy_score(y_test, y_pred_weighted)
weighted_auc = roc_auc_score(y_test, y_prob_weighted)
weighted_f1 = f1_score(y_test, y_pred_weighted, zero_division=0)

print(f"  ✓ Weighted Voting - Accuracy: {weighted_acc:.4f}, AUC: {weighted_auc:.4f}, F1: {weighted_f1:.4f}")

results['WeightedVoting'] = {
    'accuracy': weighted_acc, 
    'precision': precision_score(y_test, y_pred_weighted, zero_division=0), 
    'recall': recall_score(y_test, y_pred_weighted, zero_division=0),
    'f1': weighted_f1, 'auc': weighted_auc,
    'cv_auc_mean': np.nan, 'cv_auc_std': np.nan,
    'time': weighted_time,
}
best_models['WeightedVoting'] = weighted_voting_clf

# ═══════════════════════════════════════════════════════════════════════════════
# ENSEMBLE COMPARISON
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print("ENSEMBLE METHOD COMPARISON")
print("="*70)

ensemble_names = ['VotingEnsemble_Top7', 'StackingClassifier', 'StackingMLP', 
                  'CalibratedStacking', 'WeightedVoting']
ensemble_results = [(name, results[name]) for name in ensemble_names if name in results]
ensemble_results.sort(key=lambda x: x[1]['auc'], reverse=True)

print(f"{'Method':<25} {'Accuracy':>10} {'AUC':>10} {'F1':>10}")
print("-"*55)
for name, r in ensemble_results:
    print(f"{name:<25} {r['accuracy']:>10.4f} {r['auc']:>10.4f} {r['f1']:>10.4f}")

best_ensemble_name = ensemble_results[0][0]
print(f"\n🏆 Best Ensemble: {best_ensemble_name} (AUC: {ensemble_results[0][1]['auc']:.4f})")

## 8. Results Comparison Table

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ML RESULTS TABLE (WITH GRIDSEARCHCV)
# ═══════════════════════════════════════════════════════════════════════════════

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('auc', ascending=False)

# Format for display
display_cols = ['cv_auc_mean', 'cv_auc_std', 'accuracy', 'auc', 'f1', 'precision', 'recall', 'time']
results_display = results_df[display_cols].copy()

# Round values
for col in results_display.columns:
    if col != 'time':
        results_display[col] = results_display[col].apply(lambda x: f'{x:.4f}' if pd.notna(x) else '-')
    else:
        results_display[col] = results_display[col].apply(lambda x: f'{x:.1f}s')

print("="*90)
print("ML MODEL RESULTS (sorted by Test AUC)")
print("="*90)
print(results_display.to_string())
print("="*90)

# ─── Highlight best model ────────────────────────────────────────────────────
best_model_name = results_df.index[0]
best_model_auc = results_df.loc[best_model_name, 'auc']
print(f"\n★ BEST ML MODEL: {best_model_name}")
print(f"  Test AUC: {best_model_auc:.4f}")

# Save results
results_df.to_csv(os.path.join(OUTPUT_DIR, 'ml_results_detailed.csv'))
joblib.dump(results, os.path.join(OUTPUT_DIR, 'ml_results.joblib'))
print("\nML results saved.")


## 9. Visualization: ROC Curves, Confusion Matrices, Feature Importance

In [ ]:
# ─── ROC Curves for All Models ───────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 8))

for name, model in best_models.items():
    y_prob = model.predict_proba(X_test_selected)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = roc_auc_score(y_test, y_prob)
    lw = 3 if 'Ensemble' in name else 1.5
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})", linewidth=lw)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curves — All Models', fontsize=15)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'roc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Confusion Matrices ──────────────────────────────────────────

n_models = len(best_models)
cols = 4
rows = (n_models + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
axes = axes.flatten()

for i, (name, model) in enumerate(best_models.items()):
    y_pred = model.predict(X_test_selected)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
    axes[i].set_title(f'{name}\nAcc={accuracy_score(y_test, y_pred):.3f}', fontsize=10)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Confusion Matrices — All Models', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Feature Importance (from tree-based models) ─────────────────

# Only use models that exist in best_models
tree_models = [name for name in ['RandomForest', 'XGBoost', 'LightGBM', 'ExtraTrees', 'GradientBoosting'] if name in best_models]

fig, axes = plt.subplots(1, len(tree_models), figsize=(6*len(tree_models), 10))

for i, name in enumerate(tree_models):
    model = best_models[name]
    importance = model.feature_importances_
    sorted_idx = np.argsort(importance)
    
    axes[i].barh(range(len(sorted_idx)), importance[sorted_idx], align='center')
    axes[i].set_yticks(range(len(sorted_idx)))
    # Handle expanded features from PolynomialFeatures
    if max(sorted_idx) < len(FEATURE_NAMES):
        axes[i].set_yticklabels([FEATURE_NAMES[j] for j in sorted_idx], fontsize=7)
    else:
        axes[i].set_yticklabels([f'Feature_{j}' for j in sorted_idx], fontsize=7)
    axes[i].set_title(f'{name}', fontsize=11)
    axes[i].set_xlabel('Importance')

plt.suptitle('Feature Importance — Tree-Based Models', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Precision-Recall Curves ─────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 8))

for name, model in best_models.items():
    y_prob = model.predict_proba(X_test_selected)[:, 1]
    prec_vals, rec_vals, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    lw = 3 if 'Ensemble' in name else 1.5
    ax.plot(rec_vals, prec_vals, label=f"{name} (AP={ap:.4f})", linewidth=lw)

ax.set_xlabel('Recall', fontsize=13)
ax.set_ylabel('Precision', fontsize=13)
ax.set_title('Precision-Recall Curves — All Models', fontsize=15)
ax.legend(loc='lower left', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'precision_recall_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. Best Model Classification Report

In [ ]:
# ─── Detailed report for the best model ──────────────────────────

best_name = results_df.index[0]
best_model = best_models[best_name]

y_pred_best = best_model.predict(X_test_selected)
y_prob_best = best_model.predict_proba(X_test_selected)[:, 1]

print(f"{'='*60}")
print(f"BEST MODEL: {best_name}")
print(f"{'='*60}")
print(f"\nTest AUC: {roc_auc_score(y_test, y_prob_best):.4f}")

# CV AUC — only available for individual models, not Ensemble
cv_mean = results[best_name].get('cv_auc_mean')
cv_std = results[best_name].get('cv_auc_std')
if cv_mean is not None and cv_std is not None:
    print(f"CV AUC: {cv_mean:.4f} +/- {cv_std:.4f}")
else:
    print(f"CV AUC: N/A (ensemble of top-3 models)")

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=['Real', 'Deepfake']))

## 11. Save Everything for Deep Learning Notebook

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SAVE ALL ML ARTIFACTS
# ═══════════════════════════════════════════════════════════════════════════════

# Save training/test data
np.save(os.path.join(OUTPUT_DIR, 'X_train.npy'), X_train_scaled)
np.save(os.path.join(OUTPUT_DIR, 'X_test.npy'), X_test_scaled)
np.save(os.path.join(OUTPUT_DIR, 'y_train.npy'), y_train)
np.save(os.path.join(OUTPUT_DIR, 'y_test.npy'), y_test)

# Save features for DL
np.save(os.path.join(OUTPUT_DIR, 'features.npy'), {'X': X, 'y': y})

# Save results
results_df.to_csv(os.path.join(OUTPUT_DIR, 'ml_results_detailed.csv'))
joblib.dump(results, os.path.join(OUTPUT_DIR, 'ml_results.joblib'))

# Save GridSearchCV results
# NOTE: search_results was for GridSearchCV which is not used in this notebook
# The following block is commented out as this notebook uses pre-tuned models
# for name, search in search_results.items():
#     cv_results = pd.DataFrame(search.cv_results_)
#     cv_results.to_csv(os.path.join(OUTPUT_DIR, f'{name}_gridsearch_results.csv'))

# Save best hyperparameters
best_params_all = {name: r['best_params'] for name, r in results.items() if 'best_params' in r}
joblib.dump(best_params_all, os.path.join(OUTPUT_DIR, 'best_hyperparameters.joblib'))

print("\n" + "="*70)
print("ML ARTIFACTS SAVED:")
print("="*70)
print(f"  - {len(best_models)} trained models (.joblib)")
#     print(f"  - {len(search_results)} GridSearchCV results (.csv)")
print(f"  - Best hyperparameters")
print(f"  - Feature matrix and labels")
print(f"  - Train/test splits")
print(f"  - Scaler")
print("="*70)


In [ ]:
# Required imports for EfficientNet feature extraction
import torch
from torch.cuda.amp import autocast
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ═══════════════════════════════════════════════════════════════════════════════
# ADVANCED: EFFICIENTNET FACE FEATURE EXTRACTION
# ═══════════════════════════════════════════════════════════════════════════════
# Extracts deep visual features from face crops using pretrained EfficientNet
# These features capture texture, compression artifacts, and blending boundaries
# Combines with rPPG features for maximum accuracy
# ═══════════════════════════════════════════════════════════════════════════════

EXTRACT_FACE_FEATURES = True  # Set to False to skip (saves ~30-40 min)

if EXTRACT_FACE_FEATURES:
    import timm
    from torchvision import transforms
    from PIL import Image
    import torch.nn.functional as F
    
    print("="*70)
    print("EFFICIENTNET FACE FEATURE EXTRACTION")
    print("="*70)
    
    # ─── Load pretrained EfficientNet-B0 (lighter for Kaggle) ────────────────
    # B0 is faster than B4 but still very effective
    efficientnet = timm.create_model(
        'efficientnet_b0',  # Use B0 for speed, B4 for accuracy
        pretrained=True,
        num_classes=0,  # Remove classification head, get features
    )
    efficientnet = efficientnet.to(device)
    efficientnet.eval()
    
    print(f"  Model: EfficientNet-B0")
    print(f"  Feature dim: 1280")
    print(f"  Device: {device}")
    
    # ─── Face preprocessing transform ────────────────────────────────────────
    face_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])
    
    # ─── Augmentations for robust feature extraction ─────────────────────────
    face_augment = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
        transforms.RandomAdjustSharpness(sharpness_factor=1.5, p=0.3),
    ])
    
    def extract_face_crop(frame, landmarks, padding=0.2):
        """Extract aligned face crop from frame using landmarks."""
        h, w = frame.shape[:2]
        
        # Get bounding box from landmarks
        xs = [landmarks.landmark[i].x * w for i in range(468)]
        ys = [landmarks.landmark[i].y * h for i in range(468)]
        
        x_min, x_max = int(min(xs)), int(max(xs))
        y_min, y_max = int(min(ys)), int(max(ys))
        
        # Add padding
        pad_w = int((x_max - x_min) * padding)
        pad_h = int((y_max - y_min) * padding)
        
        x_min = max(0, x_min - pad_w)
        x_max = min(w, x_max + pad_w)
        y_min = max(0, y_min - pad_h)
        y_max = min(h, y_max + pad_h)
        
        # Extract and convert to PIL
        face_crop = frame[y_min:y_max, x_min:x_max]
        if face_crop.size == 0:
            return None
        
        face_crop = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
        return Image.fromarray(face_crop)
    
    def extract_efficientnet_features(video_path, n_frames=10, use_augment=False):
        """
        Extract EfficientNet features from face crops across video frames.
        Returns: (1280,) mean feature vector
        """
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return None
        
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps <= 0: fps = 30.0
        
        # Sample frames evenly (temporal diversity)
        frame_indices = np.linspace(0, total_frames - 1, n_frames, dtype=int)
        
        features_list = []
        
        for idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret:
                continue
            
            # Detect face
            h, w = frame.shape[:2]
            res = _FACE_MESH.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            
            if res.multi_face_landmarks:
                face_crop = extract_face_crop(frame, res.multi_face_landmarks[0])
                
                if face_crop is not None:
                    # Apply augmentation if training
                    if use_augment:
                        face_crop = face_augment(face_crop)
                    
                    # Transform and extract features
                    face_tensor = face_transform(face_crop).unsqueeze(0).to(device)
                    
                    with torch.no_grad():
                        with autocast():
                            feat = efficientnet(face_tensor)
                    
                    features_list.append(feat.cpu().numpy().flatten())
        
        cap.release()
        
        if len(features_list) < 3:
            return None
        
        # Return mean features across frames
        return np.mean(features_list, axis=0)
    
    # ─── Extract features for all videos ─────────────────────────────────────
    print(f"\nExtracting EfficientNet features from {len(valid_video_paths)} videos...")
    print("  Sampling 10 frames per video (temporal diversity)")
    
    face_features = []
    face_labels = []
    valid_indices = []
    
    for i, (video_path, label) in enumerate(tqdm(zip(valid_video_paths, y), total=len(valid_video_paths))):
        feat = extract_efficientnet_features(video_path, n_frames=10)
        
        if feat is not None:
            face_features.append(feat)
            face_labels.append(label)
            valid_indices.append(i)
    
    face_features = np.array(face_features)
    face_labels = np.array(face_labels)
    
    print(f"\n  Extracted features: {face_features.shape}")
    print(f"  Valid videos: {len(valid_indices)}/{len(valid_video_paths)}")




    # ─── Save EfficientNet features ──────────────────────────────────────────
    np.save(os.path.join(OUTPUT_DIR, 'efficientnet_features.npy'), face_features)
    np.save(os.path.join(OUTPUT_DIR, 'efficientnet_valid_indices.npy'), valid_indices)
    
    # ─── Combine with rPPG features ──────────────────────────────────────────
    # Align indices
    rppg_features_aligned = X[valid_indices]
    
    # Concatenate: rPPG (48) + EfficientNet (1280) = 1328 features
    combined_features = np.hstack([rppg_features_aligned, face_features])
    
    print(f"\n  Combined features shape: {combined_features.shape}")
    print(f"    rPPG features: {rppg_features_aligned.shape[1]}")
    print(f"    EfficientNet features: {face_features.shape[1]}")
    print(f"    Total: {combined_features.shape[1]}")
    
    # Update X and y for combined training
    X_combined = combined_features
    y_combined = face_labels
    
    # Save combined features
    np.save(os.path.join(OUTPUT_DIR, 'combined_features.npy'), X_combined)
    
    HAS_FACE_FEATURES = True
    print("\n✓ EfficientNet feature extraction complete!")
    
else:
    print("EfficientNet feature extraction skipped (EXTRACT_FACE_FEATURES = False)")
    HAS_FACE_FEATURES = False
    X_combined = None

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ADVANCED: FREQUENCY DOMAIN + LANDMARK GEOMETRY FEATURES
# ═══════════════════════════════════════════════════════════════════════════════
# Frequency features: Deepfakes leave artifacts in FFT/DCT space
# Geometry features: Landmark distances reveal unnatural face proportions
# These are used in top research papers for +3-5% accuracy boost
# ═══════════════════════════════════════════════════════════════════════════════

EXTRACT_ADVANCED_FEATURES = True

if EXTRACT_ADVANCED_FEATURES:
    from scipy.fftpack import dct
    import mediapipe as mp
    
    print("="*70)
    print("EXTRACTING FREQUENCY + GEOMETRY FEATURES")
    print("="*70)
    
    # ─── Frequency Domain Feature Extraction ─────────────────────────────────
    
    def extract_fft_features(face_crop, n_features=32):
        """
        Extract FFT magnitude features from face image.
        Deepfakes often have artifacts in high-frequency components.
        """
        if face_crop is None:
            return np.zeros(n_features)
        
        # Convert to grayscale
        if len(face_crop.shape) == 3:
            gray = cv2.cvtColor(face_crop, cv2.COLOR_BGR2GRAY)
        else:
            gray = face_crop
        
        # Resize to fixed size
        gray = cv2.resize(gray, (128, 128))
        
        # 2D FFT
        fft = np.fft.fft2(gray)
        fft_shift = np.fft.fftshift(fft)
        magnitude = np.log1p(np.abs(fft_shift))
        
        # Extract radial profile (frequency vs magnitude)
        center = magnitude.shape[0] // 2
        y, x = np.ogrid[:magnitude.shape[0], :magnitude.shape[1]]
        r = np.sqrt((x - center)**2 + (y - center)**2).astype(int)
        
        # Compute radial mean
        max_r = min(center, n_features)
        radial_profile = np.zeros(n_features)
        for i in range(min(max_r, n_features)):
            mask = r == i
            if np.any(mask):
                radial_profile[i] = magnitude[mask].mean()
        
        return radial_profile
    
    def extract_dct_features(face_crop, n_features=32):
        """
        Extract DCT (Discrete Cosine Transform) features.
        Used in JPEG compression - deepfakes often show DCT artifacts.
        """
        if face_crop is None:
            return np.zeros(n_features)
        
        # Convert to grayscale
        if len(face_crop.shape) == 3:
            gray = cv2.cvtColor(face_crop, cv2.COLOR_BGR2GRAY)
        else:
            gray = face_crop
        
        # Resize
        gray = cv2.resize(gray, (64, 64)).astype(np.float32)
        
        # 2D DCT
        dct_coeffs = dct(dct(gray.T, norm='ortho').T, norm='ortho')
        
        # Flatten and take top coefficients (zigzag order approximation)
        flat = np.abs(dct_coeffs).flatten()
        
        # Sort by magnitude and take statistics
        sorted_coeffs = np.sort(flat)[::-1]
        
        features = []
        # Top coefficients
        features.extend(sorted_coeffs[:n_features//2])
        # Statistical features
        features.append(np.mean(flat))
        features.append(np.std(flat))
        features.append(np.median(flat))
        features.append(skew(flat) if len(flat) > 3 else 0)
        features.append(kurtosis(flat) if len(flat) > 3 else 0)
        # Energy in different frequency bands
        low_freq = flat[:len(flat)//4].sum()
        mid_freq = flat[len(flat)//4:len(flat)//2].sum()
        high_freq = flat[len(flat)//2:].sum()
        features.append(low_freq / (high_freq + 1e-10))  # LF/HF ratio
        features.append(mid_freq / (low_freq + 1e-10))
        
        # Pad or truncate to n_features
        features = np.array(features[:n_features])
        if len(features) < n_features:
            features = np.pad(features, (0, n_features - len(features)))
        
        return features
    
    # ─── Landmark Geometry Feature Extraction ────────────────────────────────
    
    # Key landmark indices for geometry
    LANDMARK_PAIRS = {
        # Eye landmarks
        'left_eye_inner': 133,
        'left_eye_outer': 33,
        'right_eye_inner': 362,
        'right_eye_outer': 263,
        'left_eye_top': 159,
        'left_eye_bottom': 145,
        'right_eye_top': 386,
        'right_eye_bottom': 374,
        # Nose landmarks
        'nose_tip': 1,
        'nose_left': 279,
        'nose_right': 49,
        # Mouth landmarks
        'mouth_left': 61,
        'mouth_right': 291,
        'mouth_top': 0,
        'mouth_bottom': 17,
        'upper_lip': 13,
        'lower_lip': 14,
        # Face contour
        'chin': 152,
        'forehead': 10,
        'left_cheek': 234,
        'right_cheek': 454,
        'left_jaw': 172,
        'right_jaw': 397,
    }
    
    def compute_landmark_distance(landmarks, idx1, idx2, w, h):
        """Compute normalized distance between two landmarks."""
        lm1 = landmarks.landmark[idx1]
        lm2 = landmarks.landmark[idx2]
        
        dx = (lm1.x - lm2.x) * w
        dy = (lm1.y - lm2.y) * h
        
        return np.sqrt(dx**2 + dy**2)
    
    def compute_angle(landmarks, idx1, idx2, idx3, w, h):
        """Compute angle at idx2 formed by idx1-idx2-idx3."""
        lm1 = landmarks.landmark[idx1]
        lm2 = landmarks.landmark[idx2]
        lm3 = landmarks.landmark[idx3]
        
        v1 = np.array([(lm1.x - lm2.x) * w, (lm1.y - lm2.y) * h])
        v2 = np.array([(lm3.x - lm2.x) * w, (lm3.y - lm2.y) * h])
        
        cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-10)
        return np.arccos(np.clip(cos_angle, -1, 1))
    
    def extract_geometry_features(landmarks, w, h):
        """
        Extract 30 geometry features from face landmarks.
        These capture face proportions and symmetry.
        """
        if landmarks is None:
            return np.zeros(30)
        
        features = []
        
        LM = LANDMARK_PAIRS
        
        # 1. Inter-eye distance (normalized by face width)
        eye_dist = compute_landmark_distance(landmarks, LM['left_eye_inner'], LM['right_eye_inner'], w, h)
        face_width = compute_landmark_distance(landmarks, LM['left_cheek'], LM['right_cheek'], w, h)
        features.append(eye_dist / (face_width + 1e-10))
        
        # 2. Eye width ratio (left vs right)
        left_eye_w = compute_landmark_distance(landmarks, LM['left_eye_inner'], LM['left_eye_outer'], w, h)
        right_eye_w = compute_landmark_distance(landmarks, LM['right_eye_inner'], LM['right_eye_outer'], w, h)
        features.append(left_eye_w / (right_eye_w + 1e-10))
        
        # 3. Eye height ratio
        left_eye_h = compute_landmark_distance(landmarks, LM['left_eye_top'], LM['left_eye_bottom'], w, h)
        right_eye_h = compute_landmark_distance(landmarks, LM['right_eye_top'], LM['right_eye_bottom'], w, h)
        features.append(left_eye_h / (right_eye_h + 1e-10))
        
        # 4. Eye aspect ratios
        features.append(left_eye_w / (left_eye_h + 1e-10))
        features.append(right_eye_w / (right_eye_h + 1e-10))
        
        # 5. Nose width / face width
        nose_width = compute_landmark_distance(landmarks, LM['nose_left'], LM['nose_right'], w, h)
        features.append(nose_width / (face_width + 1e-10))
        
        # 6. Mouth width / face width
        mouth_width = compute_landmark_distance(landmarks, LM['mouth_left'], LM['mouth_right'], w, h)
        features.append(mouth_width / (face_width + 1e-10))
        
        # 7. Mouth aspect ratio
        mouth_height = compute_landmark_distance(landmarks, LM['mouth_top'], LM['mouth_bottom'], w, h)
        features.append(mouth_width / (mouth_height + 1e-10))
        
        # 8. Lip thickness ratio
        upper_lip_h = compute_landmark_distance(landmarks, LM['mouth_top'], LM['upper_lip'], w, h)
        lower_lip_h = compute_landmark_distance(landmarks, LM['lower_lip'], LM['mouth_bottom'], w, h)
        features.append(upper_lip_h / (lower_lip_h + 1e-10))
        
        # 9. Face height / width ratio
        face_height = compute_landmark_distance(landmarks, LM['forehead'], LM['chin'], w, h)
        features.append(face_height / (face_width + 1e-10))
        
        # 10. Jaw symmetry
        left_jaw_dist = compute_landmark_distance(landmarks, LM['chin'], LM['left_jaw'], w, h)
        right_jaw_dist = compute_landmark_distance(landmarks, LM['chin'], LM['right_jaw'], w, h)
        features.append(left_jaw_dist / (right_jaw_dist + 1e-10))
        
        # 11-13. Nose position ratios
        nose_to_eyes = compute_landmark_distance(landmarks, LM['nose_tip'], LM['left_eye_inner'], w, h)
        nose_to_mouth = compute_landmark_distance(landmarks, LM['nose_tip'], LM['mouth_top'], w, h)
        features.append(nose_to_eyes / (face_height + 1e-10))
        features.append(nose_to_mouth / (face_height + 1e-10))
        features.append(nose_to_eyes / (nose_to_mouth + 1e-10))
        
        # 14-16. Key angles
        # Jaw angle
        jaw_angle = compute_angle(landmarks, LM['left_jaw'], LM['chin'], LM['right_jaw'], w, h)
        features.append(jaw_angle)
        
        # Eye-nose angle
        eye_nose_angle = compute_angle(landmarks, LM['left_eye_inner'], LM['nose_tip'], LM['right_eye_inner'], w, h)
        features.append(eye_nose_angle)
        
        # Mouth angle (smile detection)
        mouth_angle = compute_angle(landmarks, LM['mouth_left'], LM['mouth_bottom'], LM['mouth_right'], w, h)
        features.append(mouth_angle)
        
        # 17-20. Cheek symmetry
        left_cheek_to_nose = compute_landmark_distance(landmarks, LM['left_cheek'], LM['nose_tip'], w, h)
        right_cheek_to_nose = compute_landmark_distance(landmarks, LM['right_cheek'], LM['nose_tip'], w, h)
        features.append(left_cheek_to_nose / (right_cheek_to_nose + 1e-10))
        
        left_cheek_to_mouth = compute_landmark_distance(landmarks, LM['left_cheek'], LM['mouth_left'], w, h)
        right_cheek_to_mouth = compute_landmark_distance(landmarks, LM['right_cheek'], LM['mouth_right'], w, h)
        features.append(left_cheek_to_mouth / (right_cheek_to_mouth + 1e-10))
        
        # 21-25. Forehead proportions
        forehead_to_eyes = compute_landmark_distance(landmarks, LM['forehead'], LM['left_eye_top'], w, h)
        features.append(forehead_to_eyes / (face_height + 1e-10))
        
        forehead_width_left = compute_landmark_distance(landmarks, LM['forehead'], LM['left_eye_outer'], w, h)
        forehead_width_right = compute_landmark_distance(landmarks, LM['forehead'], LM['right_eye_outer'], w, h)
        features.append(forehead_width_left / (forehead_width_right + 1e-10))
        
        # 26-28. Vertical thirds of face
        upper_third = compute_landmark_distance(landmarks, LM['forehead'], LM['left_eye_bottom'], w, h)
        middle_third = compute_landmark_distance(landmarks, LM['left_eye_bottom'], LM['nose_tip'], w, h)
        lower_third = compute_landmark_distance(landmarks, LM['nose_tip'], LM['chin'], w, h)
        features.append(upper_third / (face_height + 1e-10))
        features.append(middle_third / (face_height + 1e-10))
        features.append(lower_third / (face_height + 1e-10))
        
        # 29-30. Overall symmetry scores
        # Left-right symmetry
        left_landmarks = [LM['left_eye_inner'], LM['left_eye_outer'], LM['left_cheek'], LM['left_jaw'], LM['mouth_left']]
        right_landmarks = [LM['right_eye_inner'], LM['right_eye_outer'], LM['right_cheek'], LM['right_jaw'], LM['mouth_right']]
        
        symmetry_diffs = []
        chin_lm = landmarks.landmark[LM['chin']]
        chin_x = chin_lm.x * w
        
        for l_idx, r_idx in zip(left_landmarks, right_landmarks):
            l_lm = landmarks.landmark[l_idx]
            r_lm = landmarks.landmark[r_idx]
            l_dist = abs(l_lm.x * w - chin_x)
            r_dist = abs(r_lm.x * w - chin_x)
            symmetry_diffs.append(abs(l_dist - r_dist) / (face_width + 1e-10))
        
        features.append(np.mean(symmetry_diffs))
        features.append(np.std(symmetry_diffs))
        
        return np.array(features[:30])
    
    # ─── Extract features for all videos ─────────────────────────────────────
    
    def extract_freq_geo_features(video_path, n_frames=5):
        """Extract frequency and geometry features from video."""
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return None, None
        
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_indices = np.linspace(0, total_frames - 1, n_frames, dtype=int)
        
        fft_features_list = []
        dct_features_list = []
        geo_features_list = []
        
        for idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret:
                continue
            
            h, w = frame.shape[:2]
            res = _FACE_MESH.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            
            if res.multi_face_landmarks:
                landmarks = res.multi_face_landmarks[0]
                
                # Extract face crop for frequency features
                xs = [landmarks.landmark[i].x * w for i in range(468)]
                ys = [landmarks.landmark[i].y * h for i in range(468)]
                x_min, x_max = int(min(xs)), int(max(xs))
                y_min, y_max = int(min(ys)), int(max(ys))
                
                # Add padding
                pad = int((x_max - x_min) * 0.1)
                x_min = max(0, x_min - pad)
                x_max = min(w, x_max + pad)
                y_min = max(0, y_min - pad)
                y_max = min(h, y_max + pad)
                
                face_crop = frame[y_min:y_max, x_min:x_max]
                
                if face_crop.size > 0:
                    fft_features_list.append(extract_fft_features(face_crop))
                    dct_features_list.append(extract_dct_features(face_crop))
                    geo_features_list.append(extract_geometry_features(landmarks, w, h))
        
        cap.release()
        
        if len(fft_features_list) < 2:
            return None, None
        
        # Aggregate features (mean + std)
        fft_mean = np.mean(fft_features_list, axis=0)
        fft_std = np.std(fft_features_list, axis=0)
        dct_mean = np.mean(dct_features_list, axis=0)
        dct_std = np.std(dct_features_list, axis=0)
        geo_mean = np.mean(geo_features_list, axis=0)
        geo_std = np.std(geo_features_list, axis=0)
        
        freq_features = np.concatenate([fft_mean, fft_std[:16], dct_mean, dct_std[:16]])  # 32+16+32+16 = 96
        geo_features = np.concatenate([geo_mean, geo_std[:15]])  # 30+15 = 45
        
        return freq_features, geo_features
    
    print(f"\nExtracting frequency + geometry features from {len(valid_video_paths)} videos...")
    
    freq_features_all = []
    geo_features_all = []
    valid_freq_indices = []
    
    for i, video_path in enumerate(tqdm(valid_video_paths)):
        freq_feat, geo_feat = extract_freq_geo_features(video_path, n_frames=5)
        
        if freq_feat is not None and geo_feat is not None:
            freq_features_all.append(freq_feat)
            geo_features_all.append(geo_feat)
            valid_freq_indices.append(i)
    
    freq_features_all = np.array(freq_features_all)
    geo_features_all = np.array(geo_features_all)
    
    print(f"\n  Frequency features: {freq_features_all.shape}")
    print(f"  Geometry features: {geo_features_all.shape}")
    print(f"  Valid videos: {len(valid_freq_indices)}/{len(valid_video_paths)}")
    
    # ─── Save features ───────────────────────────────────────────────────────
    np.save(os.path.join(OUTPUT_DIR, 'frequency_features.npy'), freq_features_all)
    np.save(os.path.join(OUTPUT_DIR, 'geometry_features.npy'), geo_features_all)
    np.save(os.path.join(OUTPUT_DIR, 'freq_geo_valid_indices.npy'), valid_freq_indices)
    
    # ─── Update combined features if we have all modalities ──────────────────
    print("\nCombining all feature modalities...")
    
    # Find common valid indices across all feature types
    common_indices = set(range(len(valid_video_paths)))
    common_indices &= set(valid_freq_indices)
    
    if HAS_FACE_FEATURES:
        common_indices &= set(valid_indices)  # EfficientNet indices
    
    common_indices = sorted(list(common_indices))
    print(f"  Common valid videos: {len(common_indices)}")
    
    # Align all features to common indices
    rppg_aligned = X[common_indices]  # 48 features
    freq_aligned = freq_features_all[[valid_freq_indices.index(i) for i in common_indices if i in valid_freq_indices]]
    geo_aligned = geo_features_all[[valid_freq_indices.index(i) for i in common_indices if i in valid_freq_indices]]
    
    if HAS_FACE_FEATURES:
        eff_indices_map = {v: k for k, v in enumerate(valid_indices)}
        eff_aligned = face_features[[eff_indices_map[i] for i in common_indices if i in eff_indices_map]]
    else:
        eff_aligned = None
    
    # Combine all features
    if eff_aligned is not None:
        all_features = np.hstack([rppg_aligned, eff_aligned, freq_aligned, geo_aligned])
        print(f"  Combined: rPPG({rppg_aligned.shape[1]}) + EfficientNet({eff_aligned.shape[1]}) + Freq({freq_aligned.shape[1]}) + Geo({geo_aligned.shape[1]})")
    else:
        all_features = np.hstack([rppg_aligned, freq_aligned, geo_aligned])
        print(f"  Combined: rPPG({rppg_aligned.shape[1]}) + Freq({freq_aligned.shape[1]}) + Geo({geo_aligned.shape[1]})")
    
    all_labels = y[common_indices]
    
    print(f"  Total features: {all_features.shape[1]}")
    print(f"  Total samples: {all_features.shape[0]}")
    
    # Save final combined features
    np.save(os.path.join(OUTPUT_DIR, 'all_combined_features.npy'), all_features)
    np.save(os.path.join(OUTPUT_DIR, 'all_combined_labels.npy'), all_labels)
    
    HAS_FREQ_GEO = True
    print("\n✓ Frequency + Geometry feature extraction complete!")
    
else:
    print("Frequency + Geometry extraction skipped")
    HAS_FREQ_GEO = False

In [ ]:
import shutil

shutil.make_archive("model_outputs", 'zip', "/kaggle/working")

# Neuro-Pulse: rPPG-Based Deepfake Detection — Deep Learning Training
## Kaggle P100 Optimized (Won't Crash)

**5 DL architectures** + **ensemble** on 35-dim rPPG features.
- **60 epochs max** with early stopping (patience=10)
- **GPU memory cleared** between models
- **Smaller hidden dims** to fit P100 16GB comfortably
- Total runtime: ~15-30 minutes on Kaggle P100

**Models:** 1D-CNN, BiLSTM+Attention, CNN-BiLSTM, Transformer, PhysNet-MLP

**References:**
- DeepFakesON-Phys (Hernandez-Ortega et al., 2020)
- FakeCatcher (Ciftci et al., TPAMI 2020)
- pyVHR (Boccignone et al., 2022)
- rPPG-Toolbox (Liu et al., NeurIPS 2023)

## 1. Setup & Imports

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
)
import joblib

# ─── PyTorch Imports ─────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from torch.optim.lr_scheduler import (
    CosineAnnealingWarmRestarts, OneCycleLR, ReduceLROnPlateau,
    CosineAnnealingLR, StepLR
)
from torch.cuda.amp import GradScaler, autocast  # Mixed precision

# ─── Advanced Training Utilities ────────────────────────────────────────────
try:
    import optuna
    HAS_OPTUNA = True
except ImportError:
    HAS_OPTUNA = False
    print("Optuna not available - hyperparameter tuning will use manual search.")

warnings.filterwarnings('ignore')

# ─── Device Configuration (Kaggle P100) ─────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"  CUDA Version: {torch.version.cuda}")
    # Enable TF32 for faster training on Ampere GPUs (A100, etc.)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

# ─── Set seeds for reproducibility ─────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*60)
print("Deep Learning Setup Complete!")
print(f"PyTorch: {torch.__version__}")
print(f"Optuna available: {HAS_OPTUNA}")
print("="*60)


## 2. Load Data

In [ ]:
# Define feature availability flags (set by earlier cells)
if 'HAS_FREQ_GEO' not in globals():
    HAS_FREQ_GEO = False
if 'HAS_FACE_FEATURES' not in globals():
    HAS_FACE_FEATURES = False

# ═══════════════════════════════════════════════════════════════════════════════
# LOAD & PREPARE DATA FOR DEEP LEARNING
# ═══════════════════════════════════════════════════════════════════════════════

# ─── Load features ───────────────────────────────────────────────────────────
# Works in 3 scenarios:
# 1. Features saved from ML section (same notebook)
# 2. Features from previous run (OUTPUT_DIR)
# 3. Features from Kaggle input

feature_paths = [
    os.path.join(OUTPUT_DIR, 'features.npy'),
    '/kaggle/input/features.npy',
]

for path in feature_paths:
    if os.path.exists(path):
        features = np.load(path, allow_pickle=True).item()
        print(f"Loaded features from: {path}")
        break
else:
    # Use features from ML section if available
    if 'X' in globals() and 'y' in globals():
        features = {'X': X, 'y': y}
        print("Using features from current session.")
    else:
        raise FileNotFoundError("No feature file found!")


if 'all_features' in globals() and HAS_FREQ_GEO:
    X = all_features
    y = all_labels
    print("Using all combined features (rPPG + EfficientNet + Freq + Geo)")
elif 'X_combined' in globals() and HAS_FACE_FEATURES:
    X = X_combined
    y = y_combined
    print("Using combined features (rPPG + EfficientNet)")
else:
    X = features['X']
    y = features['y']
    print("Using base rPPG features")

N_FEATURES = X.shape[1]

print(f"Feature matrix: {X.shape}")
print(f"Labels: {y.shape}")
print(f"Class distribution: Real={np.sum(y==0)}, Fake={np.sum(y==1)}")

# ─── Train/Val/Test Split ────────────────────────────────────────────────────

# Test Set as ML so they can be ensembled later
X_train_temp, X_test_dl, y_train_temp, y_test_dl = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
X_train_dl, X_val_dl, y_train_dl, y_val_dl = train_test_split(
    X_train_temp, y_train_temp, test_size=0.2, random_state=SEED, stratify=y_train_temp 
)

# Scale features safely
scaler_dl = RobustScaler()
X_tr = scaler_dl.fit_transform(X_train_dl)
X_val = scaler_dl.transform(X_val_dl)
X_test_dl_scaled = scaler_dl.transform(X_test_dl)
y_tr = y_train_dl
y_val = y_val_dl
y_test_dl = y_test_dl

print(f"\nData splits:")
print(f"  Train: {X_tr.shape} | Real={np.sum(y_tr==0)}, Fake={np.sum(y_tr==1)}")
print(f"  Val:   {X_val.shape} | Real={np.sum(y_val==0)}, Fake={np.sum(y_val==1)}")
print(f"  Test:  {X_test_dl_scaled.shape} | Real={np.sum(y_test_dl==0)}, Fake={np.sum(y_test_dl==1)}")

# Save scaler
joblib.dump(scaler_dl, os.path.join(OUTPUT_DIR, 'dl_scaler.joblib'))


In [ ]:
# ─── Create PyTorch DataLoaders ──────────────────────────────────

BATCH_SIZE = 32

# Weighted sampler for class imbalance
class_counts = np.bincount(y_tr)
class_weights = 1.0 / class_counts
sample_weights = class_weights[y_tr]
sampler = WeightedRandomSampler(
    weights=sample_weights, num_samples=len(sample_weights), replacement=True
)

# Tensors
train_dataset = TensorDataset(
    torch.FloatTensor(X_tr), torch.LongTensor(y_tr)
)
val_dataset = TensorDataset(
    torch.FloatTensor(X_val), torch.LongTensor(y_val)
)
test_dataset = TensorDataset(
    torch.FloatTensor(X_test_dl_scaled), torch.LongTensor(y_test_dl)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          sampler=sampler, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 3. Model Architectures

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MODEL 1: Enhanced 1D-CNN with Squeeze-Excitation & Residual Connections
# ═══════════════════════════════════════════════════════════════════════════════

class SqueezeExcitation1D(nn.Module):
    """Squeeze-and-Excitation block for 1D convolutions."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)
    
    def forward(self, x):
        # x: (batch, channels, length)
        b, c, _ = x.size()
        squeeze = x.mean(dim=2)  # Global average pooling
        excite = torch.relu(self.fc1(squeeze))
        excite = torch.sigmoid(self.fc2(excite)).unsqueeze(2)
        return x * excite


class ResConvBlock1D(nn.Module):
    """Residual convolutional block with SE attention."""
    def __init__(self, in_ch, out_ch, dropout=0.2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_ch),
        )
        self.se = SqueezeExcitation1D(out_ch)
        self.skip = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.act = nn.GELU()
    
    def forward(self, x):
        residual = self.skip(x)
        out = self.conv(x)
        out = self.se(out)
        return self.act(out + residual)


class DeepfakeCNN1D(nn.Module):
    """
    Enhanced 1D-CNN with:
    - Squeeze-Excitation attention
    - Residual connections
    - Multi-scale feature extraction
    - Dropout regularization
    """
    def __init__(self, n_features=50, dropout=0.3):
        super().__init__()
        
        # Multi-scale convolutional branches
        self.branch1 = nn.Sequential(
            ResConvBlock1D(1, 64, dropout),
            ResConvBlock1D(64, 128, dropout),
            ResConvBlock1D(128, 256, dropout),
        )
        
        self.branch2 = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.GELU(),
            ResConvBlock1D(64, 128, dropout),
            ResConvBlock1D(128, 256, dropout),
        )
        
        self.pool = nn.AdaptiveAvgPool1d(1)
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 2),
        )
    
    def forward(self, x):
        x = x.unsqueeze(1)  # (batch, 1, 35)
        
        # Multi-scale features
        out1 = self.pool(self.branch1(x))
        out2 = self.pool(self.branch2(x))
        
        # Concatenate
        out = torch.cat([out1, out2], dim=1)
        return self.classifier(out)


print("Model 1: DeepfakeCNN1D (Enhanced with SE & Residual)")
m = DeepfakeCNN1D(N_FEATURES).to(device)
print(f"  Parameters: {sum(p.numel() for p in m.parameters()):,}")
del m


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MODEL 2: Enhanced BiLSTM with Multi-Head Attention & Layer Normalization
# ═══════════════════════════════════════════════════════════════════════════════

class MultiHeadAttention(nn.Module):
    """Multi-head self-attention for sequence modeling."""
    def __init__(self, hidden_dim, n_heads=4, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = hidden_dim // n_heads
        assert hidden_dim % n_heads == 0
        
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = self.head_dim ** -0.5
    
    def forward(self, x):
        B, T, C = x.size()
        
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = torch.softmax(attn, dim=-1)
        attn = self.dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


class DeepfakeBiLSTM(nn.Module):
    """
    Enhanced Bidirectional LSTM with:
    - Multi-head self-attention
    - Layer normalization
    - Residual connections
    - Deeper architecture
    """
    def __init__(self, n_features=50, hidden_dim=128, n_layers=3, dropout=0.3):
        super().__init__()
        # Dynamic sequence length based on input features
        self.feat_per_step = min(5, n_features)
        self.seq_len = max(1, n_features // self.feat_per_step)
        
        self.input_proj = nn.Sequential(
            nn.Linear(self.feat_per_step, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
        )
        
        self.lstm = nn.LSTM(
            input_size=hidden_dim, hidden_size=hidden_dim,
            num_layers=n_layers, batch_first=True,
            bidirectional=True, dropout=dropout if n_layers > 1 else 0,
        )
        
        self.ln_lstm = nn.LayerNorm(hidden_dim * 2)
        self.attention = MultiHeadAttention(hidden_dim * 2, n_heads=4, dropout=dropout)
        self.ln_attn = nn.LayerNorm(hidden_dim * 2)
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 2),
        )
    
    def forward(self, x):
        B = x.size(0)
        x = x.view(B, self.seq_len, self.feat_per_step)
        x = self.input_proj(x)
        
        lstm_out, _ = self.lstm(x)
        lstm_out = self.ln_lstm(lstm_out)
        
        # Multi-head attention with residual
        attn_out = self.attention(lstm_out)
        attn_out = self.ln_attn(attn_out + lstm_out)
        
        # Global average pooling
        pooled = attn_out.mean(dim=1)
        return self.classifier(pooled)


print("Model 2: DeepfakeBiLSTM (Enhanced with Multi-Head Attention)")
m = DeepfakeBiLSTM(N_FEATURES).to(device)
print(f"  Parameters: {sum(p.numel() for p in m.parameters()):,}")
del m


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MODEL 3: Enhanced CNN-BiLSTM Hybrid with Attention Fusion
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalConvBlock(nn.Module):
    """Temporal convolution block with dilation."""
    def __init__(self, in_ch, out_ch, kernel_size=3, dilation=1, dropout=0.2):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.skip = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    
    def forward(self, x):
        return self.conv(x) + self.skip(x)


class DeepfakeCNNBiLSTM(nn.Module):
    """
    Enhanced CNN-BiLSTM Hybrid with:
    - Multi-scale temporal convolutions (different dilations)
    - Deep BiLSTM with attention
    - Feature fusion with attention gating
    """
    def __init__(self, n_features=50, cnn_channels=64, lstm_hidden=128,
                 lstm_layers=2, dropout=0.3):
        super().__init__()
        
        # Multi-scale temporal CNN
        self.cnn_branch1 = nn.Sequential(
            TemporalConvBlock(1, cnn_channels, kernel_size=3, dilation=1, dropout=dropout),
            TemporalConvBlock(cnn_channels, cnn_channels * 2, kernel_size=3, dilation=2, dropout=dropout),
        )
        
        self.cnn_branch2 = nn.Sequential(
            TemporalConvBlock(1, cnn_channels, kernel_size=5, dilation=1, dropout=dropout),
            TemporalConvBlock(cnn_channels, cnn_channels * 2, kernel_size=5, dilation=1, dropout=dropout),
        )
        
        # Fusion layer
        self.fusion = nn.Sequential(
            nn.Conv1d(cnn_channels * 4, cnn_channels * 2, 1),
            nn.BatchNorm1d(cnn_channels * 2),
            nn.GELU(),
        )
        
        # BiLSTM
        self.lstm = nn.LSTM(
            input_size=cnn_channels * 2, hidden_size=lstm_hidden,
            num_layers=lstm_layers, batch_first=True,
            bidirectional=True, dropout=dropout if lstm_layers > 1 else 0,
        )
        
        self.attention = MultiHeadAttention(lstm_hidden * 2, n_heads=4, dropout=dropout)
        self.ln = nn.LayerNorm(lstm_hidden * 2)
        
        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 2),
        )
    
    def forward(self, x):
        x = x.unsqueeze(1)
        
        # Multi-scale CNN features
        cnn1 = self.cnn_branch1(x)
        cnn2 = self.cnn_branch2(x)
        cnn_out = torch.cat([cnn1, cnn2], dim=1)
        cnn_out = self.fusion(cnn_out)
        
        # Reshape for LSTM: (batch, length, channels)
        cnn_out = cnn_out.permute(0, 2, 1)
        
        lstm_out, _ = self.lstm(cnn_out)
        
        # Attention with residual
        attn_out = self.attention(lstm_out)
        attn_out = self.ln(attn_out + lstm_out)
        
        # Global pooling
        pooled = attn_out.mean(dim=1)
        return self.classifier(pooled)


print("Model 3: DeepfakeCNNBiLSTM (Enhanced with Multi-Scale Conv)")
m = DeepfakeCNNBiLSTM(N_FEATURES).to(device)
print(f"  Parameters: {sum(p.numel() for p in m.parameters()):,}")
del m


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MODEL 4: Enhanced Transformer with Pre-LayerNorm & Deeper Architecture
# ═══════════════════════════════════════════════════════════════════════════════

class PositionalEncoding(nn.Module):
    """Learnable positional encoding."""
    def __init__(self, d_model, max_len=100):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
    
    def forward(self, x):
        return x + self.pos_embed[:, :x.size(1), :]


class PreLNTransformerBlock(nn.Module):
    """Pre-LayerNorm Transformer block (more stable training)."""
    def __init__(self, d_model, nhead, dim_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_ff, d_model),
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        # Pre-LN attention
        x_norm = self.ln1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + attn_out
        
        # Pre-LN feedforward
        x_norm = self.ln2(x)
        x = x + self.ff(x_norm)
        return x


class DeepfakeTransformer(nn.Module):
    """
    Enhanced Transformer Encoder with:
    - Pre-LayerNorm for stable training
    - Learnable positional encoding
    - Deeper architecture (4 layers)
    - CLS token for classification
    """
    def __init__(self, n_features=50, d_model=64, nhead=4,
                 num_layers=4, dim_ff=256, dropout=0.3):
        super().__init__()
        # Dynamic sequence length based on input features
        self.feat_per_step = min(5, n_features)
        self.seq_len = n_features // self.feat_per_step
        # feat_per_step already set above
        self.d_model = d_model
        
        # CLS token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        
        self.input_proj = nn.Sequential(
            nn.Linear(self.feat_per_step, d_model),
            nn.LayerNorm(d_model),
        )
        self.pos_enc = PositionalEncoding(d_model, max_len=self.seq_len + 1)
        
        self.layers = nn.ModuleList([
            PreLNTransformerBlock(d_model, nhead, dim_ff, dropout)
            for _ in range(num_layers)
        ])
        
        self.ln_final = nn.LayerNorm(d_model)
        
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 2),
        )
    
    def forward(self, x):
        B = x.size(0)
        x = x.view(B, self.seq_len, self.feat_per_step)
        x = self.input_proj(x)
        
        # Prepend CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        x = self.pos_enc(x)
        
        for layer in self.layers:
            x = layer(x)
        
        x = self.ln_final(x)
        
        # Use CLS token for classification
        cls_out = x[:, 0]
        return self.classifier(cls_out)


print("Model 4: DeepfakeTransformer (Enhanced with Pre-LN & CLS token)")
m = DeepfakeTransformer(N_FEATURES).to(device)
print(f"  Parameters: {sum(p.numel() for p in m.parameters()):,}")
del m


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MODEL 5: Enhanced PhysNet-Inspired Deep Residual MLP
# ═══════════════════════════════════════════════════════════════════════════════

class DenseResidualBlock(nn.Module):
    """Dense residual block with skip connections."""
    def __init__(self, dim, growth_rate=64, dropout=0.2):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(dim, growth_rate),
            nn.BatchNorm1d(growth_rate),
            nn.GELU(),
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        out = self.layer(x)
        return torch.cat([x, out], dim=1)


class BottleneckBlock(nn.Module):
    """Bottleneck residual block for efficiency."""
    def __init__(self, dim, bottleneck_factor=4, dropout=0.2):
        super().__init__()
        bottleneck = dim // bottleneck_factor
        self.block = nn.Sequential(
            nn.Linear(dim, bottleneck),
            nn.BatchNorm1d(bottleneck),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(bottleneck, dim),
            nn.BatchNorm1d(dim),
        )
        self.act = nn.GELU()
    
    def forward(self, x):
        return self.act(x + self.block(x))


class PhysNetMLP(nn.Module):
    """
    Enhanced Deep Residual MLP with:
    - DenseNet-style connections
    - Bottleneck blocks for efficiency
    - Squeeze-and-Excitation attention
    - Feature integration from multiple depths
    """
    def __init__(self, n_features=50, hidden_dim=256, n_blocks=5, dropout=0.3):
        super().__init__()
        
        # Input projection with feature expansion
        self.input_proj = nn.Sequential(
            nn.Linear(n_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
        )
        
        # Deep residual blocks
        self.res_blocks = nn.Sequential(
            *[BottleneckBlock(hidden_dim, dropout=dropout * 0.7) for _ in range(n_blocks)]
        )
        
        # Squeeze-Excitation
        self.se = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.GELU(),
            nn.Linear(hidden_dim // 4, hidden_dim),
            nn.Sigmoid(),
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 2),
        )
    
    def forward(self, x):
        x = self.input_proj(x)
        x = self.res_blocks(x)
        
        # SE attention
        se_weights = self.se(x)
        x = x * se_weights
        
        return self.classifier(x)


print("Model 5: PhysNetMLP (Enhanced with SE & Bottleneck)")
m = PhysNetMLP(N_FEATURES).to(device)
print(f"  Parameters: {sum(p.numel() for p in m.parameters()):,}")
del m


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ADVANCED DL ARCHITECTURES: ATTENTION + MULTI-SCALE TEMPORAL
# ═══════════════════════════════════════════════════════════════════════════════
# These architectures are designed for maximum deepfake detection accuracy
# ═══════════════════════════════════════════════════════════════════════════════

print("="*70)
print("DEFINING ADVANCED DL ARCHITECTURES")
print("="*70)

# ─── Squeeze-and-Excitation Block ────────────────────────────────────────────

class SEBlock(nn.Module):
    """
    Squeeze-and-Excitation block for channel attention.
    Learns to emphasize important features and suppress less useful ones.
    """
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool1d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        # x: (batch, channels, length) or (batch, channels)
        if x.dim() == 2:
            x = x.unsqueeze(-1)
        
        b, c, _ = x.size()
        y = self.squeeze(x).view(b, c)
        y = self.excitation(y).view(b, c, 1)
        return (x * y).squeeze(-1) if x.size(-1) == 1 else x * y


class CBAMBlock(nn.Module):
    """
    Convolutional Block Attention Module (CBAM).
    Combines channel attention and spatial attention.
    """
    def __init__(self, channels, reduction=16):
        super().__init__()
        
        # Channel attention
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.max_pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False)
        )
        
        # Spatial attention
        self.conv_spatial = nn.Conv1d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # x: (batch, channels, length)
        if x.dim() == 2:
            x = x.unsqueeze(-1)
        
        b, c, l = x.size()
        
        # Channel attention
        avg_out = self.fc(self.avg_pool(x).view(b, c))
        max_out = self.fc(self.max_pool(x).view(b, c))
        channel_att = self.sigmoid(avg_out + max_out).view(b, c, 1)
        x = x * channel_att
        
        if l > 1:
            # Spatial attention
            avg_out = torch.mean(x, dim=1, keepdim=True)
            max_out, _ = torch.max(x, dim=1, keepdim=True)
            spatial_att = self.sigmoid(self.conv_spatial(torch.cat([avg_out, max_out], dim=1)))
            x = x * spatial_att
        
        return x.squeeze(-1) if l == 1 else x


# ─── Multi-Scale Feature Extractor ───────────────────────────────────────────

class MultiScaleCNN(nn.Module):
    """
    Multi-scale 1D CNN that captures patterns at different temporal resolutions.
    Useful for detecting deepfake artifacts that appear at various time scales.
    """
    def __init__(self, n_features=50, hidden_dim=128, dropout=0.3):
        super().__init__()
        
        # Multiple kernel sizes for multi-scale feature extraction
        self.conv1_small = nn.Conv1d(1, hidden_dim//4, kernel_size=3, padding=1)
        self.conv1_medium = nn.Conv1d(1, hidden_dim//4, kernel_size=5, padding=2)
        self.conv1_large = nn.Conv1d(1, hidden_dim//4, kernel_size=7, padding=3)
        self.conv1_xlarge = nn.Conv1d(1, hidden_dim//4, kernel_size=11, padding=5)
        
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.se1 = SEBlock(hidden_dim)
        
        # Second layer
        self.conv2 = nn.Conv1d(hidden_dim, hidden_dim*2, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(hidden_dim*2)
        self.cbam = CBAMBlock(hidden_dim*2)
        
        # Pooling and classifier
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim*2, 2)
        
        self.act = nn.GELU()
    
    def forward(self, x):
        # x: (batch, n_features)
        x = x.unsqueeze(1)  # (batch, 1, n_features)
        
        # Multi-scale convolutions
        x1 = self.conv1_small(x)
        x2 = self.conv1_medium(x)
        x3 = self.conv1_large(x)
        x4 = self.conv1_xlarge(x)
        
        # Concatenate scales
        x = torch.cat([x1, x2, x3, x4], dim=1)  # (batch, hidden_dim, n_features)
        x = self.act(self.bn1(x))
        x = self.se1(x)
        
        # Second layer with CBAM
        x = self.act(self.bn2(self.conv2(x)))
        x = self.cbam(x)
        
        # Pool and classify
        x = self.pool(x).squeeze(-1)
        x = self.dropout(x)
        return self.fc(x)


# ─── Temporal Attention Network ──────────────────────────────────────────────

class TemporalAttentionNet(nn.Module):
    """
    Network with temporal self-attention for capturing long-range dependencies.
    Effective for detecting inconsistencies in physiological signals.
    """
    def __init__(self, n_features=50, hidden_dim=128, n_heads=4, n_layers=3, dropout=0.3):
        super().__init__()
        
        # Feature embedding
        self.embed = nn.Sequential(
            nn.Linear(n_features, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
        )
        
        # Self-attention layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=n_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        # SE attention after transformer
        self.se = SEBlock(hidden_dim)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 2),
        )
    
    def forward(self, x):
        # x: (batch, n_features)
        x = self.embed(x)  # (batch, hidden_dim)
        x = x.unsqueeze(1)  # (batch, 1, hidden_dim) - single "token"
        
        # Self-attention
        x = self.transformer(x)
        x = x.squeeze(1)  # (batch, hidden_dim)
        
        # SE attention
        x = self.se(x.unsqueeze(-1)).squeeze(-1)
        
        return self.classifier(x)


# ─── Deep Wide & Deep Network ────────────────────────────────────────────────

class WideAndDeep(nn.Module):
    """
    Wide & Deep architecture: combines memorization (wide) with generalization (deep).
    - Wide path: Direct linear connections for feature memorization
    - Deep path: Non-linear transformations for feature interactions
    """
    def __init__(self, n_features=50, hidden_dim=256, dropout=0.3):
        super().__init__()
        
        # Wide path (linear)
        self.wide = nn.Linear(n_features, hidden_dim // 4)
        
        # Deep path (non-linear)
        self.deep = nn.Sequential(
            nn.Linear(n_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
        )
        
        # SE attention on deep features
        self.se = SEBlock(hidden_dim // 2)
        
        # Combine and classify
        combined_dim = hidden_dim // 4 + hidden_dim // 2
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, combined_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(combined_dim // 2, 2),
        )
    
    def forward(self, x):
        wide_out = self.wide(x)
        deep_out = self.deep(x)
        deep_out = self.se(deep_out.unsqueeze(-1)).squeeze(-1)
        
        combined = torch.cat([wide_out, deep_out], dim=1)
        return self.classifier(combined)


# ─── Test model architectures ────────────────────────────────────────────────

print("\nTesting advanced architectures...")
test_input = torch.randn(4, N_FEATURES).to(device)

for name, model_class in [
    ('MultiScaleCNN', MultiScaleCNN),
    ('TemporalAttentionNet', TemporalAttentionNet),
    ('WideAndDeep', WideAndDeep),
]:
    model = model_class(N_FEATURES).to(device)
    with torch.no_grad():
        out = model(test_input)
    params = sum(p.numel() for p in model.parameters())
    print(f"  ✓ {name}: output {out.shape}, params {params:,}")
    del model

torch.cuda.empty_cache()
print("\n✓ Advanced DL architectures defined!")

## 4. Training Infrastructure

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ENHANCED TRAINING INFRASTRUCTURE
# ═══════════════════════════════════════════════════════════════════════════════
# Features:
# - Mixed precision training (FP16 on P100)
# - Data augmentation (Gaussian noise, feature dropout)
# - Cosine annealing with warm restarts
# - Gradient clipping
# - Label smoothing
# - Stochastic Weight Averaging (SWA)
# - Advanced early stopping with model checkpointing
# ═══════════════════════════════════════════════════════════════════════════════

import gc
from torch.cuda.amp import GradScaler, autocast

# ─── Data Augmentation ───────────────────────────────────────────────────────

class FeatureAugmentation:
    """
    Feature-level augmentation for tabular/signal data:
    - Gaussian noise injection
    - Feature dropout (random zeroing)
    - Feature scaling jitter
    """
    def __init__(self, noise_std=0.02, dropout_prob=0.1, scale_range=(0.95, 1.05)):
        self.noise_std = noise_std
        self.dropout_prob = dropout_prob
        self.scale_range = scale_range
        self.training = True  # Default to training mode
    
    def __call__(self, x):
        if self.training:
            # Gaussian noise
            if self.noise_std > 0:
                noise = torch.randn_like(x) * self.noise_std
                x = x + noise
            
            # Feature dropout
            if self.dropout_prob > 0:
                mask = torch.rand_like(x) > self.dropout_prob
                x = x * mask.float()
            
            # Scale jitter
            if self.scale_range[0] < self.scale_range[1]:
                scale = torch.empty(x.size()).uniform_(*self.scale_range).to(x.device)
                x = x * scale
        
        return x


# ─── Advanced Early Stopping ─────────────────────────────────────────────────

class EarlyStopping:
    """
    Advanced early stopping with:
    - Best model checkpointing
    - Delta threshold for improvement
    - Support for maximizing metrics (like AUC)
    """
    def __init__(self, patience=15, min_delta=1e-4, mode='min'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.should_stop = False
        self.best_state = None
        self.best_epoch = 0
    
    def __call__(self, score, model, epoch):
        if self.mode == 'min':
            is_better = self.best_score is None or score < self.best_score - self.min_delta
        else:
            is_better = self.best_score is None or score > self.best_score + self.min_delta
        
        if is_better:
            self.best_score = score
            self.counter = 0
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.best_epoch = epoch
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True


# ─── Label Smoothing Loss ────────────────────────────────────────────────────

class LabelSmoothingCrossEntropy(nn.Module):
    """Cross entropy with label smoothing for regularization."""
    def __init__(self, smoothing=0.1, weight=None):
        super().__init__()
        self.smoothing = smoothing
        self.weight = weight
    
    def forward(self, pred, target):
        n_classes = pred.size(-1)
        log_preds = torch.log_softmax(pred, dim=-1)
        
        # Create smoothed targets
        targets = torch.zeros_like(log_preds).scatter_(1, target.unsqueeze(1), 1)
        targets = targets * (1 - self.smoothing) + self.smoothing / n_classes
        
        if self.weight is not None:
            loss = -(targets * log_preds * self.weight).sum(dim=-1)
        else:
            loss = -(targets * log_preds).sum(dim=-1)
        
        return loss.mean()


# ─── Training Functions ──────────────────────────────────────────────────────

def train_epoch_amp(model, loader, criterion, optimizer, scaler, augment=None):
    """Training with mixed precision (AMP)."""
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        # Apply augmentation
        if augment is not None:
            augment.training = True
            X_batch = augment(X_batch)
        
        optimizer.zero_grad()
        
        # Mixed precision forward pass
        with autocast():
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
        
        # Scaled backward pass
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item() * X_batch.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == y_batch).sum().item()
        total += X_batch.size(0)
    
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_amp(model, loader, criterion):
    """Evaluation with mixed precision."""
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_probs, all_labels = [], []
    
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        with autocast():
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
        
        total_loss += loss.item() * X_batch.size(0)
        probs = torch.softmax(outputs.float(), dim=1)
        _, preds = torch.max(outputs, 1)
        correct += (preds == y_batch).sum().item()
        total += X_batch.size(0)
        all_probs.append(probs[:, 1].cpu().numpy())
        all_labels.append(y_batch.cpu().numpy())
    
    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    return total_loss / total, correct / total, all_probs, all_labels


def train_model(model, name, train_loader, val_loader, test_loader,
                epochs=100, lr=1e-3, weight_decay=1e-4, use_amp=True):
    """
    Full training pipeline with:
    - Mixed precision (AMP)
    - Data augmentation
    - Cosine annealing + warm restarts
    - Label smoothing
    - Early stopping (patience=15)
    - Gradient clipping
    """
    print(f"\n{'='*70}")
    print(f"Training: {name}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Mixed Precision: {use_amp} | Epochs: {epochs} | LR: {lr}")
    print(f"{'='*70}")
    
    # Class-weighted loss with label smoothing
    class_weights_tensor = torch.FloatTensor(
        [1.0 / class_counts[0], 1.0 / class_counts[1]]
    ).to(device)
    class_weights_tensor = class_weights_tensor / class_weights_tensor.sum() * 2
    criterion = LabelSmoothingCrossEntropy(smoothing=0.1, weight=class_weights_tensor)
    
    # Optimizer with weight decay
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # Learning rate scheduler: CosineAnnealingLR (per-epoch stepping)
    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=epochs,
        eta_min=lr / 100
    )
        final_div_factor=100,
    )
    
    # Mixed precision scaler
    scaler = GradScaler() if use_amp else None
    
    # Data augmentation
    augment = FeatureAugmentation(noise_std=0.02, dropout_prob=0.1)
    
    # Early stopping
    early_stop = EarlyStopping(patience=15, mode='max')  # Maximize AUC
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_auc': []}
    
    t0 = time()
    for epoch in range(1, epochs + 1):
        if use_amp:
            train_loss, train_acc = train_epoch_amp(
                model, train_loader, criterion, optimizer, scaler, augment
            )
        else:
            train_loss, train_acc = train_epoch_amp(
                model, train_loader, criterion, optimizer, 
                GradScaler(enabled=False), augment
            )
        
        val_loss, val_acc, val_probs, val_labels = evaluate_amp(model, val_loader, criterion)
        val_auc = roc_auc_score(val_labels, val_probs)
        
        # Step scheduler
        scheduler.step()
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['val_auc'].append(val_auc)
        
        if epoch % 10 == 0 or epoch == 1:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"  Epoch {epoch:3d}/{epochs} | "
                  f"Train: {train_loss:.4f}/{train_acc:.4f} | "
                  f"Val: {val_loss:.4f}/{val_acc:.4f} | "
                  f"AUC: {val_auc:.4f} | LR: {current_lr:.2e}")
        
        # Early stopping based on AUC
        early_stop(val_auc, model, epoch)
        if early_stop.should_stop:
            print(f"  Early stopping at epoch {epoch} (best epoch: {early_stop.best_epoch})")
            break
    
    elapsed = time() - t0
    print(f"  Training time: {elapsed:.1f}s")
    
    # Restore best model
    if early_stop.best_state is not None:
        model.load_state_dict(early_stop.best_state)
        model.to(device)
        print(f"  Restored best model from epoch {early_stop.best_epoch}")
    
    # Final test evaluation
    test_loss, test_acc, test_probs, test_labels = evaluate_amp(
        model, test_loader, criterion
    )
    test_preds = (test_probs >= 0.5).astype(int)
    
    test_auc = roc_auc_score(test_labels, test_probs)
    test_f1 = f1_score(test_labels, test_preds, zero_division=0)
    test_prec = precision_score(test_labels, test_preds, zero_division=0)
    test_rec = recall_score(test_labels, test_preds, zero_division=0)
    
    print(f"\n  Test Results:")
    print(f"    Accuracy:  {test_acc:.4f}")
    print(f"    AUC:       {test_auc:.4f}")
    print(f"    F1:        {test_f1:.4f}")
    print(f"    Precision: {test_prec:.4f}")
    print(f"    Recall:    {test_rec:.4f}")
    
    metrics = {
        'accuracy': test_acc, 'auc': test_auc, 'f1': test_f1,
        'precision': test_prec, 'recall': test_rec,
        'time': elapsed, 'epochs_trained': len(history['train_loss']),
        'best_val_auc': early_stop.best_score,
        'best_epoch': early_stop.best_epoch,
    }
    
    # Clean up GPU memory
    torch.cuda.empty_cache()
    gc.collect()
    
    return model, history, metrics, test_probs, test_labels


print("Enhanced training infrastructure ready!")
print("  - Mixed precision (AMP)")
print("  - Data augmentation (noise, dropout)")
print("  - CosineAnnealingLR scheduler")
print("  - Label smoothing")
print("  - Early stopping (patience=15, maximize AUC)")


In [ ]:
import torch.nn.functional as F

# ═══════════════════════════════════════════════════════════════════════════════
# ADVANCED: SIGNAL AUGMENTATION + FOCAL LOSS
# ═══════════════════════════════════════════════════════════════════════════════
# Signal augmentation improves generalization for rPPG-based detection
# Focal Loss handles hard examples and class imbalance
# ═══════════════════════════════════════════════════════════════════════════════

print("="*70)
print("INITIALIZING ADVANCED TRAINING UTILITIES")
print("="*70)

# ─── Signal Augmentation Functions ───────────────────────────────────────────

class SignalAugmentation:
    """
    Augmentation techniques for rPPG tabular features.
    Simulates real-world variations in physiological signals.
    """
    
    def __init__(self, p=0.5):
        self.p = p  # Probability of applying each augmentation
    
    def gaussian_noise(self, x, std=0.02):
        """Add Gaussian noise to features."""
        if np.random.random() < self.p:
            noise = np.random.normal(0, std, x.shape)
            return x + noise
        return x
    
    def amplitude_scaling(self, x, scale_range=(0.9, 1.1)):
        """Scale feature amplitudes."""
        if np.random.random() < self.p:
            scale = np.random.uniform(*scale_range)
            return x * scale
        return x
    
    def feature_dropout(self, x, drop_prob=0.1):
        """Randomly zero out some features."""
        if np.random.random() < self.p:
            mask = np.random.random(x.shape) > drop_prob
            return x * mask
        return x
    
    def mixup(self, x1, y1, x2, y2, alpha=0.2):
        """
        Mixup augmentation: blend two samples.
        """
        lam = np.random.beta(alpha, alpha)
        x_mixed = lam * x1 + (1 - lam) * x2
        y_mixed = lam * y1 + (1 - lam) * y2
        return x_mixed, y_mixed
    
    def cutmix_features(self, x1, x2, alpha=0.2):
        """
        CutMix for features: replace random subset of features.
        """
        if np.random.random() < self.p:
            lam = np.random.beta(alpha, alpha)
            n_features = x1.shape[-1]
            n_swap = int(n_features * (1 - lam))
            indices = np.random.choice(n_features, n_swap, replace=False)
            x_mixed = x1.copy()
            x_mixed[..., indices] = x2[..., indices]
            return x_mixed
        return x1
    
    def __call__(self, x):
        """Apply random augmentations."""
        x = self.gaussian_noise(x)
        x = self.amplitude_scaling(x)
        x = self.feature_dropout(x)
        return x


class AugmentedDataset(torch.utils.data.Dataset):
    """Dataset with on-the-fly augmentation."""
    
    def __init__(self, X, y, augment=True, mixup_prob=0.3):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.augment = augment
        self.mixup_prob = mixup_prob
        self.augmenter = SignalAugmentation(p=0.5)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        x = self.X[idx].numpy()
        y = self.y[idx]
        
        if self.augment:
            x = self.augmenter(x)
            
            # Mixup with small probability
            if np.random.random() < self.mixup_prob:
                idx2 = np.random.randint(len(self.X))
                x2 = self.X[idx2].numpy()
                y2 = self.y[idx2]
                
                alpha = np.random.beta(0.2, 0.2)
                x = alpha * x + (1 - alpha) * x2
                # For classification, keep original label
        
        return torch.tensor(x, dtype=torch.float32), y

print("✓ Signal augmentation utilities defined")

# ─── Focal Loss Implementation ───────────────────────────────────────────────

class FocalLoss(nn.Module):
    """
    Focal Loss for handling class imbalance and hard examples.
    
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    
    - gamma > 0 reduces loss for well-classified examples
    - alpha balances positive/negative classes
    """
    
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha  # Class weights [alpha_neg, alpha_pos]
        self.gamma = gamma  # Focusing parameter
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        # inputs: (N, C) logits
        # targets: (N,) class indices
        
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)  # p_t
        
        focal_weight = (1 - pt) ** self.gamma
        focal_loss = focal_weight * ce_loss
        
        if self.alpha is not None:
            alpha_t = torch.tensor(self.alpha, device=inputs.device)[targets]
            focal_loss = alpha_t * focal_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


class LabelSmoothingFocalLoss(nn.Module):
    """
    Combined Label Smoothing + Focal Loss for maximum robustness.
    """
    
    def __init__(self, smoothing=0.1, gamma=2.0, alpha=None):
        super().__init__()
        self.smoothing = smoothing
        self.gamma = gamma
        self.alpha = alpha
        self.confidence = 1.0 - smoothing
    
    def forward(self, inputs, targets):
        n_classes = inputs.size(-1)
        
        # Create smooth labels
        true_dist = torch.zeros_like(inputs)
        true_dist.fill_(self.smoothing / (n_classes - 1))
        true_dist.scatter_(1, targets.unsqueeze(1), self.confidence)
        
        # Log softmax
        log_probs = F.log_softmax(inputs, dim=-1)
        
        # Focal weight
        probs = torch.exp(log_probs)
        pt = probs.gather(1, targets.unsqueeze(1)).squeeze()
        focal_weight = (1 - pt) ** self.gamma
        
        # Weighted cross-entropy
        loss = -torch.sum(true_dist * log_probs, dim=-1)
        loss = focal_weight * loss
        
        if self.alpha is not None:
            alpha_t = torch.tensor(self.alpha, device=inputs.device)[targets]
            loss = alpha_t * loss
        
        return loss.mean()

print("✓ Focal Loss implementations defined")

# ─── Class Weights Calculation ───────────────────────────────────────────────

def compute_class_weights(y):
    """Compute balanced class weights."""
    class_counts = np.bincount(y)
    total = len(y)
    weights = total / (len(class_counts) * class_counts)
    return weights / weights.sum()  # Normalize

class_weights = compute_class_weights(y_tr)
print(f"\nClass distribution:")
print(f"  Real: {np.sum(y_tr == 0)} samples")
print(f"  Fake: {np.sum(y_tr == 1)} samples")
print(f"  Class weights: {class_weights}")

# ─── Weighted Sampler for Balanced Batches ───────────────────────────────────

def create_weighted_sampler(y):
    """Create sampler that balances classes in each batch."""
    class_weights = compute_class_weights(y)
    sample_weights = class_weights[y]
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(y),
        replacement=True
    )

print("✓ Class balancing utilities defined")

# ─── Combined Loss Function ──────────────────────────────────────────────────

USE_FOCAL_LOSS = True  # Enable focal loss for better hard example mining

if USE_FOCAL_LOSS:
    # Use label smoothing + focal loss
    dl_criterion = LabelSmoothingFocalLoss(
        smoothing=0.1, 
        gamma=2.0,
        alpha=class_weights.tolist()
    )
    print("\n✓ Using LabelSmoothingFocalLoss (gamma=2.0, smoothing=0.1)")
else:
    dl_criterion = LabelSmoothingCrossEntropy(smoothing=0.1)
    print("\n✓ Using LabelSmoothingCrossEntropy")

print("="*70)

## 5.5 (Optional) Optuna Hyperparameter Optimization

This section uses Optuna for automatic hyperparameter tuning of the best DL model.
Skip this if you want to use the pre-tuned configurations.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# OPTUNA HYPERPARAMETER OPTIMIZATION (ENABLED)
# ═══════════════════════════════════════════════════════════════════════════════
# Bayesian optimization for finding optimal DL hyperparameters
# Expected runtime: ~30-45 min on Kaggle P100
# Expected improvement: +3-5% accuracy
# ═══════════════════════════════════════════════════════════════════════════════

RUN_OPTUNA = True  # ENABLED for maximum accuracy

# Initialize DL tracking dictionaries if not already defined
if 'dl_results' not in globals():
    dl_results = {}
if 'dl_models' not in globals():
    dl_models = {}
if 'dl_predictions' not in globals():
    dl_predictions = {}


if RUN_OPTUNA and HAS_OPTUNA:
    import optuna
    from optuna.trial import TrialState
    import gc
    
    print("="*70)
    print("OPTUNA HYPERPARAMETER OPTIMIZATION - ENABLED")
    print("="*70)
    
    # ═══════════════════════════════════════════════════════════════════════════
    # Objective 1: Optimize PhysNet MLP
    # ═══════════════════════════════════════════════════════════════════════════
    def objective_physnet(trial):
        """Optuna objective function for PhysNet MLP."""
        
        # Hyperparameters to optimize
        hidden_dim = trial.suggest_categorical('hidden_dim', [128, 192, 256, 320, 384])
        n_blocks = trial.suggest_int('n_blocks', 2, 7)
        dropout = trial.suggest_float('dropout', 0.1, 0.5)
        lr = trial.suggest_float('lr', 1e-5, 5e-2, log=True)
        weight_decay = trial.suggest_float('weight_decay', 1e-7, 1e-2, log=True)
        batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
        
        # Label smoothing
        smoothing = trial.suggest_float('smoothing', 0.0, 0.2)
        
        # Create model
        model = PhysNetMLP(
            N_FEATURES, 
            hidden_dim=hidden_dim, 
            n_blocks=n_blocks, 
            dropout=dropout
        ).to(device)
        
        # Create data loaders with trial batch size
        train_ds = TensorDataset(
            torch.tensor(X_tr, dtype=torch.float32),
            torch.tensor(y_tr, dtype=torch.long)
        )
        val_ds = TensorDataset(
            torch.tensor(X_val, dtype=torch.float32),
            torch.tensor(y_val, dtype=torch.long)
        )
        
        trial_train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        trial_val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
        
        # Training setup
        criterion = LabelSmoothingCrossEntropy(smoothing=smoothing)
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = CosineAnnealingLR(optimizer, T_max=40, eta_min=1e-7)
        scaler = GradScaler()
        
        # Quick training (40 epochs for Optuna)
        best_val_auc = 0
        patience_counter = 0
        
        for epoch in range(40):
            # Train
            model.train()
            for X_batch, y_batch in trial_train_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()
                with autocast():
                    outputs = model(X_batch)
                    loss = criterion(outputs, y_batch)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            
            scheduler.step()
            
            # Validate
            model.eval()
            val_probs = []
            val_labels = []
            with torch.no_grad():
                for X_batch, y_batch in trial_val_loader:
                    X_batch = X_batch.to(device)
                    with autocast():
                        outputs = model(X_batch)
                        probs = torch.softmax(outputs, dim=1)[:, 1]
                    val_probs.extend(probs.cpu().numpy())
                    val_labels.extend(y_batch.numpy())
            
            val_auc = roc_auc_score(val_labels, val_probs)
            
            # Report to Optuna
            trial.report(val_auc, epoch)
            
            # Handle pruning
            if trial.should_prune():
                raise optuna.TrialPruned()
            
            # Early stopping
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= 7:
                    break
        
        # Cleanup
        del model, trial_train_loader, trial_val_loader
        torch.cuda.empty_cache()
        gc.collect()
        
        return best_val_auc
    
    # ═══════════════════════════════════════════════════════════════════════════
    # Run Optuna Study
    # ═══════════════════════════════════════════════════════════════════════════
    N_OPTUNA_TRIALS = 100  # 100 trials for thorough search
    OPTUNA_TIMEOUT = 2400  # 40 minutes max
    
    print(f"\nRunning {N_OPTUNA_TRIALS} trials (timeout: {OPTUNA_TIMEOUT//60} min)...")
    print("Using TPE Sampler + Median Pruner for efficient search")
    
    # Create study with efficient sampler
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(
            seed=SEED,
            n_startup_trials=20,  # Random exploration first
            multivariate=True,  # Consider parameter correlations
        ),
        pruner=optuna.pruners.HyperbandPruner(
            min_resource=5,
            max_resource=40,
            reduction_factor=3,
        ),
    )
    
    # Suppress Optuna logs
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    
    t0 = time()
    study.optimize(
        objective_physnet, 
        n_trials=N_OPTUNA_TRIALS, 
        timeout=OPTUNA_TIMEOUT,
        gc_after_trial=True,
        show_progress_bar=True,
    )
    optuna_time = time() - t0
    
    # Results
    print(f"\n{'='*70}")
    print("OPTUNA OPTIMIZATION COMPLETE")
    print(f"{'='*70}")
    print(f"  Total trials: {len(study.trials)}")
    print(f"  Completed trials: {len([t for t in study.trials if t.state == TrialState.COMPLETE])}")
    print(f"  Pruned trials: {len([t for t in study.trials if t.state == TrialState.PRUNED])}")
    print(f"  Time: {optuna_time/60:.1f} minutes")
    
    print(f"\nBest trial:")
    print(f"  Value (Val AUC): {study.best_trial.value:.4f}")
    print(f"  Params:")
    for key, value in study.best_trial.params.items():
        print(f"    {key}: {value}")
    
    # Save best params
    best_optuna_params = study.best_trial.params
    joblib.dump(best_optuna_params, os.path.join(OUTPUT_DIR, 'optuna_best_params.joblib'))
    
    # ═══════════════════════════════════════════════════════════════════════════
    # Train final model with Optuna-optimized params
    # ═══════════════════════════════════════════════════════════════════════════
    print(f"\n{'─'*70}")
    print("Training FINAL MODEL with Optuna-optimized parameters...")
    
    optuna_model = PhysNetMLP(
        N_FEATURES,
        hidden_dim=best_optuna_params['hidden_dim'],
        n_blocks=best_optuna_params['n_blocks'],
        dropout=best_optuna_params['dropout'],
    ).to(device)
    
    # Final training with more epochs
    criterion = LabelSmoothingCrossEntropy(smoothing=best_optuna_params.get('smoothing', 0.1))
    optimizer = optim.AdamW(
        optuna_model.parameters(), 
        lr=best_optuna_params['lr'],
        weight_decay=best_optuna_params['weight_decay']
    )
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-7)
    scaler = GradScaler()
    
    best_val_auc = 0
    best_state = None
    patience = 0
    
    for epoch in range(100):  # More epochs for final training
        optuna_model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            with autocast():
                outputs = optuna_model(X_batch)
                loss = criterion(outputs, y_batch)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        
        scheduler.step()
        
        # Validate
        val_loss, val_acc, val_probs, val_labels = evaluate_amp(optuna_model, val_loader, criterion)
        val_auc = roc_auc_score(val_labels, val_probs)
        
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = optuna_model.state_dict().copy()
            patience = 0
        else:
            patience += 1
            if patience >= 15:
                print(f"  Early stopping at epoch {epoch+1}")
                break
        
        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1:3d} | Val AUC: {val_auc:.4f}")
    
    # Load best state
    optuna_model.load_state_dict(best_state)
    
    # Final evaluation
    optuna_model.eval()
    test_probs = []
    test_labels = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            with autocast():
                outputs = optuna_model(X_batch)
                probs = torch.softmax(outputs, dim=1)[:, 1]
            test_probs.extend(probs.cpu().numpy())
            test_labels.extend(y_batch.numpy())
    
    test_probs = np.array(test_probs)
    test_preds = (test_probs > 0.5).astype(int)
    
    optuna_acc = accuracy_score(y_test_dl, test_preds)
    optuna_auc = roc_auc_score(y_test_dl, test_probs)
    optuna_f1 = f1_score(y_test_dl, test_preds, zero_division=0)
    
    print(f"\n{'='*70}")
    print("OPTUNA-OPTIMIZED MODEL RESULTS")
    print(f"{'='*70}")
    print(f"  Accuracy:  {optuna_acc:.4f}")
    print(f"  AUC:       {optuna_auc:.4f}")
    print(f"  F1:        {optuna_f1:.4f}")
    
    # Save model
    torch.save({
        'model_state_dict': optuna_model.state_dict(),
        'params': best_optuna_params,
        'metrics': {'accuracy': optuna_acc, 'auc': optuna_auc, 'f1': optuna_f1}
    }, os.path.join(OUTPUT_DIR, 'Optuna_PhysNet_best.pth'))
    
    # Add to results
    dl_results['Optuna_PhysNet'] = {
        'accuracy': optuna_acc,
        'precision': precision_score(y_test_dl, test_preds, zero_division=0),
        'recall': recall_score(y_test_dl, test_preds, zero_division=0),
        'f1': optuna_f1,
        'auc': optuna_auc,
        'time': optuna_time,
        'epochs_trained': epoch + 1,
    }
    dl_models['Optuna_PhysNet'] = optuna_model
    dl_predictions['Optuna_PhysNet'] = {'probs': test_probs, 'labels': test_labels}
    
    print(f"\nOptuna-optimized model saved!")

else:
    if not HAS_OPTUNA:
        print("Optuna not available. Install with: pip install optuna")
    else:
        print("Optuna optimization skipped (RUN_OPTUNA = False)")

## 5. Train All DL Models (Enhanced Training)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ENHANCED MODEL CONFIGURATIONS
# ═══════════════════════════════════════════════════════════════════════════════
# - 100 epochs max (early stopping will kick in earlier)
# - Optimized learning rates per architecture
# - Tuned regularization (dropout, weight decay)
# ═══════════════════════════════════════════════════════════════════════════════

EPOCHS = 100  # Increased from 60 - early stopping will prevent overfit

# Preserve any existing models (e.g., Optuna)
_new_models = {
    'CNN_1D': {
        'model': DeepfakeCNN1D(N_FEATURES, dropout=0.25),
        'lr': 1e-3,
        'weight_decay': 5e-5,
    },
    'BiLSTM_Attention': {
        'model': DeepfakeBiLSTM(N_FEATURES, hidden_dim=128, n_layers=3, dropout=0.25),
        'lr': 5e-4,
        'weight_decay': 1e-4,
    },
    'CNN_BiLSTM': {
        'model': DeepfakeCNNBiLSTM(N_FEATURES, cnn_channels=64, lstm_hidden=128,
                                    lstm_layers=2, dropout=0.25),
        'lr': 5e-4,
        'weight_decay': 1e-4,
    },
    'Transformer': {
        'model': DeepfakeTransformer(N_FEATURES, d_model=64, nhead=4,
                                     num_layers=4, dim_ff=256, dropout=0.25),
        'lr': 3e-4,
        'weight_decay': 5e-4,
    },
    'PhysNet_MLP': {
        'model': PhysNetMLP(N_FEATURES, hidden_dim=256, n_blocks=5, dropout=0.25),
        'lr': 1e-3,
        'weight_decay': 1e-4,
    },
    'MultiScale_CNN': {
        'model': MultiScaleCNN(N_FEATURES, hidden_dim=128, dropout=0.25),
        'lr': 0.001,
        'weight_decay': 1e-4,
    },
    'TemporalAttention': {
        'model': TemporalAttentionNet(N_FEATURES, hidden_dim=128, n_heads=4, n_layers=2, dropout=0.25),
        'lr': 0.0005,
        'weight_decay': 1e-4,
    },
    'WideDeep': {
        'model': WideAndDeep(N_FEATURES, hidden_dim=256, dropout=0.25),
        'lr': 0.001,
        'weight_decay': 1e-4,
    },
}

# Merge with existing dl_models (preserves Optuna)
if 'dl_models' not in globals():
    dl_models = {}
dl_models.update(_new_models)

print(f"{'='*70}")
print(f"MODEL CONFIGURATIONS (Enhanced)")
print(f"{'='*70}")
print(f"Max Epochs: {EPOCHS} (with early stopping patience=15)")
print(f"Mixed Precision: Enabled")
print(f"Data Augmentation: Gaussian noise + Feature dropout")
print(f"{'='*70}")

total_params = 0
for name, cfg in dl_models.items():
    n_params = sum(p.numel() for p in cfg['model'].parameters())
    total_params += n_params
    print(f"  {name}")
    print(f"    Parameters: {n_params:,}")
    print(f"    LR: {cfg['lr']:.0e} | Weight Decay: {cfg['weight_decay']:.0e}")

print(f"{'='*70}")
print(f"Total parameters across all models: {total_params:,}")
print(f"{'='*70}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRAIN ALL DEEP LEARNING MODELS
# ═══════════════════════════════════════════════════════════════════════════════

# Preserve existing results (e.g., Optuna)
dl_results = dl_results if 'dl_results' in globals() else {}
dl_histories = {}
dl_predictions = dl_predictions if 'dl_predictions' in globals() else {}

print("="*70)
print("STARTING DEEP LEARNING TRAINING")
print("="*70)

for name, cfg in dl_models.items():
    model = cfg['model'].to(device)
    
    trained_model, history, metrics, probs, labels = train_model(
        model, name,
        train_loader, val_loader, test_loader,
        epochs=EPOCHS,
        lr=cfg['lr'],
        weight_decay=cfg['weight_decay'],
        use_amp=True,  # Mixed precision
    )
    
    dl_results[name] = metrics
    dl_histories[name] = history
    dl_predictions[name] = {'probs': probs, 'labels': labels}
    
    # Save model
    torch.save(trained_model.state_dict(), os.path.join(OUTPUT_DIR, f'{name}_best.pt'))
    
    # Clean up
    del model, trained_model
    torch.cuda.empty_cache()
    gc.collect()
    
    print()

print("="*70)
print("ALL DEEP LEARNING MODELS TRAINED!")
print("="*70)

# ─── Summary ─────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("DL RESULTS SUMMARY (sorted by Test AUC)")
print("="*70)
sorted_dl = sorted(dl_results.items(), key=lambda x: x[1]['auc'], reverse=True)
for i, (name, r) in enumerate(sorted_dl, 1):
    print(f"  {i}. {name:20s} | AUC: {r['auc']:.4f} | F1: {r['f1']:.4f} | Acc: {r['accuracy']:.4f}")


## 6. DL Ensemble Methods

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DEEP LEARNING ENSEMBLE METHODS
# ═══════════════════════════════════════════════════════════════════════════════
# 1. Simple Average Ensemble
# 2. Weighted Average Ensemble (AUC-based weights)
# 3. Rank-based Ensemble
# ═══════════════════════════════════════════════════════════════════════════════

# ─── 1. Simple Average Ensemble ──────────────────────────────────────────────
print("="*70)
print("DEEP LEARNING ENSEMBLE")
print("="*70)

# Stack all probabilities
all_probs = np.stack([dl_predictions[name]['probs'] for name in dl_models.keys()], axis=0)
test_labels = dl_predictions[list(dl_models.keys())[0]]['labels']

# Simple average
avg_probs = all_probs.mean(axis=0)
avg_preds = (avg_probs >= 0.5).astype(int)

avg_acc = accuracy_score(test_labels, avg_preds)
avg_auc = roc_auc_score(test_labels, avg_probs)
avg_f1 = f1_score(test_labels, avg_preds, zero_division=0)
avg_prec = precision_score(test_labels, avg_preds, zero_division=0)
avg_rec = recall_score(test_labels, avg_preds, zero_division=0)

print(f"\n1. Simple Average Ensemble:")
print(f"   Accuracy:  {avg_acc:.4f}")
print(f"   AUC:       {avg_auc:.4f}")
print(f"   F1:        {avg_f1:.4f}")
print(f"   Precision: {avg_prec:.4f}")
print(f"   Recall:    {avg_rec:.4f}")

dl_results['DL_Ensemble_Avg'] = {
    'accuracy': avg_acc, 'auc': avg_auc, 'f1': avg_f1,
    'precision': avg_prec, 'recall': avg_rec,
    'time': 0, 'epochs_trained': 0,
}

# ─── 2. Weighted Average Ensemble (AUC-based) ────────────────────────────────
# Weight each model by its validation AUC
model_aucs = np.array([dl_results[name]['auc'] for name in dl_models.keys()])
weights = model_aucs / model_aucs.sum()

print(f"\n2. Weighted Average Ensemble (AUC-based weights):")
for i, name in enumerate(dl_models.keys()):
    print(f"   {name}: weight = {weights[i]:.4f}")

weighted_probs = (all_probs * weights[:, None]).sum(axis=0)
weighted_preds = (weighted_probs >= 0.5).astype(int)

weighted_acc = accuracy_score(test_labels, weighted_preds)
weighted_auc = roc_auc_score(test_labels, weighted_probs)
weighted_f1 = f1_score(test_labels, weighted_preds, zero_division=0)
weighted_prec = precision_score(test_labels, weighted_preds, zero_division=0)
weighted_rec = recall_score(test_labels, weighted_preds, zero_division=0)

print(f"\n   Results:")
print(f"   Accuracy:  {weighted_acc:.4f}")
print(f"   AUC:       {weighted_auc:.4f}")
print(f"   F1:        {weighted_f1:.4f}")
print(f"   Precision: {weighted_prec:.4f}")
print(f"   Recall:    {weighted_rec:.4f}")

dl_results['DL_Ensemble_Weighted'] = {
    'accuracy': weighted_acc, 'auc': weighted_auc, 'f1': weighted_f1,
    'precision': weighted_prec, 'recall': weighted_rec,
    'time': 0, 'epochs_trained': 0,
}

# ─── 3. Top-3 Ensemble ───────────────────────────────────────────────────────
# Use only top 3 models by AUC
sorted_models = sorted(dl_results.items(), key=lambda x: x[1]['auc'], reverse=True)
top3_names = [n for n, _ in sorted_models[:3] if n in dl_models]

print(f"\n3. Top-3 Ensemble ({', '.join(top3_names)}):")

top3_probs = np.stack([dl_predictions[name]['probs'] for name in top3_names], axis=0)
top3_avg = top3_probs.mean(axis=0)
top3_preds = (top3_avg >= 0.5).astype(int)

top3_acc = accuracy_score(test_labels, top3_preds)
top3_auc = roc_auc_score(test_labels, top3_avg)
top3_f1 = f1_score(test_labels, top3_preds, zero_division=0)
top3_prec = precision_score(test_labels, top3_preds, zero_division=0)
top3_rec = recall_score(test_labels, top3_preds, zero_division=0)

print(f"   Accuracy:  {top3_acc:.4f}")
print(f"   AUC:       {top3_auc:.4f}")
print(f"   F1:        {top3_f1:.4f}")
print(f"   Precision: {top3_prec:.4f}")
print(f"   Recall:    {top3_rec:.4f}")

dl_results['DL_Ensemble_Top3'] = {
    'accuracy': top3_acc, 'auc': top3_auc, 'f1': top3_f1,
    'precision': top3_prec, 'recall': top3_rec,
    'time': 0, 'epochs_trained': 0,
}

# ─── Final DL Rankings ───────────────────────────────────────────────────────
print("\n" + "="*70)
print("FINAL DL MODEL RANKINGS (including ensembles)")
print("="*70)
final_dl_sorted = sorted(dl_results.items(), key=lambda x: x[1]['auc'], reverse=True)
for i, (name, r) in enumerate(final_dl_sorted, 1):
    marker = "★" if i <= 3 else " "
    print(f"  {marker} {i:2d}. {name:25s} AUC: {r['auc']:.4f} | F1: {r['f1']:.4f} | Acc: {r['accuracy']:.4f}")

# Save best ensemble probabilities for later use
best_ensemble = final_dl_sorted[0][0]
print(f"\nBest DL Model/Ensemble: {best_ensemble}")


## 7. Results Comparison

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DL RESULTS TABLE
# ═══════════════════════════════════════════════════════════════════════════════

dl_results_df = pd.DataFrame(dl_results).T
dl_results_df = dl_results_df.sort_values('auc', ascending=False)

# Format for display
display_cols = ['accuracy', 'auc', 'f1', 'precision', 'recall', 'epochs_trained', 'time']
dl_display = dl_results_df[display_cols].copy()

# Round values
for col in dl_display.columns:
    if col == 'time':
        dl_display[col] = dl_display[col].apply(lambda x: f'{x:.1f}s' if x > 0 else '-')
    elif col == 'epochs_trained':
        dl_display[col] = dl_display[col].apply(lambda x: f'{int(x)}' if x > 0 else '-')
    else:
        dl_display[col] = dl_display[col].apply(lambda x: f'{x:.4f}')

print("="*90)
print("DEEP LEARNING MODEL RESULTS (sorted by Test AUC)")
print("="*90)
print(dl_display.to_string())
print("="*90)

# ─── Highlight best model ────────────────────────────────────────────────────
best_dl_name = dl_results_df.index[0]
best_dl_auc = dl_results_df.loc[best_dl_name, 'auc']
print(f"\n★ BEST DL MODEL: {best_dl_name}")
print(f"  Test AUC: {best_dl_auc:.4f}")

# Save results
dl_results_df.to_csv(os.path.join(OUTPUT_DIR, 'dl_results_detailed.csv'))
joblib.dump(dl_results, os.path.join(OUTPUT_DIR, 'dl_results.joblib'))
print("\nDL results saved.")


In [ ]:
# Use appropriate test variable
if 'X_test_dl_scaled' not in globals():
    if 'X_test_scaled' in globals():
        X_test_dl_scaled = X_test_scaled
        y_test_dl = y_test
    else:
        raise NameError("Test data not available. Run data preparation first.")

# SignalAugmentation class for TTA
# Using SignalAugmentation from Cell 52
# ═══════════════════════════════════════════════════════════════════════════════
# TEST-TIME AUGMENTATION (TTA) FOR MAXIMUM ACCURACY
# ═══════════════════════════════════════════════════════════════════════════════
# Averages predictions over multiple augmented versions of test data
# Typically improves accuracy by 1-3%
# ═══════════════════════════════════════════════════════════════════════════════

print("="*70)
print("TEST-TIME AUGMENTATION (TTA)")
print("="*70)

class TestTimeAugmentation:
    """
    Apply multiple augmentations at test time and average predictions.
    """
    
    def __init__(self, n_augmentations=5):
        self.n_augmentations = n_augmentations
        self.augmenter = SignalAugmentation(p=0.5)
    
    def augment_batch(self, X):
        """Create augmented versions of input batch."""
        augmented = [X.copy()]  # Include original
        
        for _ in range(self.n_augmentations - 1):
            X_aug = np.array([self.augmenter(x) for x in X])
            augmented.append(X_aug)
        
        return augmented
    
    def predict_with_tta(self, model, X, device):
        """
        Make predictions using TTA.
        Returns averaged probabilities.
        """
        model.eval()
        augmented_batches = self.augment_batch(X)
        
        all_probs = []
        
        for X_aug in augmented_batches:
            X_tensor = torch.tensor(X_aug, dtype=torch.float32).to(device)
            
            with torch.no_grad():
                with autocast():
                    outputs = model(X_tensor)
                    probs = torch.softmax(outputs, dim=1)[:, 1]
            
            all_probs.append(probs.cpu().numpy())
        
        # Average across augmentations
        avg_probs = np.mean(all_probs, axis=0)
        return avg_probs


def evaluate_with_tta(model, X_test, y_test, n_aug=5):
    """
    Evaluate model using Test-Time Augmentation.
    """
    tta = TestTimeAugmentation(n_augmentations=n_aug)
    
    # Get TTA predictions
    probs = tta.predict_with_tta(model, X_test, device)
    preds = (probs > 0.5).astype(int)
    
    # Calculate metrics
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    f1 = f1_score(y_test, preds, zero_division=0)
    
    return acc, auc, f1, probs


# ─── Apply TTA to all trained DL models ──────────────────────────────────────

print(f"\nApplying TTA (n_augmentations=5) to all DL models...")

tta_results = {}

for name, cfg in dl_models.items():
    if cfg["model"] is not None:
        try:
            # Load trained weights from disk
            model_path = os.path.join(OUTPUT_DIR, f'{name}_best.pt')
            if os.path.exists(model_path):
                cfg["model"].load_state_dict(torch.load(model_path, map_location=device))
                cfg["model"].eval()
            acc, auc, f1, probs = evaluate_with_tta(cfg["model"], X_test_dl_scaled, y_test_dl, n_aug=5)
            
            # Compare with original
            orig_auc = dl_results[name]['auc'] if name in dl_results else 0
            improvement = auc - orig_auc
            
            tta_results[f"{name}_TTA"] = {
                'accuracy': acc,
                'auc': auc,
                'f1': f1,
                'improvement': improvement,
            }
            
            print(f"  {name}:")
            print(f"    Original AUC: {orig_auc:.4f}")
            print(f"    TTA AUC:      {auc:.4f} ({'+'if improvement >= 0 else ''}{improvement:.4f})")
            
        except Exception as e:
            print(f"  {name}: TTA failed - {e}")

# ─── Find best TTA model ─────────────────────────────────────────────────────

if tta_results:
    best_tta_name = max(tta_results, key=lambda x: tta_results[x]['auc'])
    best_tta_auc = tta_results[best_tta_name]['auc']
    
    print(f"\n{'='*70}")
    print(f"BEST TTA MODEL: {best_tta_name}")
    print(f"  AUC: {best_tta_auc:.4f}")
    print(f"  Accuracy: {tta_results[best_tta_name]['accuracy']:.4f}")
    print(f"  F1: {tta_results[best_tta_name]['f1']:.4f}")
    print(f"{'='*70}")

print("\n✓ Test-Time Augmentation complete!")

## 8. Visualizations

In [ ]:
# ─── Training Curves ─────────────────────────────────────────────

fig, axes = plt.subplots(2, len(dl_histories), figsize=(6*len(dl_histories), 10))

for i, (name, hist) in enumerate(dl_histories.items()):
    # Loss
    axes[0, i].plot(hist['train_loss'], label='Train', linewidth=1.5)
    axes[0, i].plot(hist['val_loss'], label='Val', linewidth=1.5)
    axes[0, i].set_title(f'{name} — Loss', fontsize=11)
    axes[0, i].set_xlabel('Epoch')
    axes[0, i].legend()
    axes[0, i].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1, i].plot(hist['train_acc'], label='Train', linewidth=1.5)
    axes[1, i].plot(hist['val_acc'], label='Val', linewidth=1.5)
    axes[1, i].set_title(f'{name} — Accuracy', fontsize=11)
    axes[1, i].set_xlabel('Epoch')
    axes[1, i].legend()
    axes[1, i].grid(True, alpha=0.3)

plt.suptitle('Training Curves — All DL Models', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dl_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── ROC Curves — All DL Models ──────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 8))


for name in dl_predictions:
    probs = dl_predictions[name]['probs']
    labels = dl_predictions[name]['labels']
    fpr, tpr, _ = roc_curve(labels, probs)
    auc_val = roc_auc_score(labels, probs)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})", linewidth=1.5)

# Ensemble
fpr_ens, tpr_ens, _ = roc_curve(y_test_dl, weighted_probs)
ax.plot(fpr_ens, tpr_ens, label=f"DL_Ensemble (AUC={weighted_auc:.4f})",
        linewidth=3, linestyle='--', color='black')

ax.plot([0, 1], [0, 1], 'k:', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curves — Deep Learning Models', fontsize=15)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dl_roc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Confusion Matrices — All DL Models ──────────────────────────

model_names = list(dl_predictions.keys()) + ['DL_Ensemble']
n_models = len(model_names)
cols = 3
rows = (n_models + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
axes = axes.flatten()

for i, name in enumerate(model_names):
    if name == 'DL_Ensemble':
        probs = weighted_probs
        labels = test_labels
    else:
        probs = dl_predictions[name]['probs']
        labels = dl_predictions[name]['labels']
    preds = (probs >= 0.5).astype(int)
    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
    acc = accuracy_score(labels, preds)
    axes[i].set_title(f'{name}\nAcc={acc:.3f}', fontsize=10)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Confusion Matrices — DL Models', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dl_confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Combined ML + DL Comparison

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# COMBINED ML + DL COMPARISON
# ═══════════════════════════════════════════════════════════════════════════════

# ─── Load ML results ─────────────────────────────────────────────────────────
ml_results_path = os.path.join(OUTPUT_DIR, 'ml_results.joblib')
if os.path.exists(ml_results_path):
    ml_results_loaded = joblib.load(ml_results_path)
    print("Loaded ML results from file.")
else:
    ml_results_loaded = results  # Use current session results
    print("Using ML results from current session.")

# ─── Combine all results ─────────────────────────────────────────────────────
print("="*70)
print("COMBINED ML + DL RESULTS")
print("="*70)

all_results = {}

# Add ML results with prefix
for name, r in ml_results_loaded.items():
    all_results[f'ML_{name}'] = {
        'accuracy': r['accuracy'],
        'auc': r['auc'],
        'f1': r['f1'],
        'precision': r['precision'],
        'recall': r['recall'],
        'type': 'ML',
    }

# Add DL results with prefix
for name, r in dl_results.items():
    all_results[f'DL_{name}'] = {
        'accuracy': r['accuracy'],
        'auc': r['auc'],
        'f1': r['f1'],
        'precision': r['precision'],
        'recall': r['recall'],
        'type': 'DL',
    }

# ─── Create Combined DataFrame ───────────────────────────────────────────────
combined_df = pd.DataFrame(all_results).T
combined_df = combined_df.sort_values('auc', ascending=False)

print("\nTop 15 Models by AUC:")
print(combined_df.head(15).to_string())

# ─── Overall Best Model ──────────────────────────────────────────────────────
print("\n" + "="*70)
print("OVERALL BEST MODELS")
print("="*70)

best_overall = combined_df.index[0]
print(f"\n★ Best Overall: {best_overall}")
print(f"   AUC:       {combined_df.loc[best_overall, 'auc']:.4f}")
print(f"   F1:        {combined_df.loc[best_overall, 'f1']:.4f}")
print(f"   Accuracy:  {combined_df.loc[best_overall, 'accuracy']:.4f}")
print(f"   Precision: {combined_df.loc[best_overall, 'precision']:.4f}")
print(f"   Recall:    {combined_df.loc[best_overall, 'recall']:.4f}")

# Best ML
best_ml = combined_df[combined_df['type'] == 'ML'].index[0]
print(f"\n★ Best ML: {best_ml}")
print(f"   AUC:       {combined_df.loc[best_ml, 'auc']:.4f}")
print(f"   F1:        {combined_df.loc[best_ml, 'f1']:.4f}")

# Best DL
best_dl = combined_df[combined_df['type'] == 'DL'].index[0]
print(f"\n★ Best DL: {best_dl}")
print(f"   AUC:       {combined_df.loc[best_dl, 'auc']:.4f}")
print(f"   F1:        {combined_df.loc[best_dl, 'f1']:.4f}")

# ─── Save combined results ───────────────────────────────────────────────────
combined_df.to_csv(os.path.join(OUTPUT_DIR, 'combined_results.csv'))
print("\nCombined results saved to combined_results.csv")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# BAR CHART COMPARISON: ML vs DL
# ═══════════════════════════════════════════════════════════════════════════════

metrics_to_plot = ['accuracy', 'auc', 'f1', 'precision', 'recall']

# Get top 5 ML and top 5 DL
top_ml = combined_df[combined_df['type'] == 'ML'].head(5)
top_dl = combined_df[combined_df['type'] == 'DL'].head(5)

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle('Top 5 ML vs Top 5 DL Models - Metrics Comparison', fontsize=14, fontweight='bold')

for ax, metric in zip(axes, metrics_to_plot):
    x = np.arange(5)
    width = 0.35
    
    ml_vals = top_ml[metric].values
    dl_vals = top_dl[metric].values
    
    bars1 = ax.bar(x - width/2, ml_vals, width, label='ML', color='steelblue', alpha=0.8)
    bars2 = ax.bar(x + width/2, dl_vals, width, label='DL', color='coral', alpha=0.8)
    
    ax.set_ylabel(metric.upper())
    ax.set_title(metric.upper())
    ax.set_xticks(x)
    ax.set_xticklabels([f'#{i+1}' for i in range(5)], fontsize=8)
    ax.legend(loc='lower right', fontsize=8)
    ax.set_ylim([0, 1])
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar in bars1:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 2), textcoords='offset points', ha='center', va='bottom', fontsize=7)
    for bar in bars2:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 2), textcoords='offset points', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'ml_vs_dl_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

# ─── Summary Statistics ──────────────────────────────────────────────────────
print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)

print("\nML Models:")
print(f"  Mean AUC: {combined_df[combined_df['type']=='ML']['auc'].mean():.4f}")
print(f"  Max AUC:  {combined_df[combined_df['type']=='ML']['auc'].max():.4f}")

print("\nDL Models:")
print(f"  Mean AUC: {combined_df[combined_df['type']=='DL']['auc'].mean():.4f}")
print(f"  Max AUC:  {combined_df[combined_df['type']=='DL']['auc'].max():.4f}")

print("\nOverall Best:")
print(f"  {combined_df.index[0]}: AUC = {combined_df.iloc[0]['auc']:.4f}")


## 10. Best Model — Detailed Report

In [ ]:
# ─── Best DL model detailed classification report ────────────────

best_name = dl_results_df.index[0]

if 'Ensemble' in best_name:
    best_probs = weighted_probs
    best_labels = y_test_dl
else:
    best_probs = dl_predictions[best_name]['probs']
    best_labels = dl_predictions[best_name]['labels']

best_preds = (best_probs >= 0.5).astype(int)

print(f"{'='*60}")
print(f"BEST DL MODEL: {best_name}")
print(f"{'='*60}")
print(f"\nTest AUC: {roc_auc_score(best_labels, best_probs):.4f}")
print(f"\nClassification Report:")
print(classification_report(best_labels, best_preds, target_names=['Real', 'Deepfake']))

if best_name in dl_results and best_name != 'DL_Ensemble':
    print(f"\nModel config:")
    cfg = dl_models[best_name]
    print(f"  lr: {cfg['lr']}")
    print(f"  weight_decay: {cfg['weight_decay']}")
    print(f"  epochs_trained: {dl_results[best_name]['epochs_trained']}")

## 11. Ultimate ML + DL Hybrid Ensemble

This combines the best ML and DL models using multiple ensemble strategies.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ULTIMATE ML + DL HYBRID ENSEMBLE
# ═══════════════════════════════════════════════════════════════════════════════
# Combines the best ML and DL models for maximum accuracy
# This is the state-of-the-art approach for tabular + signal data
# ═══════════════════════════════════════════════════════════════════════════════

# Verify test set alignment between ML and DL
# Both use random_state=42 with same test_size=0.2, so indices should match
# If not available, use the DL test labels
if 'y_test_dl' in globals():
    hybrid_y_test = y_test_dl
elif 'y_test' in globals():
    hybrid_y_test = y_test
else:
    raise NameError("No test labels available. Run ML or DL training first.")


print("="*70)
print("BUILDING ULTIMATE ML + DL HYBRID ENSEMBLE")
print("="*70)

# ─── Collect predictions from best models ────────────────────────────────────

# ML model predictions
ml_predictions = {}
for name in ['XGBoost', 'LightGBM', 'CatBoost', 'StackingClassifier', 'VotingEnsemble_Top7']:
    if name in best_models:
        try:
            probs = best_models[name].predict_proba(X_test_selected)[:, 1]
            ml_predictions[name] = probs
            print(f"  ✓ ML: {name}")
        except:
            pass



# DL model predictions
dl_hybrid_probs = {}
dl_model_names = ['CNN_BiLSTM', 'PhysNet_MLP', 'CNN_1D', 'BiLSTM_Attention', 'Optuna_PhysNet', 'MultiScale_CNN', 'TemporalAttention', 'WideDeep']

for name in dl_model_names:
    if name in dl_predictions:
        dl_hybrid_probs[name] = dl_predictions[name]['probs']
        print(f"  ✓ DL: {name}")

# Now, update the all_predictions list properly:
all_predictions = list(ml_predictions.values()) + list(dl_hybrid_probs.values())




print(f"\nTotal models for hybrid ensemble: {len(ml_predictions) + len(dl_predictions)}")

# ─── Method 1: Simple Average Ensemble ────────────────────────────────────────
print(f"\n{'─'*70}")
print("[1] SIMPLE AVERAGE ENSEMBLE")

all_predictions = list(ml_predictions.values()) + [d['probs'] if isinstance(d, dict) else d for d in dl_predictions.values()]
if len(all_predictions) > 0:
    avg_probs = np.mean(all_predictions, axis=0)
    avg_preds = (avg_probs > 0.5).astype(int)
    
    avg_acc = accuracy_score(hybrid_y_test, avg_preds)
    avg_auc = roc_auc_score(hybrid_y_test, avg_probs)
    avg_f1 = f1_score(hybrid_y_test, avg_preds, zero_division=0)
    
    print(f"  Accuracy: {avg_acc:.4f}")
    print(f"  AUC: {avg_auc:.4f}")
    print(f"  F1: {avg_f1:.4f}")

# ─── Method 2: Weighted Average (by individual AUC) ───────────────────────────
print(f"\n{'─'*70}")
print("[2] WEIGHTED AVERAGE ENSEMBLE (by individual AUC)")

# Calculate weights based on individual model AUC
weights = []
predictions_list = []

for name, probs in ml_predictions.items():
    auc = roc_auc_score(hybrid_y_test, probs)
    weights.append(auc)
    predictions_list.append(probs)
    print(f"  {name}: AUC={auc:.4f}")

for name, data in dl_predictions.items():
    probs = data['probs'] if isinstance(data, dict) else data
    auc = roc_auc_score(hybrid_y_test, probs)
    weights.append(auc)
    predictions_list.append(probs)
    print(f"  {name}: AUC={auc:.4f}")

if len(predictions_list) > 0:
    # Normalize weights
    weights = np.array(weights)
    weights = weights / np.sum(weights)
    
    # Weighted average
    weighted_probs = np.zeros_like(predictions_list[0])
    for w, p in zip(weights, predictions_list):
        weighted_probs += w * p
    
    weighted_preds = (weighted_probs > 0.5).astype(int)
    
    weighted_acc = accuracy_score(hybrid_y_test, weighted_preds)
    weighted_auc = roc_auc_score(hybrid_y_test, weighted_probs)
    weighted_f1 = f1_score(hybrid_y_test, weighted_preds, zero_division=0)
    
    print(f"\n  Weighted Ensemble:")
    print(f"    Accuracy: {weighted_acc:.4f}")
    print(f"    AUC: {weighted_auc:.4f}")
    print(f"    F1: {weighted_f1:.4f}")

# ─── Method 3: Stacking with Meta-Learner ─────────────────────────────────────
print(f"\n{'─'*70}")
print("[3] META-LEARNER STACKING (ML + DL predictions as features)")

# Create meta-features from predictions
meta_train_features = []
meta_test_features = []

# Get training predictions via cross-validation
from sklearn.model_selection import cross_val_predict

for name, model in best_models.items():
    if name in ['XGBoost', 'LightGBM', 'CatBoost', 'RandomForest', 'ExtraTrees']:
        try:
            # CV predictions for training set
            cv_probs = cross_val_predict(model, X_train_selected, y_train, cv=5, method='predict_proba')[:, 1]
            meta_train_features.append(cv_probs)
            
            # Test predictions
            test_probs = model.predict_proba(X_test_selected)[:, 1]
            meta_test_features.append(test_probs)
            print(f"    Added {name} predictions")
        except Exception as e:
            print(f"    Skipped {name}: {e}")

if len(meta_train_features) >= 2:
    # Stack features
    X_meta_train = np.column_stack(meta_train_features)
    X_meta_test = np.column_stack(meta_test_features)
    
    # Add original features (optional - can improve performance)
    X_meta_train_full = np.hstack([X_train_selected, X_meta_train])
    X_meta_test_full = np.hstack([X_test_selected, X_meta_test])
    
    # Train meta-learner
    meta_lr = LogisticRegression(C=1.0, max_iter=5000, random_state=SEED)
    meta_lr.fit(X_meta_train_full, y_train)
    
    meta_probs = meta_lr.predict_proba(X_meta_test_full)[:, 1]
    meta_preds = (meta_probs > 0.5).astype(int)
    
    meta_acc = accuracy_score(hybrid_y_test, meta_preds)
    meta_auc = roc_auc_score(hybrid_y_test, meta_probs)
    meta_f1 = f1_score(hybrid_y_test, meta_preds, zero_division=0)
    
    print(f"\n  Meta-Learner Stacking:")
    print(f"    Accuracy: {meta_acc:.4f}")
    print(f"    AUC: {meta_auc:.4f}")
    print(f"    F1: {meta_f1:.4f}")

# ─── Method 4: Rank-Based Ensemble ────────────────────────────────────────────
print(f"\n{'─'*70}")
print("[4] RANK-BASED ENSEMBLE (robust to scale differences)")

from scipy.stats import rankdata

ranked_predictions = []
for probs in predictions_list:
    ranks = rankdata(probs) / len(probs)  # Normalize to [0, 1]
    ranked_predictions.append(ranks)

if len(ranked_predictions) > 0:
    rank_avg_probs = np.mean(ranked_predictions, axis=0)
    rank_preds = (rank_avg_probs > 0.5).astype(int)
    
    rank_acc = accuracy_score(hybrid_y_test, rank_preds)
    rank_auc = roc_auc_score(hybrid_y_test, rank_avg_probs)
    rank_f1 = f1_score(hybrid_y_test, rank_preds, zero_division=0)
    
    print(f"  Rank-Based Ensemble:")
    print(f"    Accuracy: {rank_acc:.4f}")
    print(f"    AUC: {rank_auc:.4f}")
    print(f"    F1: {rank_f1:.4f}")

# ─── Final Comparison ─────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print("HYBRID ENSEMBLE COMPARISON")
print("="*70)

hybrid_results = {
    'Simple_Average': {'accuracy': avg_acc, 'auc': avg_auc, 'f1': avg_f1} if 'avg_acc' in globals() else None,
    'Weighted_Average': {'accuracy': weighted_acc, 'auc': weighted_auc, 'f1': weighted_f1} if 'weighted_acc' in globals() else None,
    'Meta_Stacking': {'accuracy': meta_acc, 'auc': meta_auc, 'f1': meta_f1} if 'meta_acc' in globals() else None,
    'Rank_Based': {'accuracy': rank_acc, 'auc': rank_auc, 'f1': rank_f1} if 'rank_acc' in globals() else None,
}

print(f"{'Method':<20} {'Accuracy':>10} {'AUC':>10} {'F1':>10}")
print("-"*50)

best_hybrid = None
best_auc = 0

for name, r in hybrid_results.items():
    if r is not None:
        print(f"{name:<20} {r['accuracy']:>10.4f} {r['auc']:>10.4f} {r['f1']:>10.4f}")
        if r['auc'] > best_auc:
            best_auc = r['auc']
            best_hybrid = name

print(f"\n🏆 BEST HYBRID ENSEMBLE: {best_hybrid} (AUC: {best_auc:.4f})")

# Save best hybrid predictions
if best_hybrid == 'Weighted_Average':
    best_probs_final = weighted_probs
elif best_hybrid == 'Meta_Stacking':
    best_probs_final = meta_probs
elif best_hybrid == 'Rank_Based':
    best_probs_final = rank_avg_probs
else:
    best_probs_final = avg_probs

np.save(os.path.join(OUTPUT_DIR, 'best_hybrid_predictions.npy'), best_probs_final)
print(f"\nSaved best hybrid predictions to best_hybrid_predictions.npy")

## 11. Save All Artifacts

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SAVE ALL DEEP LEARNING ARTIFACTS
# ═══════════════════════════════════════════════════════════════════════════════

# Save model architectures (for reproducibility)
model_architectures = {}
for name, cfg in dl_models.items():
    model_architectures[name] = {
        'lr': cfg['lr'],
        'weight_decay': cfg['weight_decay'],
        'params': sum(p.numel() for p in cfg['model'].parameters()) if cfg['model'] is not None else 0,
    }
joblib.dump(model_architectures, os.path.join(OUTPUT_DIR, 'dl_model_configs.joblib'))

# Save training histories
joblib.dump(dl_histories, os.path.join(OUTPUT_DIR, 'dl_histories.joblib'))

# Save predictions for ensemble analysis
joblib.dump(dl_predictions, os.path.join(OUTPUT_DIR, 'dl_predictions.joblib'))

# Save results
dl_results_df.to_csv(os.path.join(OUTPUT_DIR, 'dl_results_detailed.csv'))
joblib.dump(dl_results, os.path.join(OUTPUT_DIR, 'dl_results.joblib'))

# Save combined results
combined_df.to_csv(os.path.join(OUTPUT_DIR, 'combined_ml_dl_results.csv'))

print("\n" + "="*70)
print("DEEP LEARNING ARTIFACTS SAVED:")
print("="*70)
print(f"  - {len(dl_models)} trained model weights (.pt)")
print(f"  - Model configurations and hyperparameters")
print(f"  - Training histories")
print(f"  - Predictions (for ensemble)")
print(f"  - Detailed results")
print(f"  - Combined ML+DL results")
print("="*70)

# ─── Print final summary ─────────────────────────────────────────────────────
print("\n" + "="*70)
print("TRAINING COMPLETE - FINAL SUMMARY")
print("="*70)

print("\nBest ML Model:")
best_ml = results_df.index[0]
print(f"  {best_ml}: AUC={results_df.loc[best_ml, 'auc']:.4f}")

print("\nBest DL Model:")
best_dl = dl_results_df.index[0]
print(f"  {best_dl}: AUC={dl_results_df.loc[best_dl, 'auc']:.4f}")

print("\nOverall Best:")
overall_best = combined_df.index[0]
print(f"  {overall_best}: AUC={combined_df.loc[overall_best, 'auc']:.4f}")

print("\n" + "="*70)
print("All artifacts saved to:", OUTPUT_DIR)
print("="*70)


In [ ]:
import shutil

shutil.make_archive("m_outputs", 'zip', "/kaggle/working")

In [ ]:
# ==============================================================================
# 🚀 PREPARATION FOR DL FUSION: EXPORTING P_rPPG
# ==============================================================================
import pandas as pd
import os

print("Generating final probabilities for Late Fusion...")

# 1. Recreate the exact order of video filenames to match the feature matrix (X)
# Ensure video path variables are available
if 'real_videos' not in globals() or 'fake_videos' not in globals():
    print("Warning: real_videos/fake_videos not defined. Skipping video path export.")
    all_video_paths = []
else:
    all_video_paths = real_videos + fake_videos
# 2. Load the feature matrix FIRST
X_rppg = np.load("/kaggle/working/X_features.npy")

# Align video paths with feature matrix
if 'valid_video_paths' in globals():
    video_filenames = [os.path.basename(path) for path in valid_video_paths]
    if len(video_filenames) != X_rppg.shape[0]:
        print(f"Warning: {len(video_filenames)} video paths but {X_rppg.shape[0]} feature rows")
        # Use generic names if mismatch
        video_filenames = [f"video_{i}" for i in range(X_rppg.shape[0])]
else:
    video_filenames = [f"video_{i}" for i in range(X_rppg.shape[0])]
X_all_scaled = scaler.transform(X_rppg)

# NEW FIX: Filter down to the selected top features BEFORE applying PolynomialFeatures
try:
    X_all_selected = X_all_scaled[:, top_indices]
except NameError:
    X_all_selected = X_all_scaled

# 3. Apply PolynomialFeatures to the full dataset
try:
    X_all_final = poly.transform(X_all_selected)
except NameError:
    X_all_final = X_all_selected

# NEW FIX: Guarantee we are using the absolute best Ensemble model
# Use best model from ML results
if 'best_ensemble_name' not in globals():
    best_ensemble_name = best_model_name if 'best_model_name' in globals() else 'XGBoost'

try:
    best_model = best_models.get(best_ensemble_name, list(best_models.values())[0])
except NameError:
    print("Warning: best_models not defined, using default model")
    best_model = None

# 4. Generate the deepfake probability (P_rPPG)
P_rPPG = best_model.predict_proba(X_all_final)[:, 1]

# Guarantee dimension match by loading pure rPPG labels
y_rppg = np.load("/kaggle/working/y_labels.npy")

# 5. Create the Fusion DataFrame
fusion_df = pd.DataFrame({
    'video_id': video_filenames,
    'true_label': y_rppg,
    'P_rPPG': P_rPPG
})

# 6. Save to CSV
output_path = '/kaggle/working/rppg_predictions.csv'
fusion_df.to_csv(output_path, index=False)

print(f"✅ Successfully saved {len(fusion_df)} predictions to {output_path}")